In [ ]:
!pip install -q transformers datasets evaluate sentencepiece
!pip install -q torch
!pip install -q sentence-transformers
!pip install -q nltk spacy
!pip install bert_score
!pip install textstat
!python -m spacy download en_core_web_sm
!pip install -q scikit-learn scipy gensim networkx
!git clone https://github.com/neulab/BARTScore.git

# Keyword Extraction

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")
from sklearn.metrics.pairwise import cosine_similarity

stop_words = set(stopwords.words("english"))
from nltk import sent_tokenize

file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


def extract_keywords(processed_sentences, top_k=200):
    all_words = []
    for sent in processed_sentences:
        all_words.extend(sent.split())
    freq_dist = FreqDist(all_words)
    return [w for w, _ in freq_dist.most_common(top_k)]


def rank_sentences(sentences, keywords):
    sentence_scores = {}
    for sentence in sentences:
        score = sum(1 for kw in keywords if kw in sentence.lower())
        sentence_scores[sentence] = score
    ranked = sorted(sentence_scores.items(), key=lambda x: x[1], reverse=True)
    return [s[0] for s in ranked]


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

def chunk_text(text, tokenizer, max_tokens=900, stride=150):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from transformers import AutoModelForSequenceClassification, AutoTokenizer
sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
    )
    sentences = doc["judgement_sent"]
    
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,   
        min_len=8,
        max_len=30
    )
    sentences_raw, processed_sentences = preprocess_text(sentences)
    keywords = extract_keywords(processed_sentences, top_k=200)
    ranked_sentences = rank_sentences(sentences_raw, keywords)
    ranked_text = " ".join(ranked_sentences[:top_n])
    

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )
    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # # # -----------------------
    # # # BLEU (DOCUMENT-WISE)
    # # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # # # -----------------------
    # # # METEOR (DOCUMENT-WISE)
    # # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # # # -----------------------
    # # # BERTScore (DOCUMENT-WISE)
    # # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )
    
    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    torch.cuda.empty_cache()

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# POS-Tagging

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


import nltk
import torch
import math
from nltk import sent_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nltk.download("punkt")
import nltk
nltk.download('averaged_perceptron_tagger_eng')
def get_pos_tags(text):
    words = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(words)
    return pos_tags


POS_TAG_MAP = {
    "NN": "<NN>", "NNS": "<NN>", "NNP": "<PN>", "NNPS": "<PN>",
    "VB": "<VB>", "VBD": "<VB>", "VBG": "<VB>", "VBN": "<VB>",
    "VBP": "<VB>", "VBZ": "<VB>",
    "JJ": "<JJ>", "JJR": "<JJ>", "JJS": "<JJ>",
    "RB": "<RB>", "RBR": "<RB>", "RBS": "<RB>",
    "IN": "<PREP>",
    "MD": "<MODAL>",
    "CD": "<NUM>"
}
def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

def add_pos_tags_to_input(input_text, pos_tags):
    pos_tagged_text = ""
    for word, pos in pos_tags:
        if pos in POS_TAG_MAP:
            pos_tagged_text += word + " " + POS_TAG_MAP[pos] + " "
        else:
            pos_tagged_text += word + " "
    return pos_tagged_text.strip()
def score_sentence_pos(sentence):
    pos_tags = get_pos_tags(sentence)

    score = 0
    for _, pos in pos_tags:
        if pos.startswith("NN"):
            score += 2      # entities, legal concepts
        elif pos.startswith("VB"):
            score += 2      # actions, rulings
        elif pos == "MD":
            score += 3      # shall, must, may (very important)
        elif pos == "CD":
            score += 1.5    # sections, dates
        elif pos.startswith("JJ"):
            score += 1

    return score
def summarize_with_pos_features(sentences, top_n=15):
    if len(sentences) <= top_n:
        return " ".join(sentences)

    scored = [(score_sentence_pos(s), s) for s in sentences]
    scored.sort(reverse=True, key=lambda x: x[0])

    selected_sentences = [s for _, s in scored[:top_n]]
    return " ".join(selected_sentences)


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))


predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []


for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
    )
    sentences = doc["judgement_sent"]

    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,   
        min_len=8,
        max_len=30
    )
    ranked_text = summarize_with_pos_features(
        sentences,
        top_n=top_n   
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

   
    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )
    
    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# Sentence positioning

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)


def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))


def slice_text(sentences,
               start_portion=0.20,
               middle_portion=0.20,
               end_portion=0.60,
               ratio=0.25,
               min_len=8,
               max_len=30):

    N = len(sentences)

    # Step 1: dynamic summary length
    target_len = estimate_summary_length(sentences, ratio, min_len, max_len)

    # Step 2: sentence allocation
    start_k = math.ceil(target_len * start_portion)
    middle_k = math.ceil(target_len * middle_portion)
    end_k = target_len - start_k - middle_k

    # Step 3: define document regions
    start_boundary = math.floor(0.20 * N)
    middle_start = math.floor(0.40 * N)
    middle_end = math.floor(0.60 * N)
    end_boundary = math.floor(0.80 * N)

    start_candidates = sentences[:start_boundary]
    middle_candidates = sentences[middle_start:middle_end]
    end_candidates = sentences[end_boundary:]

    # Step 4: avoid overflow if region has fewer sentences
    start_k = min(start_k, len(start_candidates))
    middle_k = min(middle_k, len(middle_candidates))
    end_k = min(end_k, len(end_candidates))

    # Step 5: select sentences
    selected = (
        start_candidates[:start_k] +
        middle_candidates[:middle_k] +
        end_candidates[-end_k:]
    )

    return selected

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary

predictions, references = [], []
rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))


import gc 

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    
    sentences = [
        preprocess(s)
        for s in doc["judgement_sent"]
        # if not is_metadata_sentence(s)
    ]
    judgement = " ".join(sentences)
    selected_sentences = slice_text(sentences)
    ranked_text = " ".join(selected_sentences)

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    
    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    
    
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )


    del reference
    del judgement
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# Sentence Length

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import spacy
import textstat
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
# from summac.model_summac import SummaCConv, SummaCZS
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
    
nlp = spacy.load("en_core_web_sm")

# -------------------------------
# Sentence Readability & Length Scoring
# -------------------------------
def score_sentences_length(sentences):
    sentence_scores = {}

    for i, sent in enumerate(sentences):
        length_score = len(sent.split())

        # Flesch Reading Ease (higher = easier)
        readability_score = textstat.flesch_reading_ease(sent)

        # Prefer moderate length (~20 words)
        score = readability_score / (1 + abs(length_score - 20))

        sentence_scores[i] = score

    return sentence_scores
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))


# -------------------------------
# Readability & Length Summarizer
# -------------------------------
def summarize_with_length(sentences, top_n):

    if len(sentences) <= top_n:
        return " ".join(sentences)

    scores = score_sentences_length(sentences)

    ranked_sentences = sorted(
        ((score, sentences[i]) for i, score in scores.items()),
        reverse=True
    )

    summary_sentences = [s for _, s in ranked_sentences[:top_n]]
    return " ".join(summary_sentences)
def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
# -----------------------
# FactCC (CHUNKED)
# -----------------------

def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))
from nltk import sent_tokenize

import gc    
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):

    reference = " ".join(doc["headnote_sent"])
    sentences = [preprocess(s) for s in doc["judgement_sent"]]
    judgement = " ".join(sentences)
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,   # can tune (0.20–0.35)
        min_len=8,
        max_len=30
    )
    ranked_text = summarize_with_length(sentences, top_n=top_n)

    # -----------------------
    # Generate summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    del reference
    del judgement
    del sentences
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r

    # -----------------------
    # MEMORY CLEANUP
    # -----------------------
    torch.cuda.empty_cache()
    gc.collect()

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# Legal BERT Similarity

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification
)

from sklearn.metrics.pairwise import cosine_similarity
from bart_score import BARTScorer
with open("/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json", "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained("/kaggle/input/datasets/pawank1999/distilbart-9500-output").to(device)
tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/pawank1999/distilbart-9500-output")
model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

bart_scorer = BARTScorer(device=device, checkpoint="facebook/bart-large-cnn")
factcc_model_name = "manueldeprada/FactCC"
factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)
factcc_model.eval()
tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained(
    "nlpaueb/legal-bert-base-uncased"
).to(device)
model_bert.eval()
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
def get_sentence_embeddings(sentences):
    embeddings = []
    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(emb[0])

    return np.array(embeddings)
def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def select_top_sentences(sentences, embeddings, threshold=0.4):
    n = len(sentences)
    counts = np.zeros(n)

    for i in range(n):
        for j in range(i + 1, n):
            sim = cosine_similarity(
                [embeddings[i]], [embeddings[j]]
            )[0][0]

            if sim > threshold:
                counts[i] += 1
                counts[j] += 1

    sorted_indices = np.argsort(-counts)
    target_len = estimate_summary_length(sentences)
    top_indices = sorted(sorted_indices[:target_len])
    selected_sentences = [sentences[i] for i in top_indices]
    return " ".join(selected_sentences)

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores, factcc_scores = [], []

for doc in tqdm(dataset, desc="Evaluating"):

    reference = " ".join(doc["headnote_sent"])
    sentences = [preprocess(s) for s in doc["judgement_sent"] if len(s.strip()) > 5]
    judgement = " ".join(sentences)
    if len(sentences) < 3:
        continue

    embeddings = get_sentence_embeddings(sentences)
    ranked_text = select_top_sentences(sentences, embeddings)

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )


    torch.cuda.empty_cache()
print("\n📊 LEGAL-BERT SIMILARITY SENTENCE SELECTION RESULTS\n")

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")



# LSA Based

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from bart_score import BARTScorer
with open("/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json", "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained("/kaggle/input/datasets/pawank1999/distilbart-9500-output").to(device)
tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/pawank1999/distilbart-9500-output")
model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

bart_scorer = BARTScorer(device=device, checkpoint="facebook/bart-large-cnn")
factcc_model_name = "manueldeprada/FactCC"
factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)
factcc_model.eval()
tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained(
    "nlpaueb/legal-bert-base-uncased"
).to(device)
model_bert.eval()
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
    
def compute_sentence_embeddings_lsa(sentences, n_components=100):
    """
    Compute LSA-based sentence embeddings
    """
    vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(sentences)
    n_components = min(n_components, tfidf_matrix.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    embeddings = svd.fit_transform(tfidf_matrix)
    return embeddings
    
def compute_similarity_scores(embeddings, threshold=0.4):
    """
    Count how many sentences each sentence is similar to
    """
    n = len(embeddings)
    counts = np.zeros(n)

    sim_matrix = cosine_similarity(embeddings)

    for i in range(n):
        for j in range(n):
            if i != j and sim_matrix[i][j] > threshold:
                counts[i] += 1

    return counts
    
def select_top_sentences(sentences, counts, target_len):
    """
    Select top central sentences while preserving order
    """
    ranked_indices = np.argsort(-counts)
    top_indices = sorted(ranked_indices[:target_len])
    return [sentences[i] for i in top_indices]
    return " ".join([sentences[i] for i in top_indices])
    
def lsa_sentence_selection(sentences,
                           n_components=100,
                           similarity_threshold=0.4):
    """
    End-to-end LSA-based sentence selection
    """
    if len(sentences) < 3:
        return " ".join(sentences)

    embeddings = compute_sentence_embeddings_lsa(
        sentences,
        n_components=n_components
    )

    counts = compute_similarity_scores(
        embeddings,
        threshold=similarity_threshold
    )

    target_len = estimate_summary_length(sentences)

    selected_sentences = select_top_sentences(
        sentences,
        counts,
        target_len
    )

    return " ".join(selected_sentences)

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores, factcc_scores = [], []


for doc in tqdm(dataset, desc="Evaluating"):

    reference = " ".join(doc["headnote_sent"])
    sentences = [preprocess(s) for s in doc["judgement_sent"] if len(s.strip()) > 5]
    judgement = " ".join(sentences)
    if len(sentences) < 3:
        continue

    ranked_text = lsa_sentence_selection(
        sentences,
        n_components=100,
        similarity_threshold=0.4
    )

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    torch.cuda.empty_cache()
print("\n📊 RESULTS\n")

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# TextRank

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

def build_similarity_matrix(sentences):
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sentences)
    similarity_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()
    return similarity_matrix

def summarize_with_textrank(sentences, top_n):
    if len(sentences) <= top_n:
        return " ".join(sentences)

    similarity_matrix = build_similarity_matrix(sentences)
    nx_graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(nx_graph)

    ranked_sentences = sorted(
        ((scores[i], s) for i, s in enumerate(sentences)),
        reverse=True
    )

    summary_sentences = [s for _, s in ranked_sentences[:top_n]]
    return " ".join(summary_sentences)
    
def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
    
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

    
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
    )
    sentences = doc["judgement_sent"]

    sentences = [
        preprocess(s)
        for s in doc["judgement_sent"]
    ]
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,   
        min_len=8,
        max_len=30
    )

    ranked_text = summarize_with_textrank(
        sentences,
        top_n=top_n   
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# LexRank

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer


import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))


# -----------------------
# LexRank Similarity Matrix
# -----------------------
def build_lexrank_matrix(sentences):

    vectorizer = TfidfVectorizer(
        stop_words='english'
    )

    tfidf = vectorizer.fit_transform(sentences)

    similarity = cosine_similarity(tfidf)

    return similarity


# -----------------------
# LexRank Sentence Selection
# -----------------------
def summarize_with_lexrank(
    sentences,
    top_n=10,
    threshold=0.1
):

    if len(sentences) <= top_n:
        return " ".join(sentences)

    similarity_matrix = build_lexrank_matrix(sentences)

    # create graph
    graph = nx.Graph()

    num_sent = len(sentences)

    for i in range(num_sent):
        for j in range(num_sent):

            if i != j and similarity_matrix[i][j] > threshold:

                graph.add_edge(
                    i,
                    j,
                    weight=float(similarity_matrix[i][j])
                )

    # pagerank centrality
    scores = nx.pagerank(
        graph,
        weight='weight'
    )

    ranked = sorted(
        ((scores.get(i, 0), s, i)
         for i, s in enumerate(sentences)),
        reverse=True
    )

    selected_idx = sorted(
        [idx for (_, _, idx) in ranked[:top_n]]
    )

    summary = " ".join(
        [sentences[i] for i in selected_idx]
    )

    return summary

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
    
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

    
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
        # if not is_metadata_sentence(s)
    )
    sentences = doc["judgement_sent"]

    sentences = [
        preprocess(s)
        for s in doc["judgement_sent"]
    ]
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,   # can tune (0.20–0.35)
        min_len=8,
        max_len=30
    )
    # 🔹 TextRank sentence selection
    ranked_text = summarize_with_lexrank(
        sentences,
        top_n=top_n,
        threshold=0.12
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# Kl Divergence

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
    
def compute_distribution(text):
    words = text.split()
    counts = Counter(words)
    total = sum(counts.values())
    return {w: c / total for w, c in counts.items()}
    
def kl_divergence(p, q, epsilon=1e-12):
    kl = 0.0
    for w in p:
        pw = p[w]
        qw = q.get(w, epsilon)
        kl += pw * math.log(pw / qw)
    return kl
    
def summarize_with_kl(sentences, top_n=15):
    doc_text = " ".join(sentences)
    p_doc = compute_distribution(doc_text)

    selected = []
    remaining = sentences.copy()

    while len(selected) < top_n and remaining:
        best_sentence = None
        best_kl = float("inf")

        for sent in remaining:
            candidate = selected + [sent]
            candidate_text = " ".join(candidate)
            p_candidate = compute_distribution(candidate_text)

            kl = kl_divergence(p_doc, p_candidate)

            if kl < best_kl:
                best_kl = kl
                best_sentence = sent

        selected.append(best_sentence)
        remaining.remove(best_sentence)

    return " ".join(selected)
    
def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
        # if not is_metadata_sentence(s)
    )
    sentences = doc["judgement_sent"]

    sentences = [
        preprocess(s)
        for s in doc["judgement_sent"]
    ]
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,  
        min_len=8,
        max_len=30
    )
    
    ranked_text = summarize_with_kl(
        sentences,
        top_n=top_n  
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # # -----------------------
    # # BLEU (DOCUMENT-WISE)
    # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # # -----------------------
    # # METEOR (DOCUMENT-WISE)
    # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # # -----------------------
    # # BERTScore (DOCUMENT-WISE)
    # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )


    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# LDA based

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences

import nltk
import torch
import math
from nltk import sent_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def lda_preprocess(sentences):
    return [
        s.lower()
        for s in sentences
        if len(s.split()) > 5
    ]
def compute_lda_sentence_scores(sentences, n_topics=10):
    """
    sentences: list[str]
    returns: topic_weight per sentence
    """

    vectorizer = CountVectorizer(
        stop_words="english",
        max_df=0.9,
        min_df=2
    )
    doc_term_matrix = vectorizer.fit_transform(sentences)

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method="batch"
    )
    lda.fit(doc_term_matrix)
    topic_distributions = lda.transform(doc_term_matrix)
    sentence_scores = topic_distributions.max(axis=1)
    return sentence_scores
    
def summarize_with_lda(sentences, top_n=15):
    if len(sentences) <= top_n:
        return " ".join(sentences)

    clean_sentences = lda_preprocess(sentences)

    scores = compute_lda_sentence_scores(clean_sentences)

    ranked = sorted(
        zip(clean_sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    selected = [s for s, _ in ranked[:top_n]]

    return " ".join(selected)


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
    )
    sentences = doc["judgement_sent"]
    
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,  
        min_len=8,
        max_len=30
    )
    ranked_text = summarize_with_lda(
        sentences,
        top_n=top_n
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # # -----------------------
    # # BLEU (DOCUMENT-WISE)
    # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # # -----------------------
    # # METEOR (DOCUMENT-WISE)
    # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # # -----------------------
    # # BERTScore (DOCUMENT-WISE)
    # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# JS divergence

In [ ]:
import sys
import os

# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon
file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences

import nltk
import torch
import math
from nltk import sent_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer

def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
    
def get_probability_distribution(texts, vectorizer=None):
    """
    texts: list[str]
    returns: normalized probability distribution
    """
    if vectorizer is None:
        vectorizer = CountVectorizer(stop_words="english")

    counts = vectorizer.fit_transform(texts)
    word_counts = counts.sum(axis=0).A1

    prob_dist = word_counts / word_counts.sum()
    return prob_dist, vectorizer
def get_sentence_distribution(sentence, vectorizer):
    counts = vectorizer.transform([sentence])
    word_counts = counts.toarray().flatten()

    if word_counts.sum() == 0:
        return None

    return word_counts / word_counts.sum()
def compute_jsd_scores(sentences):
    """
    returns: list of (sentence, jsd_score)
    """

    # Document distribution
    doc_dist, vectorizer = get_probability_distribution(sentences)

    scored_sentences = []

    for sent in sentences:
        sent_dist = get_sentence_distribution(sent, vectorizer)

        if sent_dist is None:
            continue

        jsd = jensenshannon(sent_dist, doc_dist)
        scored_sentences.append((sent, jsd))

    return scored_sentences
    
def summarize_with_jsd(sentences, top_n=15):
    if len(sentences) <= top_n:
        return " ".join(sentences)
    jsd_scores = compute_jsd_scores(sentences)
    ranked = sorted(jsd_scores, key=lambda x: x[1])
    selected_sentences = [s for s, _ in ranked[:top_n]]
    return " ".join(selected_sentences)


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []


total_loss = 0.0
total_tokens = 0
TOP_N = 15 
for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
    )
    sentences = doc["judgement_sent"]
    
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25, 
        min_len=8,
        max_len=30
    )
    ranked_text = summarize_with_jsd(
        sentences,
        top_n=top_n
    )
    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # # -----------------------
    # # BLEU (DOCUMENT-WISE)
    # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # # -----------------------
    # # METEOR (DOCUMENT-WISE)
    # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # # -----------------------
    # # BERTScore (DOCUMENT-WISE)
    # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # # -----------------------
    # # BARTScore (DOCUMENT-WISE)
    # # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )


    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )


    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# NMF based

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon
file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences

from nltk import sent_tokenize

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def nmf_preprocess(sentences):
    return [
        s.strip()
        for s in sentences
        if len(s.split()) > 5
    ]
def estimate_n_topics(sentences,
                      min_topics=8,
                      max_topics=30):
    N = len(sentences)
    k = int(np.sqrt(N))
    return max(min_topics, min(k, max_topics))    
def compute_nmf_sentence_scores(sentences):
    """
    sentences: list[str]
    returns: sentence importance scores
    """
    n_topics = estimate_n_topics(sentences)
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_df=0.9,
        min_df=2
    )
    tfidf_matrix = vectorizer.fit_transform(sentences)
    nmf = NMF(
        n_components=n_topics,
        random_state=42,
        init="nndsvda",
        max_iter=500
    )
    W = nmf.fit_transform(tfidf_matrix)
    sentence_scores = W.max(axis=1)
    return sentence_scores
    
def summarize_with_nmf(sentences,
                       ratio=0.25,
                       min_len=8,
                       max_len=30):

    top_n = estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

    if len(sentences) <= top_n:
        return " ".join(sentences)

    clean_sentences = nmf_preprocess(sentences)

    if len(clean_sentences) < 5:
        return " ".join(sentences[:top_n])

    scores = compute_nmf_sentence_scores(clean_sentences)

    ranked = sorted(
        zip(clean_sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    selected = [s for s, _ in ranked[:top_n]]

    return " ".join(selected)

def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
    
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
    )
    sentences = doc["judgement_sent"]
    
    ranked_text = summarize_with_nmf(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )
    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # # -----------------------
    # # BLEU (DOCUMENT-WISE)
    # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # # -----------------------
    # # METEOR (DOCUMENT-WISE)
    # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # # -----------------------
    # # BERTScore (DOCUMENT-WISE)
    # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# hierarchical LDA

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
from transformers import AutoModelForSeq2SeqLM
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()

def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)
    
ROLE_PRIORITY = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "RLC": 0.55,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PRE_NOT_RELIED": 0.4,
    "PREAMBLE": 0.3,
    "NONE": 0.2
}


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import HdpModel
from gensim.corpora import Dictionary

def hlda_summarization(sentences,
                       roles,
                       ratio=0.25,
                       min_len=8,
                       max_len=30):

    n = len(sentences)
    if n < 5:
        return " ".join(sentences)

    summary_size = estimate_summary_length(
        sentences, ratio, min_len, max_len
    )
    # -----------------------
    # Tokenization
    # -----------------------
    tokenized = [
        [w for w in sent.lower().split() if w.isalpha()]
        for sent in sentences
    ]
    dictionary = Dictionary(tokenized)
    corpus = [dictionary.doc2bow(doc) for doc in tokenized]
    # -----------------------
    # Train HDP model
    # -----------------------
    hdp = HdpModel(corpus, dictionary)
    # -----------------------
    # Sentence scoring
    # -----------------------
    scores = []

    for i, (doc_bow, role) in enumerate(zip(corpus, roles)):
        topic_dist = hdp[doc_bow]
        if topic_dist:
            probs = np.array([p for _, p in topic_dist])
            probs = probs / probs.sum()
            entropy = -np.sum(probs * np.log(probs + 1e-9))
        else:
            entropy = 0.0
        role_score = ROLE_PRIORITY.get(role, 0)
        score = 0.6 * role_score + 0.4 * entropy
        scores.append((i, score))

    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    selected_indices = [idx for idx, _ in ranked[:summary_size]]
    selected_indices.sort()
    summary = " ".join([sentences[i] for i in selected_indices])
    return summary
    
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
import gc


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )
    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]
    
    ranked_text = hlda_summarization(
        sentences,
        roles,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    del reference
    del judgement
    del sentences
    del roles
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# SBERT + KMeans

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon
file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences

from nltk import sent_tokenize

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

def summarize_with_sbert_kmeans(sentences,
                                ratio=0.25,
                                min_len=8,
                                max_len=30):

    if len(sentences) < 5:
        return " ".join(sentences)

    # Adaptive number of clusters
    n_clusters = estimate_n_clusters(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )
    if len(sentences) <= n_clusters:
        return " ".join(sentences)
    # Encode sentences
    embeddings = sbert_model.encode(
        sentences,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    # KMeans clustering
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )
    kmeans.fit(embeddings)
    selected_indices = []

    # Select representative per cluster
    for i in range(n_clusters):
        cluster_indices = np.where(kmeans.labels_ == i)[0]
        if len(cluster_indices) == 0:
            continue
        cluster_embeddings = embeddings[cluster_indices]
        centroid = kmeans.cluster_centers_[i]
        distances = np.linalg.norm(
            cluster_embeddings - centroid,
            axis=1
        )
        closest_idx = cluster_indices[np.argmin(distances)]
        selected_indices.append(closest_idx)

    # Remove duplicates (rare but safe)
    selected_indices = list(set(selected_indices))
    # Sort by original order (important for coherence)
    selected_indices.sort()
    selected_sentences = [sentences[i] for i in selected_indices]
    return " ".join(selected_sentences)
    
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
    )
    sentences = doc["judgement_sent"]
    
    ranked_text = summarize_with_sbert_kmeans(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # # -----------------------
    # # BLEU (DOCUMENT-WISE)
    # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # # -----------------------
    # # METEOR (DOCUMENT-WISE)
    # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # # -----------------------
    # # BERTScore (DOCUMENT-WISE)
    # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )
  
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )


    torch.cuda.empty_cache()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# Submodular Optimization

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

# dataset = dataset[9500:]   # test split


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences

from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()

def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)
ROLE_PRIORITY = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "RLC": 0.55,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PRE_NOT_RELIED": 0.4,
    "PREAMBLE": 0.3,
    "NONE": 0.2
}

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

def submodular_summarization(sentences, roles,
                             ratio=0.25,
                             min_len=8,
                             max_len=30,
                             lambda_div=0.3):

    n = len(sentences)
    if n < 5:
        return " ".join(sentences)

    summary_size = estimate_summary_length(
        sentences, ratio, min_len, max_len
    )

    embeddings = legalbert_encode(sentences)
    similarity_matrix = cosine_similarity(embeddings)

    selected = set()
    covered = np.zeros(n)

    for _ in range(summary_size):

        best_gain = -float("inf")
        best_idx = None

        for i in range(n):
            if i in selected:
                continue

            # Coverage gain
            new_cover = np.maximum(covered, similarity_matrix[:, i])
            coverage_gain = new_cover.sum() - covered.sum()

            # Redundancy penalty
            redundancy = sum(similarity_matrix[i][j] for j in selected)

            gain = coverage_gain - lambda_div * redundancy
            gain += ROLE_PRIORITY.get(roles[i], 0)

            if gain > best_gain:
                best_gain = gain
                best_idx = i

        if best_idx is None:
            break

        selected.add(best_idx)
        covered = np.maximum(covered, similarity_matrix[:, best_idx])

    selected = sorted(list(selected))
    summary = " ".join([sentences[i] for i in selected])

    return summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
import gc

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):

    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )
    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]
    
    ranked_text = submodular_summarization(
        sentences,
        roles,
        ratio=0.25,
        min_len=8,
        max_len=30
    )
    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )


    
    del reference
    del judgement
    del sentences
    del roles
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# Doc2Vec + MMR

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()

def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)

ROLE_PRIORITY = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "RLC": 0.55,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PRE_NOT_RELIED": 0.4,
    "PREAMBLE": 0.3,
    "NONE": 0.2
}

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
def doc2vec_mmr_summarization(sentences, roles,
                              ratio=0.25,
                              min_len=8,
                              max_len=30,
                              lambda_param=0.7):
    
    n = len(sentences)
    if n == 0:
        return []

    # -----------------------------------
    # Dynamic summary length
    # -----------------------------------
    k = estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

    # -----------------------------------
    # Train Doc2Vec
    # -----------------------------------
    tagged_docs = [
        TaggedDocument(sent.split(), [i])
        for i, sent in enumerate(sentences)
    ]

    model = Doc2Vec(
        tagged_docs,
        vector_size=100,
        window=5,
        min_count=1,
        epochs=40
    )

    embeddings = np.array([model.dv[i] for i in range(n)])

    # -----------------------------------
    # Normalize role priority
    # -----------------------------------
    role_scores = np.array([
        ROLE_PRIORITY.get(role, 0)
        for role in roles
    ])

    if role_scores.max() > 0:
        role_scores = role_scores / role_scores.max()

    # -----------------------------------
    # MMR Selection
    # -----------------------------------
    selected = []
    remaining = list(range(n))

    # Start with highest role priority
    first_idx = max(remaining, key=lambda i: role_scores[i])
    selected.append(first_idx)
    remaining.remove(first_idx)

    while len(selected) < min(k, n) and remaining:

        mmr_scores = []

        for i in remaining:

            relevance = role_scores[i]

            redundancy = max(
                cosine_similarity(
                    embeddings[i].reshape(1, -1),
                    embeddings[j].reshape(1, -1)
                )[0][0]
                for j in selected
            )

            mmr = (
                lambda_param * relevance
                - (1 - lambda_param) * redundancy
            )

            mmr_scores.append((i, mmr))

        best_idx = max(mmr_scores, key=lambda x: x[1])[0]

        selected.append(best_idx)
        remaining.remove(best_idx)

    return sorted(selected)

def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

import gc

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )
    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]
    selected_indices = doc2vec_mmr_summarization(sentences, roles)
    ranked_text = " ".join([sentences[i] for i in selected_indices])

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    del reference
    del judgement
    del sentences
    del roles
    del selected_indices
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# RST based 

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import  AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
nltk.download("punkt")
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


import torch
import numpy as np
from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()

def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)
ROLE_PRIORITY = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "RLC": 0.55,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PRE_NOT_RELIED": 0.4,
    "PREAMBLE": 0.3,
    "NONE": 0.2
}


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

def identify_discourse_relations(sentences):

    relations = []

    markers = {

        # ---------------------------
        # ELABORATION / ADDITION
        # ---------------------------
        'elaboration': [
            'furthermore', 'moreover', 'additionally', 'in addition',
            'besides', 'apart from that', 'coupled with',
            'what is more', 'it is further submitted',
            'it is further contended', 'it is also argued',
            'it was further argued', 'it is also pertinent to note',
            'it is relevant to note', 'it is significant to note',
            'another aspect is', 'it is worth mentioning',
            'it is worthwhile to note', 'it is important to note',
            'it must also be noted', 'it may also be noted',
            'likewise', 'similarly'
        ],
    
        # ---------------------------
        # CONTRAST / DISTINCTION
        # ---------------------------
        'contrast': [
            'however', 'but', 'nevertheless', 'nonetheless',
            'on the contrary', 'on the other hand',
            'in contrast', 'yet', 'whereas',
            'be that as it may', 'having said that',
            'at the same time', 'though', 'although',
            'even though', 'despite', 'in spite of',
            'conversely', 'per contra',
            'distinguished from', 'is distinguishable from',
            'cannot be equated with'
        ],
    
        # ---------------------------
        # CAUSE / REASONING
        # ---------------------------
        'cause': [
            'because', 'since', 'as', 'inasmuch as',
            'for the reason that', 'owing to',
            'due to', 'in view of', 'in light of',
            'having regard to', 'considering that',
            'given that', 'for this reason',
            'for the foregoing reasons',
            'insofar as', 'in as much as',
            'by reason of'
        ],
    
        # ---------------------------
        # RESULT / CONSEQUENCE
        # ---------------------------
        'result': [
            'therefore', 'thus', 'hence', 'consequently',
            'accordingly', 'as a result',
            'in consequence', 'in view thereof',
            'in view of the above',
            'for the reasons stated above',
            'it follows that',
            'this court holds that',
            'we hold that', 'we are of the view that',
            'we are of the opinion that',
            'it is clear that',
            'it is evident that'
        ],
    
        # ---------------------------
        # ISSUE FRAMING
        # ---------------------------
        'issue': [
            'the question that arises',
            'the issue before us',
            'the point for determination',
            'the short question',
            'the core issue',
            'the principal issue',
            'the controversy involved',
            'the moot question',
            'the matter in issue',
            'the point that falls for consideration'
        ],
    
        # ---------------------------
        # ARGUMENT ATTRIBUTION
        # ---------------------------
        'argument': [
            'it is contended', 'it was contended',
            'it is submitted', 'it was submitted',
            'learned counsel submitted',
            'learned counsel contended',
            'according to the appellant',
            'according to the respondent',
            'the appellant submits',
            'the respondent contends',
            'it is argued', 'it was argued',
            'the contention of the appellant',
            'the contention of the respondent'
        ],
    
        # ---------------------------
        # PRECEDENT CITATION SIGNAL
        # ---------------------------
        'precedent': [
            'in the case of',
            'in state of',  # as in State of X vs Y
            'reliance is placed on',
            'placed reliance upon',
            'as held in',
            'as observed in',
            'as laid down in',
            'as decided in',
            'as ruled in',
            'it was held in',
            'this court in',
            'the apex court in'
        ],
    
        # ---------------------------
        # CLARIFICATION / LIMITATION
        # ---------------------------
        'clarification': [
            'for the avoidance of doubt',
            'it is clarified that',
            'it is made clear that',
            'needless to say',
            'it may be clarified',
            'it must be clarified',
            'without prejudice to',
            'subject to',
            'provided that'
        ],
    
        # ---------------------------
        # CONCLUSION / DISPOSAL
        # ---------------------------
        'conclusion': [
            'in conclusion',
            'in the result',
            'the appeal is allowed',
            'the appeal is dismissed',
            'the petition is allowed',
            'the petition is dismissed',
            'ordered accordingly',
            'no order as to costs',
            'the matter is remitted',
            'the impugned order is set aside',
            'stands disposed of'
        ],
    
        # ---------------------------
        # LATIN LEGAL MAXIMS
        # ---------------------------
        'latin': [
            'prima facie',
            'ipso facto',
            'inter alia',
            'per se',
            'mutatis mutandis',
            'suo motu',
            'res judicata',
            'actus reus',
            'mens rea',
            'a fortiori',
            'pari materia',
            'sub silentio',
            'obiter dicta',
            'ratio decidendi'
        ]
    }

    for i, sent in enumerate(sentences):
        sent_lower = sent.lower()

        for rel_type, markers_list in markers.items():
            if any(f" {marker} " in f" {sent_lower} " for marker in markers_list):
                relations.append((i, rel_type))

    return relations

def rst_based_summarization(
        sentences,
        roles,
        ratio=0.25,
        min_len=8,
        max_len=30
    ):

    n = len(sentences)
    if n < 5:
        return " ".join(sentences)

    summary_size = estimate_summary_length(
        sentences, ratio, min_len, max_len
    )

    # -----------------------
    # Identify discourse relations
    # -----------------------
    relations = identify_discourse_relations(sentences)

    # Simplified nucleus detection:
    # sentences WITHOUT weak discourse markers are treated as nucleus
    satellite_indices = {i for i, _ in relations}
    nucleus_indices = [i for i in range(n) if i not in satellite_indices]

    # -----------------------
    # Sentence scoring
    # -----------------------
    scores = []

    for i in range(n):
        role_score = ROLE_PRIORITY.get(roles[i], 0)
        is_nucleus = i in nucleus_indices

        score = role_score * (2.0 if is_nucleus else 1.0)
        scores.append((i, score))

    # -----------------------
    # Select top sentences
    # -----------------------
    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    selected_indices = [idx for idx, _ in ranked[:summary_size]]

    # Preserve original order
    selected_indices.sort()

    summary = " ".join([sentences[i] for i in selected_indices])

    return summary

   
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
import gc


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):

    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )
    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]
    
    ranked_text = rst_based_summarization(
        sentences,
        roles,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )
    
    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    
    del reference
    del judgement
    del sentences
    del roles
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# Ensemble model

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()

def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)

ROLE_PRIORITY = {"ANALYSIS": 1.0, "RATIO": 1.0, "RPC": 0.95, "FAC": 0.9, "ISSUE": 0.85, "RLC": 0.8, "PRE_RELIED": 0.7, "STA": 0.6, "RLC": 0.55, "ARG_PETITIONER": 0.5, "ARG_RESPONDENT": 0.5, "PRE_NOT_RELIED": 0.40, "PREAMBLE": 0.3, "NONE": 0.2 }

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
def lexrank(sentences, similarity_threshold=0.15):
    n = len(sentences)
    
    if n == 0:
        return []

    # TF-IDF representation
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Cosine similarity matrix
    similarity_matrix = cosine_similarity(tfidf_matrix)

    # Remove weak edges
    similarity_matrix[similarity_matrix < similarity_threshold] = 0

    # Build graph
    graph = nx.from_numpy_array(similarity_matrix)

    # PageRank
    scores = nx.pagerank(graph)

    # Return sorted scores
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

from sklearn.decomposition import TruncatedSVD

def lsa_summarization(sentences, n_components=100):
    n = len(sentences)

    if n == 0:
        return []

    # TF-IDF representation
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Limit components
    n_components = min(n_components, tfidf_matrix.shape[1] - 1)

    if n_components <= 0:
        return [(i, 0.0) for i in range(n)]

    # SVD
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    reduced_matrix = svd.fit_transform(tfidf_matrix)

    # Sentence importance = vector norm in reduced space
    scores = np.linalg.norm(reduced_matrix, axis=1)

    return sorted(
        [(i, score) for i, score in enumerate(scores)],
        key=lambda x: x[1],
        reverse=True
    )
def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))


def ensemble_summarization(sentences, roles):
    n = len(sentences)

    if n == 0:
        return []

    # -----------------------------------
    # Dynamically estimate summary length
    # -----------------------------------
    k = estimate_summary_length(sentences)

    # ------------------------
    # Method 1: LexRank
    # ------------------------
    lexrank_scores = lexrank(sentences)

    # ------------------------
    # Method 2: LSA
    # ------------------------
    lsa_scores = lsa_summarization(sentences)

    # ------------------------
    # Method 3: Role-based
    # ------------------------
    role_scores = [(i, ROLE_PRIORITY.get(role, 0))
                   for i, role in enumerate(roles)]

    # ------------------------
    # Safe Normalization
    # ------------------------
    def normalize(scores):
        scores_dict = dict(scores)
        if not scores_dict:
            return {}

        max_val = max(scores_dict.values())
        if max_val == 0:
            return {k: 0 for k in scores_dict}

        return {k: v / max_val for k, v in scores_dict.items()}

    lexrank_norm = normalize(lexrank_scores)
    lsa_norm = normalize(lsa_scores)
    role_norm = normalize(role_scores)

    # ------------------------
    # Weighted Combination
    # ------------------------
    combined = {}

    for i in range(n):
        combined[i] = (
            0.4 * lexrank_norm.get(i, 0) +
            0.3 * lsa_norm.get(i, 0) +
            0.3 * role_norm.get(i, 0)
        )

    # ------------------------
    # Select Top-k
    # ------------------------
    top_k = sorted(
        combined.items(),
        key=lambda x: x[1],
        reverse=True
    )[:min(k, n)]

    selected_indices = sorted([idx for idx, _ in top_k])

    return selected_indices

def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))
from nltk import sent_tokenize
import gc 


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

import gc
for doc in tqdm(dataset, desc="Evaluating"):

    # -----------------------
    # Reference Summary
    # -----------------------
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )
    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]

    selected_indices = ensemble_summarization(sentences, roles)
    ranked_text = " ".join([sentences[i] for i in selected_indices])
    
    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    del reference
    del judgement
    del sentences
    del roles
    del selected_indices
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# Legal Concept Graph

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
from transformers import AutoModelForSeq2SeqLM
nltk.download("punkt")
nltk.download("stopwords")
import spacy

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)
def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()

def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)
ROLE_PRIORITY = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "RLC": 0.55,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PRE_NOT_RELIED": 0.4,
    "PREAMBLE": 0.3,
    "NONE": 0.2
}

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


STATUTE_PATTERN = re.compile(r"""
    (?:s\.|sec(?:tion)?s?\.?)\s*\d+[A-Za-z]?(?:[\s,]*(?:and)?\s*\d+[A-Za-z]?)*
    | S\.?\s*\d+[A-Za-z]?                                                       
    | Section\s+\d+[A-Za-z]?(?:\(\d+\))*                                        
    | Sec\.?\s*\d+[A-Za-z]?                                                      
    | Article\s+\d+[A-Za-z]?(?:\(\d+\))*                                         
    | Art\.?\s*\d+[A-Za-z]?                                                      
    | Act\s+\d+[A-Za-z]?\s+of\s+\d{3,5}                                          
""", re.IGNORECASE | re.VERBOSE)

def legal_concept_graph(sentences,
                        roles,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    n = len(sentences)
    k = estimate_summary_length(sentences, ratio, min_len, max_len)

    legal_entities = []

    for sent in sentences:
        doc = nlp(sent)
        entities = []

        # PERSON entities
        entities.extend([ent.text for ent in doc.ents if ent.label_ == 'PERSON'])

        # Statutory references (using strong pattern)
        statutes = STATUTE_PATTERN.findall(sent)
        entities.extend(statutes)

        legal_entities.append(list(set(entities)))  # remove duplicates

    # ---------------------------
    # Build Concept Graph
    # ---------------------------
    G = nx.Graph()

    for i in range(n):
        G.add_node(f'sent_{i}', type='sentence', role=roles[i])

    all_concepts = set()
    for ents in legal_entities:
        all_concepts.update(ents)

    for concept in all_concepts:
        G.add_node(concept, type='concept')

    for i, ents in enumerate(legal_entities):
        for ent in ents:
            G.add_edge(f'sent_{i}', ent)

    # ---------------------------
    # Sentence Scoring
    # ---------------------------
    scores = []

    max_role = max(ROLE_PRIORITY.values()) if ROLE_PRIORITY else 1
    max_degree = max([G.degree(f'sent_{i}') for i in range(n)]) if n > 0 else 1
    if max_degree == 0:
        max_degree = 1

    for i in range(n):

        role_score = ROLE_PRIORITY.get(roles[i], 0) / max_role
        degree_score = G.degree(f'sent_{i}') / max_degree

        # Slightly boost statutory-heavy sentences
        statute_count = len(STATUTE_PATTERN.findall(sentences[i]))
        statute_boost = min(statute_count / 3, 1)  # cap boost

        final_score = (
            0.5 * role_score +
            0.3 * degree_score +
            0.2 * statute_boost
        )

        scores.append((i, final_score))

    selected = sorted(scores,
                      key=lambda x: x[1],
                      reverse=True)[:min(k, n)]

    selected_indices = sorted([i for i, _ in selected])

    return selected_indices


def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
import gc


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )
    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]
    selected_indices = legal_concept_graph(sentences, roles)
    ranked_text = " ".join([sentences[i] for i in selected_indices])

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )
    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    
    del reference
    del judgement
    del sentences
    del roles
    del selected_indices
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()


print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# HTPosum

In [ ]:
import sys
import os

# ============================================================
# BARTScore Path
# ============================================================
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")


# ============================================================
# IMPORTS
# ============================================================

import json
import gc
import re
import math
import torch
import evaluate
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import defaultdict

from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    pipeline
)

from sentence_transformers import SentenceTransformer
from bart_score import BARTScorer

import nltk

nltk.download("punkt")
nltk.download("stopwords")

# ============================================================
# STOPWORDS
# ============================================================

stop_words = set(stopwords.words("english"))

# ============================================================
# DATASET
# ============================================================

file_path = (
    "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"
)

with open(file_path, "r") as f:
    dataset = json.load(f)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_dataset = dataset[:5000]
test_dataset = dataset[9500:]

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

# ============================================================
# ABSTRACTION MODEL
# ============================================================

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/"
    "distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/"
    "distilbart-9500-output"
)

model.eval()

# ============================================================
# EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# BARTScore
# ============================================================

bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# ============================================================
# FACTCC
# ============================================================

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    factcc_model_name
)

factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

# ============================================================
# SBERT
# ============================================================

sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device=device
)

# ============================================================
# PREPROCESS
# ============================================================

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

def build_pseudo_sections(
    sentences,
    num_sections=4
):

    N = len(sentences)

    chunk_size = math.ceil(
        N / num_sections
    )

    sections = []

    start = 0

    while start < N:

        end = min(
            start + chunk_size,
            N
        )

        sections.append(
            list(range(start, end))
        )

        start = end

    return sections

class TripletPositionEmbedding(nn.Module):

    def __init__(
        self,
        dim,
        max_depth=5,
        max_sections=50,
        max_sentences=2000
    ):

        super().__init__()

        self.depth_emb = nn.Embedding(
            max_depth,
            dim
        )

        self.section_emb = nn.Embedding(
            max_sections,
            dim
        )

        self.sent_emb = nn.Embedding(
            max_sentences,
            dim
        )

    def forward(
        self,
        depth,
        section_pos,
        sent_pos
    ):

        return (
            self.depth_emb(depth)
            +
            self.section_emb(section_pos)
            +
            self.sent_emb(sent_pos)
        ) / 3.0

class GraphSAGELayer(nn.Module):

    def __init__(
        self,
        hidden_dim
    ):

        super().__init__()

        self.W1 = nn.Linear(
            hidden_dim,
            hidden_dim
        )

        self.W2 = nn.Linear(
            hidden_dim,
            hidden_dim
        )

        self.norm = nn.LayerNorm(
            hidden_dim
        )

    def forward(
        self,
        x,
        adj
    ):

        degree = (
            adj.sum(
                dim=-1,
                keepdim=True
            ) + 1e-8
        )

        neigh = (
            torch.matmul(adj, x)
            / degree
        )

        h = (
            self.W1(x)
            +
            self.W2(neigh)
        )

        h = F.relu(h)

        return self.norm(
            h + x
        )
# ============================================================
# SUMMARY LENGTH
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )

    
# ============================================================
# CLEANED LEGALSUMMNET
# ============================================================

class HTPosum(nn.Module):

    def __init__(
        self,
        device,
        hidden_dim=768,
        gnn_layers=2
    ):

        super().__init__()

        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        )

        self.bert = AutoModel.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        ).to(device)
   
        for param in self.bert.parameters():
            param.requires_grad = False
        
        self.bert.config.use_cache = False
        self.triplet_emb = (
            TripletPositionEmbedding(
                hidden_dim
            )
        )

        self.gnn_layers = nn.ModuleList(
            [
                GraphSAGELayer(
                    hidden_dim
                )
                for _ in range(
                    gnn_layers
                )
            ]
        )

        self.dropout = nn.Dropout(0.2)

        self.classifier = nn.Sequential(
            nn.Linear(
                hidden_dim * 3,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(
                512,
                1
            )
        )

    # ========================================================
    # ENCODE SENTENCES
    # ========================================================

    def encode_sentences(
        self,
        sentences,
        batch_size=16
    ):
    
        embeddings = []
    
        for i in range(0, len(sentences), batch_size):
    
            batch = sentences[i:i+batch_size]
    
            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=96
            ).to(self.device)
    
            with torch.set_grad_enabled(self.training):
    
                outputs = self.bert(**inputs)
    
                cls = outputs.last_hidden_state[:, 0, :]
    
            embeddings.append(cls.detach() if not self.training else cls)
    
            del inputs
            del outputs
    
            torch.cuda.empty_cache()
    
        return torch.cat(embeddings, dim=0)

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        sentences
    ):
    
        sent_emb = self.encode_sentences(
            sentences
        )
    
        N = len(sentences)
    
        sections = build_pseudo_sections(
            sentences
        )
    
        section_embs = []
    
        for sec in sections:
    
            section_embs.append(
                sent_emb[sec].mean(
                    dim=0
                )
            )
    
        section_embs = torch.stack(
            section_embs
        )
    
        global_emb = sent_emb.mean(
            dim=0,
            keepdim=True
        )
    
        nodes = torch.cat(
            [
                sent_emb,
                section_embs,
                global_emb
            ],
            dim=0
        )
        depths = []
        section_ids = []
        sent_ids = []
    
        for sec_idx, sec in enumerate(sections):
    
            for local_idx, sent_idx in enumerate(sec):
    
                depths.append(2)
                section_ids.append(sec_idx)
                sent_ids.append(local_idx)
    
        for sec_idx in range(len(sections)):
    
            depths.append(1)
            section_ids.append(sec_idx)
            sent_ids.append(0)
    
        depths.append(0)
        section_ids.append(0)
        sent_ids.append(0)
        
        depths = torch.tensor(
            depths,
            device=self.device
        )
    
        section_ids = torch.tensor(
            section_ids,
            device=self.device
        )
    
        sent_ids = torch.tensor(
            sent_ids,
            device=self.device
        )
    
        pos_emb = self.triplet_emb(
            depths,
            section_ids,
            sent_ids
        )
    
        nodes = nodes + pos_emb
        total_nodes = (
            N+len(sections)+1
        )
    
        adj = torch.zeros(
            total_nodes,
            total_nodes,
            device=self.device
        )
        adj.fill_diagonal_(1)

        global_node = total_nodes - 1

        for sec_idx, sec in enumerate(sections):
        
            sec_node = N + sec_idx
        
            for sent_idx in sec:
        
                adj[sent_idx, sec_node] = 1
                adj[sec_node, sent_idx] = 1

        for sec_idx in range(
            len(sections)
        ):
    
            sec_node = N + sec_idx
    
            adj[sec_node, global_node] = 1
            adj[global_node, sec_node] = 1

        for layer in self.gnn_layers:

            nodes = layer(
                nodes,
                adj
            )
        sent_nodes = nodes[:N]

        sec_nodes = nodes[
            N:
            N + len(sections)
        ]
    
        global_repr = nodes[-1]
        features = []

        for sec_idx, sec in enumerate(sections):
    
            for sent_idx in sec:
    
                features.append(
                    torch.cat(
                        [
                            sent_nodes[sent_idx],
                            sec_nodes[sec_idx],
                            global_repr
                        ]
                    )
                )
    
        features = torch.stack(
            features
        )
    
        logits = self.classifier(
            features
        ).squeeze(-1)
    
        return logits

# ============================================================
# SOFT LABELS
# ============================================================

def create_sentence_labels(
    sentences,
    reference,
    model
):
    was_training = model.training

    model.eval()

    with torch.no_grad():

        sent_emb = model.encode_sentences(
            sentences
        )

        ref_sentences = sent_tokenize(
            reference
        )

        if len(ref_sentences) == 0:

            return torch.zeros(
                len(sentences)
            )

        ref_emb = model.encode_sentences(
            ref_sentences
        )

        sims = F.cosine_similarity(
            sent_emb.unsqueeze(1),
            ref_emb.unsqueeze(0),
            dim=-1
        )

        scores = sims.max(
            dim=1
        ).values

        threshold = scores.mean()

        labels = (
            scores >= threshold
        ).float()

    if was_training:
        model.train()

    return labels
  

# ============================================================
# TRAINING
# ============================================================

def train_htposum(
    model,
    dataset,
    oracle_labels,
    epochs=3,
    lr=2e-5,
    lambda_ext=0.7
):

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=lr
    )

    bce_loss_fn = nn.BCEWithLogitsLoss()

    model.train()

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=torch.cuda.is_available()
    )

    for epoch in range(epochs):

        total_loss = 0
        count = 0

        for idx, doc in enumerate(
            tqdm(
                dataset,
                desc=f"Epoch {epoch+1}"
            )
        ):

            # =============================================
            # SENTENCES
            # =============================================

            if isinstance(
                doc["judgement_sent"][0],
                dict
            ):

                sentences = [
                    s["sentence"]
                    for s in doc["judgement_sent"]
                ]

            else:

                sentences = doc["judgement_sent"]

            if len(sentences) < 2:
                continue

            # =============================================
            # PRECOMPUTED LABELS
            # =============================================

            labels = oracle_labels[idx].to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):

                logits = model(
                    sentences
                )

                loss = bce_loss_fn(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            total_loss += loss.item()

            count += 1

            del logits
            del loss

            torch.cuda.empty_cache()
            gc.collect()

        avg_loss = total_loss / max(count, 1)

        print(
            f"Epoch {epoch+1} "
            f"Loss: {avg_loss:.4f}"
        )

    model.eval()

# ============================================================
# INFERENCE
# ============================================================

def htposum_select(
    sentences,
    model,
    num_sentences=10
):

    model.eval()

    with torch.no_grad():

        logits = model(
            sentences
        )
        scores = logits.cpu().numpy()

    top_idx = np.argsort(
        scores
    )[-num_sentences:]

    top_idx = sorted(
        top_idx
    )

    selected = [
        sentences[i]
        for i in top_idx
    ]

    return " ".join(selected)

# ============================================================
# CHUNKING
# ============================================================

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

# ============================================================
# GENERATE SUMMARY
# ============================================================

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

# ============================================================
# FACTCC
# ============================================================

def factcc_chunked_fast(
    pred,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():

            logits = factcc_model(**inputs).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))


# ============================================================
# BUILD MODEL
# ============================================================

htposum_model = HTPosum(
    device
).to(device)

# ============================================================
# PRECOMPUTE ORACLE LABELS (RUN ONCE)
# ============================================================

oracle_labels = []

htposum_model.eval()

for doc in tqdm(
    train_dataset,
    desc="Building Oracle Labels"
):

    if isinstance(
        doc["judgement_sent"][0],
        dict
    ):
        sentences = [
            s["sentence"]
            for s in doc["judgement_sent"]
        ]
    else:
        sentences = doc["judgement_sent"]

    reference = " ".join(
        doc["headnote_sent"]
    )

    labels = create_sentence_labels(
        sentences,
        reference,
        htposum_model
    )

    oracle_labels.append(
        labels.cpu()
    )

# ============================================================
# TRAIN MODEL
# ============================================================

train_htposum(
    htposum_model,
    train_dataset,
    oracle_labels,
    epochs=3,
    lr=2e-5,
    lambda_ext=0.7
)

# ============================================================
# EVALUATION
# ============================================================

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []

bart_scores = []
factcc_scores = []

# ============================================================
# TEST LOOP
# ============================================================

for doc in tqdm(
    test_dataset,
    desc="Evaluating"
):

    # ========================================================
    # REFERENCE
    # ========================================================

    reference = " ".join(
        doc["headnote_sent"]
    )

    # ========================================================
    # JUDGEMENT
    # ========================================================

    sentences = doc["judgement_sent"]

    judgement = " ".join(
        preprocess(s)
        for s in sentences
    )

    # ========================================================
    # EXTRACTIVE SELECTION
    # ========================================================

    num_select = estimate_summary_length(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    ranked_text = htposum_select(
        sentences=sentences,
        model=htposum_model,
        num_sentences=num_select
    )

    # ========================================================
    # ABSTRACT GENERATION
    # ========================================================

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # ========================================================
    # ROUGE
    # ========================================================

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )

    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # ========================================================
    # BLEU
    # ========================================================

    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )

    bleu_scores.append(
        bleu_doc["bleu"]
    )

    # ========================================================
    # METEOR
    # ========================================================

    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )

    meteor_scores.append(
        meteor_doc["meteor"]
    )

    # ========================================================
    # BERTScore
    # ========================================================

    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
        bert_doc["f1"][0]
    )

    # ========================================================
    # BARTScore
    # ========================================================

    bart_scores.append(
        bart_scorer.score(
            [pred],
            [reference]
        )[0]
    )

    # ========================================================
    # FACTCC
    # ========================================================

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(
            pred,
            judgement_chunks
        )
    )

    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del pred
    del ranked_text
    del judgement_chunks

    torch.cuda.empty_cache()
    gc.collect()

# ============================================================
# FINAL RESULTS
# ============================================================

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# RankSUM

In [ ]:

# ============================================================
# STANDARD LIBRARIES
# ============================================================

import os
import sys
import gc
import re
import json
import math

# ============================================================
# CORE LIBRARIES
# ============================================================

import numpy as np
from tqdm import tqdm

# ============================================================
# PYTORCH
# ============================================================

import torch
import torch.nn.functional as F

# ============================================================
# NLP LIBRARIES
# ============================================================

import nltk
import spacy
import evaluate

from nltk.corpus import stopwords
from nltk import sent_tokenize, word_tokenize

# ============================================================
# TRANSFORMERS
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)

from sentence_transformers import SentenceTransformer

# ============================================================
# SCIKIT-LEARN
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# BARTScore
# ============================================================

sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

from bart_score import BARTScorer

# ============================================================
# INITIAL SETUP
# ============================================================

nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================

DATASET_PATH = (
    "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"
)

with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)


# dataset = dataset[400:]

# ============================================================
# LOAD SUMMARIZATION MODEL
# ============================================================

MODEL_PATH = (
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH
).to(device)

summarizer_model.eval()

# ============================================================
# LOAD EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# LOAD BARTScore
# ============================================================

bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# ============================================================
# LOAD FACTCC
# ============================================================

FACTCC_MODEL = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    FACTCC_MODEL
)

factcc_model = AutoModelForSequenceClassification.from_pretrained(
    FACTCC_MODEL
).to(device)

factcc_model.eval()

# ============================================================
# LOAD SBERT
# ============================================================

sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device=device
)

# ============================================================
# TEXT PREPROCESSING
# ============================================================

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ============================================================
# RANKSUM IMPLEMENTATION
# ============================================================

from gensim import corpora
from gensim.models import LdaModel

# ============================================================
# TOPIC RANK EXTRACTOR
# ============================================================

def build_lda_model(dataset_texts, num_topics=10):

    processed_docs = []

    for text in dataset_texts:

        tokens = [
            w.lower()
            for w in word_tokenize(text)
            if w.isalpha() and w.lower() not in stop_words
        ]

        processed_docs.append(tokens)

    dictionary = corpora.Dictionary(processed_docs)

    corpus = [
        dictionary.doc2bow(doc)
        for doc in processed_docs
    ]

    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        passes=10,
        random_state=42
    )

    return lda_model, dictionary


# ============================================================
# DOCUMENT TOPIC VECTOR
# ============================================================

def get_topic_vector(text, lda_model, dictionary, num_topics=10):

    tokens = [
        w.lower()
        for w in word_tokenize(text)
        if w.isalpha() and w.lower() not in stop_words
    ]

    bow = dictionary.doc2bow(tokens)

    topic_dist = lda_model.get_document_topics(
        bow,
        minimum_probability=0
    )

    vec = np.zeros(num_topics)

    for tid, prob in topic_dist:
        vec[tid] = prob

    return vec


# ============================================================
# TOPIC RANK SCORES
# ============================================================
def topic_rank_scores(
    sentences,
    lda_model,
    dictionary,
    num_topics=10
):

    document_text = " ".join(sentences)

    doc_topic = get_topic_vector(
        document_text,
        lda_model,
        dictionary,
        num_topics
    )

    scores = []

    for sent in sentences:

        tokens = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha()
        ]

        word_vectors = []

        for token in tokens:

            bow = dictionary.doc2bow([token])

            topic_dist = lda_model.get_document_topics(
                bow,
                minimum_probability=0
            )

            vec = np.zeros(num_topics)

            for tid, prob in topic_dist:
                vec[tid] = prob

            word_vectors.append(vec)

        if len(word_vectors) == 0:
            sent_topic = np.zeros(num_topics)
        else:
            sent_topic = np.mean(
                word_vectors,
                axis=0
            )

        dist = np.linalg.norm(
            sent_topic - doc_topic
        )

        scores.append(-dist)

    scores = np.array(scores)

    scores = (
        scores - scores.min()
    ) / (
        scores.max() - scores.min() + 1e-8
    )

    return scores


# ============================================================
# SBERT SEMANTIC RANK EXTRACTOR
# ============================================================

def semantic_rank_scores(sentences):

    sent_emb = sbert_model.encode(
        sentences,
        convert_to_numpy=True
    )

    doc_emb = np.mean(sent_emb, axis=0)

    scores = []

    for i in range(len(sentences)):

        remaining = np.delete(
            sent_emb,
            i,
            axis=0
        )

        if len(remaining) == 0:
            scores.append(0)
            continue

        new_doc_emb = np.mean(
            remaining,
            axis=0
        )

        dist = np.linalg.norm(
            doc_emb - new_doc_emb
        )

        scores.append(dist)

    scores = np.array(scores)

    scores = (
        scores - scores.min()
    ) / (
        scores.max() - scores.min() + 1e-8
    )

    return scores


import networkx as nx
def keyword_rank_scores(sentences):

    processed = []

    for sent in sentences:

        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha()
            and w.lower() not in stop_words
        ]

        processed.append(words)

    G = nx.Graph()

    window = 4

    for words in processed:

        for i in range(len(words)):

            for j in range(
                i + 1,
                min(i + window, len(words))
            ):

                w1 = words[i]
                w2 = words[j]

                if G.has_edge(w1, w2):
                    G[w1][w2]["weight"] += 1
                else:
                    G.add_edge(
                        w1,
                        w2,
                        weight=1
                    )

    if len(G.nodes()) == 0:
        return np.zeros(len(sentences))

    keyword_scores = nx.pagerank(
        G,
        weight="weight"
    )

    sentence_scores = []

    for words in processed:

        score = sum(
            keyword_scores.get(w, 0)
            for w in words
        )

        sentence_scores.append(score)

    sentence_scores = np.array(
        sentence_scores
    )

    sentence_scores = (
        sentence_scores
        -
        sentence_scores.min()
    ) / (
        sentence_scores.max()
        -
        sentence_scores.min()
        +
        1e-8
    )

    return sentence_scores


# ============================================================
# POSITION RANK EXTRACTOR
# ============================================================

def position_rank_scores(sentences):

    N = len(sentences)

    scores = np.array([
        N - i
        for i in range(N)
    ])

    scores = (
        scores - scores.min()
    ) / (
        scores.max() - scores.min() + 1e-8
    )

    return scores


# ============================================================
# NOVELTY FILTER
# ============================================================

def ngrams(tokens, n):

    return set(
        zip(
            *[
                tokens[i:]
                for i in range(n)
            ]
        )
    )

def is_novel(
    candidate,
    selected,
    t1=0.80,
    t2=3
):

    if len(selected) == 0:
        return True

    cand_emb = sbert_model.encode(
        [candidate],
        convert_to_numpy=True
    )[0]

    cand_tokens = word_tokenize(
        candidate.lower()
    )

    cand_bi = ngrams(
        cand_tokens,
        2
    )

    cand_tri = ngrams(
        cand_tokens,
        3
    )

    for sent in selected:

        sent_emb = sbert_model.encode(
            [sent],
            convert_to_numpy=True
        )[0]

        sim = cosine_similarity(
            [cand_emb],
            [sent_emb]
        )[0][0]

        sent_tokens = word_tokenize(
            sent.lower()
        )

        sent_bi = ngrams(
            sent_tokens,
            2
        )

        sent_tri = ngrams(
            sent_tokens,
            3
        )

        overlap = (
            len(
                cand_bi.intersection(
                    sent_bi
                )
            )
            +
            len(
                cand_tri.intersection(
                    sent_tri
                )
            )
        )

        if (
            sim >= t1
            and overlap >= t2
        ):
            return False

    return True

def score_to_rank(scores):

    order = np.argsort(scores)

    ranks = np.empty_like(order)

    ranks[order] = np.arange(
        1,
        len(scores)+1
    )

    return ranks
# ============================================================
# FINAL RANKSUM SUMMARIZER
# ============================================================

def ranksum_summarize(
    sentences,
    lda_model,
    dictionary,
    num_sentences=10,
    alpha=0.30,
    beta=0.20,
    gamma=0.35,
    delta=0.15
):

    if len(sentences) == 0:
        return ""

    # --------------------------------------------------------
    # INDIVIDUAL SCORES
    # --------------------------------------------------------

    RT = score_to_rank(
        topic_rank_scores(
            sentences,
            lda_model,
            dictionary
        )
    )
    
    RK = score_to_rank(
        keyword_rank_scores(
            sentences
        )
    )
    
    RE = score_to_rank(
        semantic_rank_scores(
            sentences
        )
    )
    
    RP = score_to_rank(
        position_rank_scores(
            sentences
        )
    )

    # --------------------------------------------------------
    # FINAL FUSION
    # --------------------------------------------------------

    final_scores = (
        alpha * RT
        + beta * RK
        + gamma * RE
        + delta * RP
    )

    ranked_idx = np.argsort(final_scores)[::-1]

    selected = []

    for idx in ranked_idx:

        sent = sentences[idx]

        if is_novel(sent, selected):

            selected.append(sent)

        if len(selected) >= num_sentences:
            break

    return " ".join(selected)


# ============================================================
# SUMMARY LENGTH ESTIMATION
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )



# ============================================================
# TEXT CHUNKING
# ============================================================

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

# ============================================================
# CHUNK-BASED SUMMARY GENERATION
# ============================================================

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

# ============================================================
# FACTCC EVALUATION
# ============================================================

def factcc_chunked_fast(
    prediction,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            prediction,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():

            logits = factcc_model(
                **inputs
            ).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))


all_docs = []

for doc in dataset:

    text = " ".join(
        s["sentence"]
        for s in doc["judgement_roles"]
    )

    all_docs.append(text)

lda_model, dictionary = build_lda_model(
    all_docs,
    num_topics=10
)

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []
bart_scores = []

factcc_scores = []

for doc in tqdm(dataset, desc="RankSum"):

    # ----------------------------------------------------
    # REFERENCE SUMMARY
    # ----------------------------------------------------

    reference = " ".join(
            doc["headnote_sent"]
    )

    # ----------------------------------------------------
    # SOURCE JUDGMENT
    # ----------------------------------------------------

    sentences = [
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and s.get("sentence")
    ]
    judgement = " ".join(sentences)

    num_select = estimate_summary_length(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )
    
    structured_input = ranksum_summarize(
        sentences=sentences,
        lda_model=lda_model,
        dictionary=dictionary,
        num_sentences=num_select,
        alpha=0.30,
        beta=0.20,
        gamma=0.35,
        delta=0.15
    )

    # ----------------------------------------------------
    # SUMMARY GENERATION
    # ----------------------------------------------------

    prediction = generate_summary_chunked(
            structured_input,
            tokenizer,
            summarizer_model
    )

    # ----------------------------------------------------
    # ROUGE
    # ----------------------------------------------------

    rouge_result = rouge.compute(
            predictions=[prediction],
            references=[reference]
    )

    rouge1_scores.append(
            rouge_result["rouge1"]
    )

    rouge2_scores.append(
            rouge_result["rouge2"]
    )

    rougel_scores.append(
            rouge_result["rougeL"]
    )

    rougelsum_scores.append(
            rouge_result["rougeLsum"]
    )

    # ----------------------------------------------------
    # BLEU
    # ----------------------------------------------------

    bleu_result = bleu.compute(
            predictions=[prediction],
            references=[[reference]]
    )

    bleu_scores.append(
            bleu_result["bleu"]
    )

    # ----------------------------------------------------
    # METEOR
    # ----------------------------------------------------

    meteor_result = meteor.compute(
            predictions=[prediction],
            references=[reference]
    )

    meteor_scores.append(
            meteor_result["meteor"]
    )

    # ----------------------------------------------------
    # BERTScore
    # ----------------------------------------------------

    bert_result = bertscore.compute(
        predictions=[prediction],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
            bert_result["f1"][0]
    )

    # ----------------------------------------------------
    # BARTScore
    # ----------------------------------------------------

    bart_scores.append(
        bart_scorer.score(
                [prediction],
                [reference]
        )[0]
    )

    # ----------------------------------------------------
    # FACTCC
    # ----------------------------------------------------

    judgement_chunks = chunk_text(
            judgement,
            tokenizer,
            max_tokens=900,
            stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(
                prediction,
                judgement_chunks
        )
    )

    # ----------------------------------------------------
    # MEMORY CLEANUP
    # ----------------------------------------------------

    torch.cuda.empty_cache()

    gc.collect()

# ========================================================
# PRINT RESULTS
# ========================================================

print(f"ROUGE-1    : {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2    : {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L    : {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM : {np.mean(rougelsum_scores):.4f}")
print(f"BLEU       : {np.mean(bleu_scores):.4f}")
print(f"METEOR     : {np.mean(meteor_scores):.4f}")
print(f"BERTScore  : {np.mean(bert_scores):.4f}")
print(f"BARTScore  : {np.mean(bart_scores):.4f}")
print(f"FactCC     : {np.mean(factcc_scores):.4f}")


# Bayesian Optimizer based

In [ ]:
import sys
import os

# ============================================================
# BARTScore Path
# ============================================================
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")


# ============================================================
# IMPORTS
# ============================================================

import json
import gc
import re
import math
import torch
import evaluate
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    pipeline
)

from sentence_transformers import SentenceTransformer
from bart_score import BARTScorer

import nltk

nltk.download("punkt")
nltk.download("stopwords")

# ============================================================
# STOPWORDS
# ============================================================

stop_words = set(stopwords.words("english"))

# ============================================================
# DATASET
# ============================================================

file_path = (
    "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgements/data12.json"
)

with open(file_path, "r") as f:
    dataset = json.load(f)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_dataset = dataset[:5000]
test_dataset = dataset[9500:]

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

# ============================================================
# ABSTRACTION MODEL
# ============================================================

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/"
    "distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/"
    "distilbart-9500-output"
)

model.eval()

# ============================================================
# EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# BARTScore
# ============================================================

bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# ============================================================
# FACTCC
# ============================================================

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    factcc_model_name
)

factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

# ============================================================
# SBERT
# ============================================================

sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device=device
)

# ============================================================
# NLI MODEL
# ============================================================

nli_model_name = "facebook/bart-large-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(
    nli_model_name
)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    nli_model_name
).to(device)

nli_model.eval()

# ============================================================
# QA MODEL
# ============================================================

qa_model = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    device=0 if torch.cuda.is_available() else -1
)

# ============================================================
# PREPROCESS
# ============================================================

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ============================================================
# SUMMARY LENGTH
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

# ============================================================
# BAYESIAN OPTIMIZATION SCORE FUSION
# Knowledge-Based Systems 2023 style baseline
# ============================================================

LEGAL_TERMS = {
    # ========================================================
    # Court, judges, parties, proceedings
    # ========================================================
    "court", "courts", "judge", "judges", "justice", "bench",
    "division_bench", "constitution_bench", "larger_bench",
    "single_judge", "chief_justice", "registrar", "registry",
    "tribunal", "authority", "commission", "forum", "board",
    "supreme_court", "high_court", "district_court",
    "sessions_court", "trial_court", "magistrate", "subordinate_court",
    "appellant", "respondent", "petitioner", "defendant",
    "plaintiff", "accused", "complainant", "prosecutrix",
    "victim", "deceased", "informant", "witness", "intervenor",
    "applicant", "opposite_party", "state", "union",
    "prosecution", "defence", "counsel", "advocate",
    "learned_counsel", "senior_counsel", "amicus_curiae",
    "solicitor_general", "attorney_general", "public_prosecutor",
    "standing_counsel",

    # ========================================================
    # Judgment, order, appeal, petition
    # ========================================================
    "judgment", "judgement", "order", "decree", "ruling",
    "decision", "finding", "findings", "observation", "direction",
    "relief", "prayer", "pleading", "pleadings", "affidavit",
    "counter_affidavit", "rejoinder", "application", "petition",
    "writ_petition", "civil_appeal", "criminal_appeal",
    "special_leave_petition", "slp", "transfer_petition",
    "review_petition", "curative_petition", "contempt_petition",
    "interlocutory_application", "miscellaneous_application",
    "revision", "reference", "review", "appeal", "cross_appeal",
    "cross_objection", "leave", "special_leave", "certificate",
    "remand", "remitted", "disposed", "dismissed", "allowed",
    "partly_allowed", "set_aside", "quashed", "affirmed",
    "modified", "upheld", "reversed", "restored", "stayed",
    "interim_order", "final_order", "impugned_order",
    "speaking_order", "ex_parte", "in_limine",

    # ========================================================
    # Constitutional law and writ jurisdiction
    # ========================================================
    "constitution", "constitutional", "article", "articles",
    "fundamental_right", "fundamental_rights", "directive_principles",
    "basic_structure", "preamble", "constitutional_validity",
    "constitutional_bench", "judicial_review", "separation_of_powers",
    "federalism", "legislative_competence", "colourable_legislation",
    "repugnancy", "reasonable_restriction", "arbitrariness",
    "manifest_arbitrariness", "equality", "equal_protection",
    "natural_justice", "due_process", "procedure_established_by_law",
    "right_to_life", "personal_liberty", "free_speech",
    "freedom_of_speech", "religious_freedom", "minority_rights",
    "reservation", "scheduled_caste", "scheduled_tribe",
    "other_backward_classes", "obc", "sc", "st",
    "article_14", "article_19", "article_21", "article_32",
    "article_136", "article_141", "article_142", "article_226",
    "article_227", "article_300a", "writ", "writs",
    "habeas_corpus", "mandamus", "certiorari", "prohibition",
    "quo_warranto", "public_interest_litigation", "pil",
    "locus_standi", "maintainability", "alternative_remedy",
    "efficacious_remedy", "territorial_jurisdiction",

    # ========================================================
    # Statutes, rules, interpretation
    # ========================================================
    "act", "acts", "statute", "statutory", "section", "sections",
    "subsection", "clause", "subclause", "proviso", "explanation",
    "schedule", "rule", "rules", "regulation", "regulations",
    "notification", "ordinance", "amendment", "enactment",
    "legislation", "delegated_legislation", "bye_laws",
    "circular", "guideline", "scheme", "policy", "government_order",
    "gazette", "interpretation", "construction", "literal_rule",
    "purposive_interpretation", "mischief_rule", "harmonious_construction",
    "ejusdem_generis", "noscitur_a_sociis", "expressio_unius",
    "casus_omissus", "prospective", "retrospective",
    "prospective_overruling", "ultra_vires", "intra_vires",

    # ========================================================
    # Precedent and legal reasoning
    # ========================================================
    "precedent", "precedents", "case_law", "citation", "citations",
    "relied", "distinguished", "overruled", "referred", "followed",
    "binding_precedent", "persuasive_precedent", "ratio",
    "ratio_decidendi", "obiter", "obiter_dicta", "dictum",
    "per_incuriam", "sub_silentio", "stare_decisis",
    "doctrine", "principle", "settled_law", "question_of_law",
    "question_of_fact", "mixed_question", "substantial_question_of_law",
    "issue", "issues", "facts", "material_facts", "cause_of_action",
    "burden", "onus", "standard_of_proof", "preponderance",
    "beyond_reasonable_doubt",

    # ========================================================
    # Criminal law
    # ========================================================
    "criminal", "crime", "offence", "offense", "accused",
    "conviction", "convicted", "acquittal", "acquitted",
    "sentence", "sentencing", "punishment", "imprisonment",
    "life_imprisonment", "death_penalty", "capital_punishment",
    "fine", "bail", "anticipatory_bail", "regular_bail",
    "default_bail", "custody", "judicial_custody", "police_custody",
    "remand", "arrest", "detention", "illegal_detention",
    "fir", "first_information_report", "charge", "charges",
    "charge_sheet", "chargesheet", "framing_of_charge",
    "trial", "investigation", "cognizance", "summons",
    "warrant", "non_bailable_warrant", "nbw", "proclamation",
    "confession", "dying_declaration", "motive", "intention",
    "mens_rea", "actus_reus", "common_intention", "common_object",
    "conspiracy", "abetment", "attempt", "murder", "culpable_homicide",
    "homicide", "assault", "rape", "sexual_assault", "kidnapping",
    "abduction", "theft", "robbery", "dacoity", "cheating",
    "forgery", "criminal_breach_of_trust", "defamation",
    "corruption", "prevention_of_corruption", "ndps", "pocso",
    "juvenile", "juvenile_justice", "probation", "parole",
    "furlough", "victim_compensation",

    # ========================================================
    # Criminal statutes and provisions
    # ========================================================
    "ipc", "indian_penal_code", "crpc", "code_of_criminal_procedure",
    "evidence_act", "indian_evidence_act", "pocso_act",
    "ndps_act", "prevention_of_corruption_act", "arms_act",
    "sc_st_act", "uapa", "unlawful_activities",
    "section_302", "section_304", "section_307", "section_376",
    "section_420", "section_498a", "section_34", "section_149",
    "section_120b", "section_161", "section_164", "section_313",
    "section_438", "section_439",

    # ========================================================
    # Civil procedure and civil law
    # ========================================================
    "civil", "suit", "plaint", "written_statement",
    "replication", "issues_framed", "injunction", "temporary_injunction",
    "permanent_injunction", "mandatory_injunction", "declaration",
    "specific_performance", "damages", "compensation",
    "mesne_profits", "possession", "title", "ownership",
    "easement", "partition", "specific_relief", "limitation",
    "delay", "laches", "res_judicata", "constructive_res_judicata",
    "estoppel", "waiver", "acquiescence", "lis_pendens",
    "cpc", "code_of_civil_procedure", "order", "rule",
    "order_7_rule_11", "order_39", "order_41", "order_47",
    "section_9", "section_11", "section_96", "section_100",
    "execution", "decree_holder", "judgment_debtor",

    # ========================================================
    # Evidence
    # ========================================================
    "evidence", "proof", "proved", "unproved", "disproved",
    "admissible", "inadmissible", "admissibility", "relevancy",
    "documentary_evidence", "oral_evidence", "circumstantial_evidence",
    "direct_evidence", "hearsay", "presumption", "rebuttal",
    "exhibit", "material_exhibit", "cross_examination",
    "examination_in_chief", "re_examination", "hostile_witness",
    "expert_evidence", "medical_evidence", "forensic",
    "dna", "post_mortem", "panchnama", "seizure", "recovery",
    "chain_of_custody", "burden_of_proof", "standard_of_proof",

    # ========================================================
    # Contract, commercial, arbitration
    # ========================================================
    "contract", "agreement", "breach", "breach_of_contract",
    "consideration", "offer", "acceptance", "promise",
    "void", "voidable", "valid", "invalid", "fraud",
    "misrepresentation", "coercion", "undue_influence",
    "specific_performance", "damages", "liquidated_damages",
    "indemnity", "guarantee", "bank_guarantee", "tender",
    "bid", "license", "lease", "concession", "commercial",
    "arbitration", "arbitral", "arbitrator", "tribunal",
    "arbitral_award", "award", "conciliation", "mediation",
    "settlement", "negotiation", "arbitration_act",
    "section_9", "section_11", "section_34", "section_37",

    # ========================================================
    # Property, land, acquisition
    # ========================================================
    "property", "land", "immovable_property", "movable_property",
    "transfer", "sale", "gift", "mortgage", "lease", "licence",
    "license", "tenancy", "tenant", "landlord", "rent",
    "eviction", "possession", "adverse_possession",
    "acquisition", "land_acquisition", "compensation",
    "market_value", "award", "collector", "requisition",
    "mutation", "revenue_record", "khata", "khasra",
    "jamabandi", "patta", "nazul", "ceiling", "urban_land",

    # ========================================================
    # Service, labour, employment
    # ========================================================
    "service", "employment", "employee", "employer",
    "workman", "labour", "industrial_dispute", "termination",
    "dismissal", "removal", "suspension", "reinstatement",
    "back_wages", "seniority", "promotion", "appointment",
    "recruitment", "reservation", "disciplinary_proceedings",
    "departmental_inquiry", "misconduct", "charge_memo",
    "show_cause_notice", "natural_justice", "pension",
    "gratuity", "provident_fund", "wages", "minimum_wages",
    "compassionate_appointment",

    # ========================================================
    # Tax, finance, insolvency
    # ========================================================
    "tax", "taxation", "income_tax", "gst", "customs",
    "excise", "service_tax", "assessment", "assessee",
    "reassessment", "deduction", "exemption", "penalty",
    "interest", "refund", "demand", "show_cause_notice",
    "adjudication", "appellate_authority", "tribunal",
    "insolvency", "bankruptcy", "ibc", "corporate_debtor",
    "financial_creditor", "operational_creditor",
    "resolution_professional", "cirp", "liquidation",
    "moratorium", "nclt", "nclat",

    # ========================================================
    # Corporate, company, securities
    # ========================================================
    "company", "corporation", "director", "shareholder",
    "board_of_directors", "shares", "securities", "sebi",
    "merger", "amalgamation", "winding_up", "oppression",
    "mismanagement", "memorandum", "articles_of_association",
    "corporate_governance", "fraudulent_transaction",

    # ========================================================
    # Family, personal law, succession
    # ========================================================
    "marriage", "divorce", "maintenance", "alimony",
    "custody", "guardianship", "adoption", "domestic_violence",
    "dowry", "matrimonial", "restitution", "cruelty",
    "desertion", "succession", "inheritance", "will",
    "probate", "letters_of_administration", "partition",
    "coparcenary", "hindu_law", "muslim_law", "personal_law",

    # ========================================================
    # Administrative, public law, governance
    # ========================================================
    "administrative", "administrative_law", "public_authority",
    "statutory_authority", "government", "state_government",
    "central_government", "local_authority", "municipality",
    "panchayat", "policy", "scheme", "tender", "blacklisting",
    "licence", "permission", "sanction", "approval",
    "discretion", "malafide", "mala_fide", "arbitrary",
    "irrational", "proportionality", "legitimate_expectation",
    "promissory_estoppel", "public_duty", "public_function",
    "public_interest",

    # ========================================================
    # Environment, human rights, PIL
    # ========================================================
    "environment", "environmental", "pollution",
    "forest", "wildlife", "ecology", "sustainable_development",
    "precautionary_principle", "polluter_pays",
    "public_trust_doctrine", "ngt", "national_green_tribunal",
    "human_rights", "right_to_education", "right_to_privacy",
    "right_to_information", "rti", "food_security",
    "public_health", "rehabilitation", "compensation",

    # ========================================================
    # Common judicial phrases / outcome markers
    # ========================================================
    "held", "held_that", "observed", "observed_that",
    "submitted", "contended", "argued", "urged",
    "pleaded", "averred", "alleged", "denied",
    "admitted", "proved", "established", "considered",
    "concluded", "directed", "ordered", "declared",
    "prima_facie", "inter_alia", "in_view_of",
    "in_the_result", "accordingly", "therefore",
    "hence", "however", "nevertheless", "whereas",
    "aforesaid", "herein", "hereinafter", "thereof",
    "thereunder", "aforesaid", "impugned", "instant_case",
    "present_case", "facts_and_circumstances",
    "ends_of_justice", "interest_of_justice",
    "miscarriage_of_justice", "abuse_of_process",
    "abuse_of_process_of_law",

    # ========================================================
    # Latin / legal maxims often seen in judgments
    # ========================================================
    "audi_alteram_partem", "nemo_judex_in_causa_sua",
    "actus_curiae_neminem_gravabit", "actus_reus",
    "mens_rea", "res_ipsa_loquitur", "res_judicata",
    "sub_judice", "suo_motu", "suo_moto", "de_novo",
    "ex_parte", "ex_post_facto", "ex_gratia",
    "in_rem", "in_personam", "inter_se", "inter_partes",
    "pari_materia", "per_se", "per_curiam",
    "functus_officio", "coram_non_judice",
    "sine_die", "sine_qua_non"
}


def normalize_array(values):

    values = np.asarray(
        values,
        dtype=np.float32
    )

    if len(values) == 0:
        return values

    min_v = values.min()
    max_v = values.max()

    if max_v - min_v < 1e-8:
        return np.zeros_like(values)

    return (
        values - min_v
    ) / (
        max_v - min_v + 1e-8
    )


def get_tokens(text):

    return [
        w.lower()
        for w in word_tokenize(text)
        if w.isalpha()
        and w.lower() not in stop_words
    ]

def compute_legal_term_score(
    sentence,
    lambda1=0.5
):

    text = sentence.lower()

    text_clean = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text_clean = re.sub(
        r"\s+",
        " ",
        text_clean
    ).strip()

    tokens = [
        w
        for w in word_tokenize(text_clean)
        if w.isalpha()
    ]

    token_set = set(
        tokens
    )

    count = 0

    for term in LEGAL_TERMS:

        if "_" in term:

            phrase = term.replace(
                "_",
                " "
            )

            if re.search(
                r"\b" + re.escape(phrase) + r"\b",
                text_clean
            ):

                count += 1

        else:

            if term in token_set:

                count += 1

    return (
        float(count > 0)
        +
        lambda1 * count
    )

def get_ngrams_simple(tokens, n):

    return set(
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    )


def lcs_length_simple(x, y):

    m = len(x)
    n = len(y)

    dp = [
        [0] * (n + 1)
        for _ in range(m + 1)
    ]

    for i in range(1, m + 1):

        for j in range(1, n + 1):

            if x[i - 1] == y[j - 1]:

                dp[i][j] = dp[i - 1][j - 1] + 1

            else:

                dp[i][j] = max(
                    dp[i - 1][j],
                    dp[i][j - 1]
                )

    return dp[m][n]


def rouge_f1_summary(prediction, reference):

    pred_tokens = get_tokens(
        prediction
    )

    ref_tokens = get_tokens(
        reference
    )

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    def f1_score(pred_units, ref_units):

        if len(pred_units) == 0 or len(ref_units) == 0:
            return 0.0

        overlap = len(
            pred_units & ref_units
        )

        precision = overlap / max(
            len(pred_units),
            1
        )

        recall = overlap / max(
            len(ref_units),
            1
        )

        if precision + recall == 0:
            return 0.0

        return (
            2 * precision * recall
        ) / (
            precision + recall
        )

    pred_uni = set(
        pred_tokens
    )

    ref_uni = set(
        ref_tokens
    )

    pred_bi = get_ngrams_simple(
        pred_tokens,
        2
    )

    ref_bi = get_ngrams_simple(
        ref_tokens,
        2
    )

    r1 = f1_score(
        pred_uni,
        ref_uni
    )

    r2 = f1_score(
        pred_bi,
        ref_bi
    )

    lcs = lcs_length_simple(
        pred_tokens,
        ref_tokens
    )

    precision_l = lcs / max(
        len(pred_tokens),
        1
    )

    recall_l = lcs / max(
        len(ref_tokens),
        1
    )

    if precision_l + recall_l == 0:
        rl = 0.0
    else:
        rl = (
            2 * precision_l * recall_l
        ) / (
            precision_l + recall_l
        )

    return (
        r1 + r2 + rl
    ) / 3.0


def get_document_sentences(doc):

    if (
        "judgement_sent" in doc
        and len(doc["judgement_sent"]) > 0
    ):

        if isinstance(
            doc["judgement_sent"][0],
            dict
        ):

            return [
                s["sentence"]
                for s in doc["judgement_sent"]
                if "sentence" in s
            ]

        return doc["judgement_sent"]

    if "judgement_roles" in doc:

        return [
            s["sentence"]
            for s in doc["judgement_roles"]
            if isinstance(s, dict)
            and "sentence" in s
        ]

    return []


def get_reference_summary(doc):

    return " ".join(
        doc["headnote_sent"]
    )


def build_training_texts(dataset):

    texts = []

    for doc in dataset:

        sentences = get_document_sentences(
            doc
        )

        text = " ".join(
            preprocess(s)
            for s in sentences
        )

        if text.strip():
            texts.append(text)

    return texts


# ============================================================
# BAYESIAN OPTIMIZATION WEIGHT SEARCH
# ============================================================

def expected_improvement(
    X_candidates,
    gp,
    best_y,
    xi=0.01
):

    mu, sigma = gp.predict(
        X_candidates,
        return_std=True
    )

    sigma = sigma.reshape(
        -1
    )

    improvement = mu - best_y - xi

    Z = improvement / (
        sigma + 1e-8
    )

    ei = (
        improvement * norm.cdf(Z)
        +
        sigma * norm.pdf(Z)
    )

    ei[sigma == 0.0] = 0.0

    return ei


def sample_weight_vectors(
    n_samples,
    n_features
):

    return np.random.dirichlet(
        alpha=np.ones(n_features),
        size=n_samples
    )


# ============================================================
# BO SCORE FUSION SUMMARIZER
# ============================================================

class BOScoreFusionSummarizer:

    def __init__(
        self,
        max_features=20000,
        n_topics=10,
        use_sbert=False,
        random_state=42
    ):

        self.max_features = max_features
        self.n_topics = n_topics
        self.use_sbert = use_sbert
        self.random_state = random_state

        self.tfidf_vectorizer = TfidfVectorizer(
            stop_words="english",
            max_features=max_features,
            ngram_range=(1, 2),
            min_df=1
        )

        self.count_vectorizer = CountVectorizer(
            stop_words="english",
            max_features=max_features,
            min_df=1
        )

        self.lda_model = LatentDirichletAllocation(
            n_components=n_topics,
            random_state=random_state,
            learning_method="batch"
        )

        self.weights = None

        self.feature_names = [
            "tfidf_score",
            "centroid_similarity",
            "position_score",
            "length_score",
            "legal_term_score",
            "numeric_score",
            "lda_topic_score",
            "opening_similarity"
        ]

        if use_sbert:
            self.feature_names.append(
                "sbert_similarity"
            )

    # ========================================================
    # FIT CORPUS MODELS
    # ========================================================

    def fit_corpus_models(
        self,
        train_dataset
    ):

        train_texts = build_training_texts(
            train_dataset
        )

        self.tfidf_vectorizer.fit(
            train_texts
        )

        count_matrix = self.count_vectorizer.fit_transform(
            train_texts
        )

        self.lda_model.fit(
            count_matrix
        )

    # ========================================================
    # SENTENCE FEATURE EXTRACTION
    # ========================================================

    def compute_features(
        self,
        sentences
    ):

        cleaned_sentences = [
            preprocess(s)
            for s in sentences
        ]

        n = len(
            cleaned_sentences
        )

        if n == 0:
            return np.zeros(
                (
                    0,
                    len(self.feature_names)
                ),
                dtype=np.float32
            )

        document_text = " ".join(
            cleaned_sentences
        )

        # ----------------------------------------------------
        # TF-IDF sentence score
        # ----------------------------------------------------

        sent_tfidf = self.tfidf_vectorizer.transform(
            cleaned_sentences
        )

        doc_tfidf = self.tfidf_vectorizer.transform(
            [document_text]
        )

        tfidf_score = np.asarray(
            sent_tfidf.sum(axis=1)
        ).reshape(-1)

        # ----------------------------------------------------
        # Sentence-to-document centroid similarity
        # ----------------------------------------------------

        centroid_similarity = cosine_similarity(
            sent_tfidf,
            doc_tfidf
        ).reshape(-1)

        # ----------------------------------------------------
        # Position score
        # Earlier sentences are generally important
        # ----------------------------------------------------

        if n == 1:

            position_score = np.ones(
                n
            )

        else:

            position_score = np.array(
                [
                    1.0 - (i / (n - 1))
                    for i in range(n)
                ],
                dtype=np.float32
            )

        # ----------------------------------------------------
        # Length score
        # Avoid too short sentences; normalize sentence length
        # ----------------------------------------------------

        token_lengths = np.array(
            [
                len(get_tokens(s))
                for s in cleaned_sentences
            ],
            dtype=np.float32
        )

        length_score = np.clip(
            token_lengths,
            1,
            60
        )


        # ----------------------------------------------------
        # Legal term score
        # Supports both single-word and multi-word legal terms
        # ----------------------------------------------------
        
        legal_term_score = np.array(
            [
                compute_legal_term_score(
                    sent,
                    lambda1=0.5
                )
                for sent in cleaned_sentences
            ],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Numeric score
        # ----------------------------------------------------

        numeric_score = np.array(
            [
                len(
                    re.findall(
                        r"\d+",
                        sent
                    )
                )
                for sent in cleaned_sentences
            ],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # LDA topic similarity
        # ----------------------------------------------------

        sent_count = self.count_vectorizer.transform(
            cleaned_sentences
        )

        doc_count = self.count_vectorizer.transform(
            [document_text]
        )

        sent_topics = self.lda_model.transform(
            sent_count
        )

        doc_topic = self.lda_model.transform(
            doc_count
        )

        lda_topic_score = cosine_similarity(
            sent_topics,
            doc_topic
        ).reshape(-1)

        # ----------------------------------------------------
        # Opening/title-like similarity
        # Legal documents may not have title.
        # Use first three sentences as pseudo-title/context.
        # ----------------------------------------------------

        opening_text = " ".join(
            cleaned_sentences[:3]
        )

        opening_tfidf = self.tfidf_vectorizer.transform(
            [opening_text]
        )

        opening_similarity = cosine_similarity(
            sent_tfidf,
            opening_tfidf
        ).reshape(-1)

        feature_matrix = np.vstack(
            [
                normalize_array(tfidf_score),
                normalize_array(centroid_similarity),
                normalize_array(position_score),
                normalize_array(length_score),
                normalize_array(legal_term_score),
                normalize_array(numeric_score),
                normalize_array(lda_topic_score),
                normalize_array(opening_similarity)
            ]
        ).T

        # ----------------------------------------------------
        # Optional SBERT semantic similarity
        # Slower, but closer to semantic score fusion.
        # ----------------------------------------------------

        if self.use_sbert:

            sent_emb = sbert_model.encode(
                sentences,
                convert_to_numpy=True,
                show_progress_bar=False
            )

            doc_emb = sent_emb.mean(
                axis=0,
                keepdims=True
            )

            sbert_similarity = cosine_similarity(
                sent_emb,
                doc_emb
            ).reshape(-1)

            feature_matrix = np.column_stack(
                [
                    feature_matrix,
                    normalize_array(sbert_similarity)
                ]
            )

        return feature_matrix

    # ========================================================
    # SCORE SENTENCES
    # ========================================================

    def score_sentences(
        self,
        sentences,
        weights=None
    ):

        if weights is None:
            weights = self.weights

        if weights is None:
            weights = np.ones(
                len(self.feature_names)
            ) / len(self.feature_names)

        feature_matrix = self.compute_features(
            sentences
        )

        if len(feature_matrix) == 0:
            return np.array([])

        scores = np.dot(
            feature_matrix,
            weights
        )

        return scores

    # ========================================================
    # SUMMARIZE USING WEIGHTED FUSION
    # ========================================================

    def summarize(
        self,
        sentences,
        num_sentences=10,
        weights=None
    ):

        if len(sentences) == 0:
            return ""

        if len(sentences) <= num_sentences:
            return " ".join(sentences)

        scores = self.score_sentences(
            sentences,
            weights=weights
        )

        ranked_idx = np.argsort(
            scores
        )[::-1]

        selected_idx = ranked_idx[
            :num_sentences
        ]

        selected_idx = sorted(
            selected_idx
        )

        selected_sentences = [
            sentences[i]
            for i in selected_idx
        ]

        return " ".join(
            selected_sentences
        )

    # ========================================================
    # OBJECTIVE FUNCTION FOR BO
    # ========================================================

    def evaluate_weights(
        self,
        val_dataset,
        weights,
        max_docs=200
    ):

        scores = []

        eval_docs = val_dataset[
            :max_docs
        ]

        for doc in eval_docs:

            sentences = get_document_sentences(
                doc
            )

            reference = get_reference_summary(
                doc
            )

            if len(sentences) < 2:
                continue

            num_select = estimate_summary_length(
                sentences,
                ratio=0.25,
                min_len=8,
                max_len=30
            )

            extractive_summary = self.summarize(
                sentences,
                num_sentences=num_select,
                weights=weights
            )

            score = rouge_f1_summary(
                extractive_summary,
                reference
            )

            scores.append(
                score
            )

        if len(scores) == 0:
            return 0.0

        return float(
            np.mean(scores)
        )

    # ========================================================
    # BAYESIAN OPTIMIZATION OF FUSION WEIGHTS
    # ========================================================

    def optimize_weights(
        self,
        val_dataset,
        n_init=10,
        n_iter=25,
        candidate_pool=200,
        max_docs=200
    ):

        n_features = len(
            self.feature_names
        )

        X_obs = []
        y_obs = []

        # ----------------------------------------------------
        # Initial random Dirichlet weight samples
        # ----------------------------------------------------

        init_weights = sample_weight_vectors(
            n_init,
            n_features
        )

        print(
            "\nInitializing Bayesian Optimization..."
        )

        for w in tqdm(
            init_weights,
            desc="BO Init"
        ):

            score = self.evaluate_weights(
                val_dataset,
                w,
                max_docs=max_docs
            )

            X_obs.append(
                w
            )

            y_obs.append(
                score
            )

        # ----------------------------------------------------
        # BO loop using Gaussian Process + Expected Improvement
        # ----------------------------------------------------

        kernel = (
            ConstantKernel(1.0)
            *
            Matern(
                length_scale=np.ones(n_features),
                nu=2.5
            )
            +
            WhiteKernel(noise_level=1e-5)
        )

        for it in range(n_iter):

            gp = GaussianProcessRegressor(
                kernel=kernel,
                normalize_y=True,
                random_state=self.random_state,
                n_restarts_optimizer=2
            )

            gp.fit(
                np.array(X_obs),
                np.array(y_obs)
            )

            candidates = sample_weight_vectors(
                candidate_pool,
                n_features
            )

            ei = expected_improvement(
                candidates,
                gp,
                best_y=max(y_obs)
            )

            best_candidate = candidates[
                int(np.argmax(ei))
            ]

            score = self.evaluate_weights(
                val_dataset,
                best_candidate,
                max_docs=max_docs
            )

            X_obs.append(
                best_candidate
            )

            y_obs.append(
                score
            )

            print(
                f"BO Iter {it + 1}/{n_iter} | "
                f"Score={score:.4f} | "
                f"Best={max(y_obs):.4f}"
            )

        best_idx = int(
            np.argmax(y_obs)
        )

        self.weights = np.array(
            X_obs[best_idx],
            dtype=np.float32
        )

        print(
            "\nBest BO Score:",
            y_obs[best_idx]
        )

        print(
            "Optimized Feature Weights:"
        )

        for name, weight in zip(
            self.feature_names,
            self.weights
        ):

            print(
                f"{name:25s}: {weight:.4f}"
            )

        return self.weights

    # ========================================================
    # FULL FIT
    # ========================================================

    def fit(
        self,
        train_dataset,
        val_dataset,
        n_init=10,
        n_iter=25,
        candidate_pool=200,
        max_docs=200
    ):

        print(
            "\nFitting corpus-level TF-IDF and LDA models..."
        )

        self.fit_corpus_models(
            train_dataset
        )

        self.optimize_weights(
            val_dataset=val_dataset,
            n_init=n_init,
            n_iter=n_iter,
            candidate_pool=candidate_pool,
            max_docs=max_docs
        )
        
# ============================================================
# CHUNKING
# ============================================================

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

# ============================================================
# GENERATE SUMMARY
# ============================================================

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

# ============================================================
# FACTCC
# ============================================================

def factcc_chunked_fast(
    pred,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():

            logits = factcc_model(**inputs).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))


# ============================================================
# TRAIN / VALIDATION SPLIT FOR BAYESIAN OPTIMIZATION
# ============================================================

bo_train_dataset = train_dataset[:-300]
bo_val_dataset = train_dataset[-300:]

# ============================================================
# INITIALIZE BO SCORE FUSION BASELINE
# ============================================================

bo_summarizer = BOScoreFusionSummarizer(
    max_features=20000,
    n_topics=10,
    use_sbert=True,      # keep False for speed; True is slower
    random_state=42
)

bo_summarizer.fit(
    train_dataset=bo_train_dataset,
    val_dataset=bo_val_dataset,
    n_init=8,
    n_iter=20,
    candidate_pool=150,
    max_docs=200
)
# ============================================================
# EVALUATION
# ============================================================

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []

bart_scores = []
factcc_scores = []

# ============================================================
# TEST LOOP
# ============================================================

for doc in tqdm(
    test_dataset,
    desc="Evaluating"
):

    # ========================================================
    # REFERENCE
    # ========================================================

    reference = " ".join(
        doc["headnote_sent"]
    )

    # ========================================================
    # JUDGEMENT
    # ========================================================

    sentences = doc["judgement_sent"]

    judgement = " ".join(
        preprocess(s)
        for s in sentences
    )

    # ========================================================
    # EXTRACTIVE SELECTION
    # ========================================================

    num_select = estimate_summary_length(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    ranked_text = bo_summarizer.summarize(
        sentences=sentences,
        num_sentences=num_select
    )

    # ========================================================
    # ABSTRACT GENERATION
    # ========================================================

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # ========================================================
    # ROUGE
    # ========================================================

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )

    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # ========================================================
    # BLEU
    # ========================================================

    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )


    bleu_scores.append(
        bleu_doc["bleu"]
    )

    # ========================================================
    # METEOR
    # ========================================================

    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )

    meteor_scores.append(
        meteor_doc["meteor"]
    )

    # ========================================================
    # BERTScore
    # ========================================================

    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
        bert_doc["f1"][0]
    )

    # ========================================================
    # BARTScore
    # ========================================================

    bart_scores.append(
        bart_scorer.score(
            [pred],
            [reference]
        )[0]
    )

    # ========================================================
    # FACTCC
    # ========================================================

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(
            pred,
            judgement_chunks
        )
    )

    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del pred
    del ranked_text
    del judgement_chunks

    torch.cuda.empty_cache()
    gc.collect()

# ============================================================
# FINAL RESULTS
# ============================================================

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")





# GETS

In [ ]:
# ============================================================
# PURE TERMDIFFUSUM IMPLEMENTATION
# (NO RHETORICAL ROLES)
# ============================================================
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import os
import gc
import re
import json
import math
import torch
import random
import nltk
import evaluate
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import string
import networkx as nx

from nltk.stem import WordNetLemmatizer
from nltk import pos_tag


nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("omw-1.4")
from tqdm import tqdm
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)

from sklearn.metrics.pairwise import cosine_similarity

from bart_score import BARTScorer
from sentence_transformers import SentenceTransformer

# ============================================================
# NLTK
# ============================================================

nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

# ============================================================
# DEVICE
# ============================================================

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device0 = torch.device("cuda:0")
device1 = torch.device("cuda:1")

# ============================================================
# LOAD DATASET
# ============================================================

file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgements/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_dataset = dataset[:5000]
test_dataset = dataset[-500:]

print(f"Train Size: {len(train_dataset)}")
print(f"Test Size : {len(test_dataset)}")

# ============================================================
# LOAD ABSTRATIVE MODEL
# ============================================================

abstractive_model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device1)

abstractive_tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

abstractive_model.eval()

# ============================================================
# EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# BARTScore
# ============================================================

# bart_scorer = BARTScorer(
#     device=device,
#     checkpoint="facebook/bart-large-cnn"
# )
bart_scorer = BARTScorer(
    device=device1,
    checkpoint="facebook/bart-large-cnn"
)
# ============================================================
# FACTCC
# ============================================================

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    factcc_model_name
)

# factcc_model = AutoModelForSequenceClassification.from_pretrained(
#     factcc_model_name
# ).to(device)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device1)

factcc_model.eval()

# ============================================================
# SBERT
# ============================================================

# sbert_model = SentenceTransformer(
#     "all-mpnet-base-v2",
#     device=device
# )
sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device="cuda:1"
)

# LEGAL_TERMS = {
#     "court","judge","judgment","appeal","petitioner",
#     "respondent","section","article","constitution",
#     "tribunal","criminal","civil","evidence",
#     "conviction","sentence","writ","plaintiff",
#     "defendant","bench","order","act"
# }
LEGAL_TERMS = {
    # Core Judicial Terms
    "court","judge","bench","order","judgment","decree","ruling","decision","opinion",
    "petitioner","respondent","plaintiff","defendant","appellant","accused","complainant",

    # Constitutional & Statutory References
    "article","section","clause","subsection","provision","statute","act","code","rule","regulation",
    "constitution","amendment","fundamental_rights","directive_principles","preamble",

    # Institutions & Proceedings
    "tribunal","commission","authority","high_court","supreme_court","sessions_court",
    "writ","mandamus","certiorari","habeas_corpus","prohibition","quo_warranto",
    "appeal","revision","review","special_leave_petition","slp","interim_order",

    # Criminal & Civil Law
    "criminal","civil","offence","crime","charge","fir","investigation","prosecution",
    "conviction","acquittal","sentence","punishment","bail","remand","custody","trial",
    "evidence","witness","testimony","cross_examination","affidavit",

    # Procedural & Administrative
    "plea","petition","application","submission","argument","hearing","proceedings",
    "interim_relief","stay_order","injunction","contempt","costs","damages",
    "jurisdiction","maintainability","locus_standi","cause_of_action","ratio_decidendi","obiter_dicta",

    # Specialized Legal Vocabulary
    "precedent","case_law","citation","headnote","doctrine","principle","interpretation",
    "arbitration","mediation","conciliation","settlement","negotiation",
    "liability","negligence","tort","contract","agreement","breach"
}

def sentence_entropy(sentence):

    words = word_tokenize(sentence.lower())

    words = [
        w for w in words
        if w.isalpha()
    ]

    if len(words) == 0:
        return 0

    counts = Counter(words)

    total = len(words)

    entropy = 0

    for c in counts.values():

        p = c / total

        entropy -= p * math.log(p + 1e-12)

    return entropy

def legal_term_score(sentence):

    words = set(
        word_tokenize(sentence.lower())
    )

    count = sum(
        1 for w in words
        if w in LEGAL_TERMS
    )

    return (
        int(count > 0)
        +
        0.5 * count
    )

def position_score(idx, total):

    return math.exp(
        -((idx-total/2)**2)
        /
        (total/2 + 1e-8)
    )

def compute_sentence_weights(sentences):

    scores = []

    N = len(sentences)

    for i,s in enumerate(sentences):

        score = (
            sentence_entropy(s)
            +
            legal_term_score(s)
            +
            position_score(i,N)
        )

        scores.append(score)

    scores = np.array(scores)

    scores = (
        scores - scores.min()
    ) / (
        scores.max() - scores.min()
        + 1e-8
    )

    return torch.tensor(
        scores,
        dtype=torch.float
    )


# ============================================================
# GETS: GRAPH-BASED EXTRACTIVE TEXT SUMMARIZATION
# Sentence Scoring Scheme for Extractive Summary Generation
# ============================================================

lemmatizer = WordNetLemmatizer()


# ============================================================
# BASIC CONTRACTION EXPANSION
# ============================================================

CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'t": " not",
    "'ve": " have",
    "'m": " am"
}


def expand_contractions(text):

    text = text.lower()

    for contraction, expansion in CONTRACTIONS.items():

        text = text.replace(
            contraction,
            expansion
        )

    return text


# ============================================================
# GETS PREPROCESSING
# lowercase → contraction expansion → punctuation removal
# tokenization → lemmatization → stopword removal → POS filtering
# ============================================================

def gets_preprocess_sentence(
    sentence,
    keep_pos=True
):

    sentence = expand_contractions(
        sentence
    )

    sentence = re.sub(
        r"[^a-z0-9\s]",
        " ",
        sentence
    )

    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    ).strip()

    tokens = word_tokenize(
        sentence
    )

    cleaned = []

    for tok in tokens:

        tok = tok.lower().strip()

        if not tok:
            continue

        if tok in stop_words:
            continue

        if len(tok) <= 1:
            continue

        lemma = lemmatizer.lemmatize(
            tok
        )

        cleaned.append(
            lemma
        )

    if not keep_pos:

        return cleaned

    # --------------------------------------------------------
    # POS-based content-word filtering
    # GETS uses POS processing to retain informative words.
    # Here we keep nouns, proper nouns, verbs, adjectives.
    # --------------------------------------------------------

    try:

        tagged = pos_tag(
            cleaned
        )

        content_tokens = [
            word
            for word, tag in tagged
            if tag.startswith("NN")
            or tag.startswith("VB")
            or tag.startswith("JJ")
        ]

        if len(content_tokens) > 0:
            return content_tokens

        return cleaned

    except Exception:

        return cleaned


# ============================================================
# WORD-SENTENCE COUNT MATRIX
# Rows    = sentences
# Columns = words
# Values  = word frequency in sentence
# ============================================================

def build_word_sentence_matrix(
    processed_sentences
):

    vocab = sorted(
        list(
            set(
                word
                for sent in processed_sentences
                for word in sent
            )
        )
    )

    word_to_idx = {
        word: idx
        for idx, word in enumerate(vocab)
    }

    matrix = np.zeros(
        (
            len(processed_sentences),
            len(vocab)
        ),
        dtype=np.float32
    )

    for sent_idx, tokens in enumerate(processed_sentences):

        counts = Counter(
            tokens
        )

        for word, count in counts.items():

            if word in word_to_idx:

                matrix[
                    sent_idx,
                    word_to_idx[word]
                ] = count

    return matrix, vocab


# ============================================================
# JACCARD SENTENCE SIMILARITY
# Original GETS-style statistical relation between sentences
# ============================================================

def jaccard_similarity(
    tokens_a,
    tokens_b
):

    set_a = set(
        tokens_a
    )

    set_b = set(
        tokens_b
    )

    if len(set_a) == 0 and len(set_b) == 0:
        return 0.0

    union = set_a | set_b

    if len(union) == 0:
        return 0.0

    inter = set_a & set_b

    return len(inter) / len(union)


# ============================================================
# SENTENCE-SENTENCE SIMILARITY MATRIX
# ============================================================

def build_sentence_similarity_matrix(
    processed_sentences
):

    n = len(
        processed_sentences
    )

    sim_matrix = np.zeros(
        (
            n,
            n
        ),
        dtype=np.float32
    )

    for i in range(n):

        for j in range(i + 1, n):

            sim = jaccard_similarity(
                processed_sentences[i],
                processed_sentences[j]
            )

            sim_matrix[i, j] = sim
            sim_matrix[j, i] = sim

    return sim_matrix


# ============================================================
# GRAPH PREPARATION
# Sentence = node
# Edge = sentence relation
# Weight = similarity score
# Weak edges removed using percentile threshold
# ============================================================

def build_gets_sentence_graph(
    sim_matrix,
    percentile=60
):

    n = sim_matrix.shape[0]

    graph = nx.Graph()

    for i in range(n):

        graph.add_node(
            i
        )

    nonzero_sims = sim_matrix[
        sim_matrix > 0
    ]

    if len(nonzero_sims) == 0:

        return graph

    threshold = np.percentile(
        nonzero_sims,
        percentile
    )

    for i in range(n):

        for j in range(i + 1, n):

            weight = float(
                sim_matrix[i, j]
            )

            if weight > 0 and weight >= threshold:

                graph.add_edge(
                    i,
                    j,
                    weight=weight
                )

    return graph


# ============================================================
# GETS GRAPH-BASED SENTENCE SCORING
# rank(sentence_i) = sum incoming weights + sum outgoing weights
# For undirected graph, this is weighted degree.
# ============================================================

def gets_graph_sentence_scores(
    graph,
    n_sentences
):

    scores = np.zeros(
        n_sentences,
        dtype=np.float32
    )

    for node in range(n_sentences):

        if node not in graph:
            continue

        weighted_degree = 0.0

        for _, _, edge_data in graph.edges(
            node,
            data=True
        ):

            weighted_degree += float(
                edge_data.get(
                    "weight",
                    0.0
                )
            )

        scores[node] = weighted_degree

    if scores.max() - scores.min() < 1e-8:

        return np.ones_like(
            scores
        )

    scores = (
        scores - scores.min()
    ) / (
        scores.max() - scores.min() + 1e-8
    )

    return scores


# ============================================================
# GRAPH-BASED CLUSTERING POST-PROCESSING
# GETS reports clustered and non-clustered variants.
# Here we use graph connected components as lightweight
# graph-based clustering.
# ============================================================

def gets_graph_clusters(
    graph,
    n_sentences
):

    if graph.number_of_edges() == 0:

        return [
            [i]
            for i in range(n_sentences)
        ]

    clusters = [
        sorted(
            list(component)
        )
        for component in nx.connected_components(
            graph
        )
    ]

    covered = set(
        idx
        for cluster in clusters
        for idx in cluster
    )

    for i in range(n_sentences):

        if i not in covered:

            clusters.append(
                [i]
            )

    return clusters


# ============================================================
# SELECT SENTENCES FROM CLUSTERS
# Pick high-scoring sentences across graph clusters
# to reduce redundancy and improve coverage.
# ============================================================

def gets_clustered_selection(
    scores,
    clusters,
    num_sentences
):

    selected = []

    cluster_candidates = []

    for cluster in clusters:

        if len(cluster) == 0:
            continue

        best_idx = max(
            cluster,
            key=lambda x: scores[x]
        )

        cluster_score = scores[
            best_idx
        ]

        cluster_candidates.append(
            (
                best_idx,
                cluster_score
            )
        )

    cluster_candidates = sorted(
        cluster_candidates,
        key=lambda x: x[1],
        reverse=True
    )

    # --------------------------------------------------------
    # First pass: one best sentence per important cluster
    # --------------------------------------------------------

    for idx, _ in cluster_candidates:

        if idx not in selected:

            selected.append(
                idx
            )

        if len(selected) >= num_sentences:
            break

    # --------------------------------------------------------
    # Second pass: fill remaining slots by global graph score
    # --------------------------------------------------------

    if len(selected) < num_sentences:

        ranked = np.argsort(
            scores
        )[::-1]

        for idx in ranked:

            idx = int(idx)

            if idx not in selected:

                selected.append(
                    idx
                )

            if len(selected) >= num_sentences:
                break

    selected = sorted(
        selected
    )

    return selected


# ============================================================
# NON-CLUSTERED GETS SELECTION
# ============================================================

def gets_nonclustered_selection(
    scores,
    num_sentences
):

    ranked = np.argsort(
        scores
    )[::-1]

    selected = sorted(
        [
            int(idx)
            for idx in ranked[:num_sentences]
        ]
    )

    return selected


# ============================================================
# FINAL GETS EXTRACTIVE SELECTOR
# This returns extractive text to feed into DistilBART.
# ============================================================

def gets_extract(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30,
    clustered=True,
    edge_percentile=60
):

    if len(sentences) == 0:
        return ""

    if len(sentences) == 1:
        return sentences[0]

    num_select = estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

    num_select = min(
        num_select,
        len(sentences)
    )

    # --------------------------------------------------------
    # Preprocessing
    # --------------------------------------------------------

    processed_sentences = [
        gets_preprocess_sentence(
            sent,
            keep_pos=True
        )
        for sent in sentences
    ]

    # --------------------------------------------------------
    # Word-sentence statistical matrix
    # This is retained because GETS establishes relations
    # between words and sentences through statistical operations.
    # --------------------------------------------------------

    word_sentence_matrix, vocab = build_word_sentence_matrix(
        processed_sentences
    )

    # --------------------------------------------------------
    # Sentence similarity graph
    # --------------------------------------------------------

    sim_matrix = build_sentence_similarity_matrix(
        processed_sentences
    )

    graph = build_gets_sentence_graph(
        sim_matrix,
        percentile=edge_percentile
    )

    # --------------------------------------------------------
    # Graph-based sentence scoring
    # --------------------------------------------------------

    scores = gets_graph_sentence_scores(
        graph,
        len(sentences)
    )

    # --------------------------------------------------------
    # Fallback:
    # If graph is too sparse, use sentence-word frequency score.
    # --------------------------------------------------------

    if graph.number_of_edges() == 0:

        freq_scores = word_sentence_matrix.sum(
            axis=1
        )

        if freq_scores.max() - freq_scores.min() > 1e-8:

            scores = (
                freq_scores - freq_scores.min()
            ) / (
                freq_scores.max() - freq_scores.min() + 1e-8
            )

    # --------------------------------------------------------
    # Clustered or non-clustered GETS
    # --------------------------------------------------------

    if clustered:

        clusters = gets_graph_clusters(
            graph,
            len(sentences)
        )

        selected_indices = gets_clustered_selection(
            scores=scores,
            clusters=clusters,
            num_sentences=num_select
        )

    else:

        selected_indices = gets_nonclustered_selection(
            scores=scores,
            num_sentences=num_select
        )

    selected_sentences = [
        sentences[i]
        for i in selected_indices
    ]

    return " ".join(
        selected_sentences
    )

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        # inputs = tokenizer(
        #     chunk,
        #     return_tensors="pt",
        #     truncation=True,
        #     max_length=1024
        # ).to(device)
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device1)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

def factcc_chunked_fast(
    pred,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device1)

        with torch.no_grad():

            logits = factcc_model(**inputs).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))

# ============================================================
# EVALUATION
# ============================================================

# diffusion_model.eval()

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []
bart_scores = []
factcc_scores = []

for doc in tqdm(
    test_dataset,
    desc="Evaluating"
):

    reference = " ".join(
        doc["headnote_sent"]
    )

    document_sentences = doc["judgement_sent"]

    judgement = " ".join(
        document_sentences
    )

    target_len = estimate_summary_length(
        document_sentences
    )

    # ========================================================
    # EXTRACTIVE SUMMARY
    # ========================================================

    extractive_summary = gets_extract(
        sentences=document_sentences,
        ratio=0.25,
        min_len=8,
        max_len=30,
        clustered=False,
        edge_percentile=60
    )

    # extractive_summary = " ".join(
    #     extracted_sentences
    # )

    # ========================================================
    # ABSTRACTIVE SUMMARY
    # ========================================================

    pred = generate_summary_chunked(
        extractive_summary,
        abstractive_tokenizer,
        abstractive_model
    )

    # ========================================================
    # ROUGE
    # ========================================================

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )

    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # ========================================================
    # BLEU
    # ========================================================

    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )

    bleu_scores.append(
        bleu_doc["bleu"]
    )

    # ========================================================
    # METEOR
    # ========================================================

    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )

    meteor_scores.append(
        meteor_doc["meteor"]
    )

    # ========================================================
    # BERTSCORE
    # ========================================================

    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
        bert_doc["f1"][0]
    )

    # ========================================================
    # BARTSCORE
    # ========================================================

    bart_scores.append(
        bart_scorer.score(
            [pred],
            [reference]
        )[0]
    )

    # ========================================================
    # FACTCC
    # ========================================================

    judgement_chunks = chunk_text(
        judgement,
        abstractive_tokenizer
    )

    factcc_scores.append(
        factcc_chunked_fast(
            pred,
            judgement_chunks
        )
    )
    # del extracted_sentences
    del extractive_summary
    del pred
    del judgement
    del judgement_chunks

    gc.collect()
    torch.cuda.empty_cache()

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================ FINAL RESULTS ================\n")

print(f"ROUGE-1     : {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     : {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     : {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  : {np.mean(rougelsum_scores):.4f}")

print(f"BLEU        : {np.mean(bleu_scores):.4f}")
print(f"METEOR      : {np.mean(meteor_scores):.4f}")

print(f"BERTScore   : {np.mean(bert_scores):.4f}")
print(f"BARTScore   : {np.mean(bart_scores):.4f}")

print(f"FactCC      : {np.mean(factcc_scores):.4f}")

print("\n================================================")

# HHGraphSUM

In [ ]:
import sys
import os

# ============================================================
# BARTScore Path
# ============================================================
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")


# ============================================================
# IMPORTS
# ============================================================

import json
import gc
import re
import math
import torch
import evaluate
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    pipeline
)

from sentence_transformers import SentenceTransformer
from bart_score import BARTScorer

import nltk

nltk.download("punkt")
nltk.download("stopwords")

# ============================================================
# STOPWORDS
# ============================================================

stop_words = set(stopwords.words("english"))

# ============================================================
# DATASET
# ============================================================

file_path = (
    "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgements/data12.json"
)

with open(file_path, "r") as f:
    dataset = json.load(f)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_dataset = dataset[:5000]
test_dataset = dataset[9500:]

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

# ============================================================
# ABSTRACTION MODEL
# ============================================================

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()

# ============================================================
# EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# BARTScore
# ============================================================

bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# ============================================================
# FACTCC
# ============================================================

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    factcc_model_name
)

factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

# ============================================================
# SBERT
# ============================================================

sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device=device
)

# ============================================================
# PREPROCESS
# ============================================================

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ============================================================
# SUMMARY LENGTH
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )



# ============================================================
# HHGRAPHSUM IMPLEMENTATION
# Hierarchical Heterogeneous Graph Learning
# for Extractive Document Summarization
# ============================================================

# ============================================================
# TOKENIZATION HELPERS
# ============================================================

def simple_word_tokenize(text):

    tokens = [
        w.lower()
        for w in word_tokenize(text)
        if w.isalpha()
    ]

    return tokens


def get_sentences_from_doc(doc):

    if isinstance(doc["judgement_sent"][0], dict):

        sentences = [
            s["sentence"]
            for s in doc["judgement_sent"]
            if isinstance(s, dict)
            and "sentence" in s
        ]

    else:

        sentences = doc["judgement_sent"]

    return sentences


# ============================================================
# BUILD VOCABULARY FROM TRAINING DATA
# ============================================================

def build_hhgraphsum_vocab(
    dataset,
    min_freq=3,
    max_vocab_size=50000
):

    counter = Counter()

    for doc in tqdm(
        dataset,
        desc="Building HHGraphSum Vocabulary"
    ):

        sentences = get_sentences_from_doc(
            doc
        )

        for sent in sentences:

            counter.update(
                simple_word_tokenize(sent)
            )

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for word, freq in counter.most_common(
        max_vocab_size - 2
    ):

        if freq >= min_freq:

            vocab[word] = len(vocab)

    return vocab


# ============================================================
# ORACLE LABELS FOR SUPERVISED EXTRACTIVE TRAINING
# ============================================================

def get_ngrams(tokens, n):

    return set(
        zip(
            *[
                tokens[i:]
                for i in range(n)
            ]
        )
    )


def lcs_length(x, y):

    m = len(x)
    n = len(y)

    dp = [
        [0] * (n + 1)
        for _ in range(m + 1)
    ]

    for i in range(m):

        for j in range(n):

            if x[i] == y[j]:

                dp[i + 1][j + 1] = (
                    dp[i][j] + 1
                )

            else:

                dp[i + 1][j + 1] = max(
                    dp[i][j + 1],
                    dp[i + 1][j]
                )

    return dp[m][n]


def rouge_oracle_score(
    sentence,
    reference
):

    sent_tokens = simple_word_tokenize(
        sentence
    )

    ref_tokens = simple_word_tokenize(
        reference
    )

    if len(sent_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    sent_uni = set(sent_tokens)
    ref_uni = set(ref_tokens)

    rouge1 = len(
        sent_uni & ref_uni
    ) / max(
        len(ref_uni),
        1
    )

    sent_bi = get_ngrams(
        sent_tokens,
        2
    )

    ref_bi = get_ngrams(
        ref_tokens,
        2
    )

    if len(ref_bi) == 0:
        rouge2 = 0.0
    else:
        rouge2 = len(
            sent_bi & ref_bi
        ) / max(
            len(ref_bi),
            1
        )

    rougeL = lcs_length(
        sent_tokens,
        ref_tokens
    ) / max(
        len(ref_tokens),
        1
    )

    return (
        rouge1
        +
        rouge2
        +
        rougeL
    ) / 3.0


def create_hhgraphsum_labels(
    sentences,
    reference,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    n = len(sentences)

    labels = torch.zeros(
        n,
        dtype=torch.float
    )

    if n == 0:
        return labels

    k = estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

    scores = []

    for idx, sent in enumerate(sentences):

        score = rouge_oracle_score(
            sent,
            reference
        )

        scores.append(
            (
                idx,
                score
            )
        )

    ranked = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    selected = [
        idx
        for idx, _ in ranked[:min(k, n)]
    ]

    labels[selected] = 1.0

    return labels


# ============================================================
# SINUSOIDAL POSITION ENCODING
# ============================================================

def sinusoidal_position_encoding(
    length,
    dim,
    device
):

    pe = torch.zeros(
        length,
        dim,
        device=device
    )

    position = torch.arange(
        0,
        length,
        dtype=torch.float,
        device=device
    ).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(
            0,
            dim,
            2,
            device=device
        ).float()
        *
        (
            -math.log(10000.0)
            /
            dim
        )
    )

    pe[:, 0::2] = torch.sin(
        position * div_term
    )

    pe[:, 1::2] = torch.cos(
        position * div_term[:pe[:, 1::2].shape[1]]
    )

    return pe


# ============================================================
# DOCUMENT GRAPH CONSTRUCTION
# word-word, sentence-word, word-sentence, sentence-sentence
# ============================================================

def build_hhgraphsum_graph(
    sentences,
    vocab,
    device,
    max_words=700,
    max_sent_len=64
):

    tokenized = [
        simple_word_tokenize(sent)
        for sent in sentences
    ]

    # --------------------------------------------------------
    # collect document word frequency and restrict graph size
    # --------------------------------------------------------

    freq = Counter()

    for toks in tokenized:
        freq.update(toks)

    selected_words = [
        w
        for w, _ in freq.most_common(max_words)
    ]

    local_word_to_idx = {
        w: i
        for i, w in enumerate(selected_words)
    }

    word_ids = [
        vocab.get(w, vocab["<UNK>"])
        for w in selected_words
    ]

    if len(word_ids) == 0:
        word_ids = [vocab["<UNK>"]]
        local_word_to_idx = {
            "<UNK>": 0
        }

    W = len(word_ids)
    N = len(sentences)

    # --------------------------------------------------------
    # padded sentence token IDs for space-time learning
    # --------------------------------------------------------

    sent_token_ids = []

    for toks in tokenized:

        ids = [
            vocab.get(tok, vocab["<UNK>"])
            for tok in toks[:max_sent_len]
        ]

        if len(ids) < max_sent_len:

            ids += [
                vocab["<PAD>"]
            ] * (
                max_sent_len - len(ids)
            )

        sent_token_ids.append(ids)

    sent_token_ids = torch.tensor(
        sent_token_ids,
        dtype=torch.long,
        device=device
    )

    # --------------------------------------------------------
    # adjacency matrices
    # --------------------------------------------------------

    A_ww = torch.zeros(
        W,
        W,
        device=device
    )

    A_sw = torch.zeros(
        W,
        N,
        device=device
    )

    A_ws = torch.zeros(
        N,
        W,
        device=device
    )

    tf_matrix = torch.zeros(
        N,
        W,
        device=device
    )

    # --------------------------------------------------------
    # word-word directed order edges
    # --------------------------------------------------------

    for toks in tokenized:

        local_seq = [
            local_word_to_idx[tok]
            for tok in toks
            if tok in local_word_to_idx
        ]

        for i in range(len(local_seq) - 1):

            src = local_seq[i]
            dst = local_seq[i + 1]

            A_ww[src, dst] = 1.0

    # self loops
    A_ww = torch.maximum(
        A_ww,
        torch.eye(W, device=device)
    )

    # --------------------------------------------------------
    # word-sentence incidence and TF
    # --------------------------------------------------------

    df = torch.zeros(
        W,
        device=device
    )

    for sent_idx, toks in enumerate(tokenized):

        counts = Counter(
            [
                tok
                for tok in toks
                if tok in local_word_to_idx
            ]
        )

        for tok, count in counts.items():

            wid = local_word_to_idx[tok]

            tf_matrix[sent_idx, wid] = float(count)

            A_ws[sent_idx, wid] = 1.0
            A_sw[wid, sent_idx] = 1.0

        for tok in set(counts.keys()):

            wid = local_word_to_idx[tok]
            df[wid] += 1.0

    # --------------------------------------------------------
    # TF-IDF edge weights for word-sentence graph
    # --------------------------------------------------------

    idf = torch.log(
        (
            torch.tensor(
                float(N),
                device=device
            )
            +
            1.0
        )
        /
        (
            df
            +
            1.0
        )
    ) + 1.0

    tfidf = tf_matrix * idf.unsqueeze(0)

    A_ws_weight = tfidf

    A_sw_weight = tfidf.T

    # --------------------------------------------------------
    # sentence-sentence fully connected graph
    # --------------------------------------------------------

    A_ss = torch.ones(
        N,
        N,
        device=device
    )

    return {
        "sent_token_ids": sent_token_ids,
        "word_ids": torch.tensor(
            word_ids,
            dtype=torch.long,
            device=device
        ),
        "A_ww": A_ww,
        "A_sw": A_sw,
        "A_ws": A_ws,
        "A_sw_weight": A_sw_weight,
        "A_ws_weight": A_ws_weight,
        "A_ss": A_ss
    }


# ============================================================
# GRAPH ATTENTION BLOCK
# ============================================================

class DenseGraphAttention(nn.Module):

    def __init__(
        self,
        query_dim,
        source_dim,
        hidden_dim,
        dropout=0.1
    ):

        super().__init__()

        self.q_proj = nn.Linear(
            query_dim,
            hidden_dim
        )

        self.k_proj = nn.Linear(
            source_dim,
            hidden_dim
        )

        self.v_proj = nn.Linear(
            source_dim,
            hidden_dim
        )

        self.attn = nn.Linear(
            hidden_dim * 2 + 1,
            1
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.norm = nn.LayerNorm(
            hidden_dim
        )

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )

        self.out_proj = nn.Linear(
            query_dim,
            hidden_dim
        )

    def forward(
        self,
        query_x,
        source_x,
        adj,
        edge_weight=None
    ):

        Q = self.q_proj(
            query_x
        )

        K = self.k_proj(
            source_x
        )

        V = self.v_proj(
            source_x
        )

        T = Q.size(0)
        S = K.size(0)

        Q_exp = Q.unsqueeze(1).expand(
            T,
            S,
            -1
        )

        K_exp = K.unsqueeze(0).expand(
            T,
            S,
            -1
        )

        if edge_weight is None:

            edge_feat = torch.zeros(
                T,
                S,
                1,
                device=query_x.device
            )

        else:

            edge_feat = edge_weight.unsqueeze(-1)

        attn_input = torch.cat(
            [
                Q_exp,
                K_exp,
                edge_feat
            ],
            dim=-1
        )

        scores = F.leaky_relu(
            self.attn(attn_input).squeeze(-1)
        )

        mask = adj <= 0

        scores = scores.masked_fill(
            mask,
            -1e4
        )

        alpha = torch.softmax(
            scores,
            dim=-1
        )

        alpha = self.dropout(
            alpha
        )

        out = torch.matmul(
            alpha,
            V
        )

        residual = self.out_proj(
            query_x
        )

        out = self.norm(
            out + residual
        )

        out = self.norm(
            out + self.ffn(out)
        )

        return out


# ============================================================
# HHGRAPHSUM MODEL
# ============================================================

class HHGraphSum(nn.Module):

    def __init__(
        self,
        vocab,
        device,
        word_dim=100,
        hidden_dim=256,
        cnn_kernels=(2, 3, 4),
        lstm_layers=1,
        transformer_layers=1,
        dropout=0.1
    ):

        super().__init__()

        self.vocab = vocab
        self.device = device
        self.hidden_dim = hidden_dim

        # ----------------------------------------------------
        # word embedding X_w
        # ----------------------------------------------------

        self.word_emb = nn.Embedding(
            len(vocab),
            word_dim,
            padding_idx=vocab["<PAD>"]
        )

        # ----------------------------------------------------
        # multi-scale CNN for spatial sentence features
        # ----------------------------------------------------

        conv_out = hidden_dim // len(cnn_kernels)

        self.convs = nn.ModuleList(
            [
                nn.Conv1d(
                    word_dim,
                    conv_out,
                    kernel_size=k,
                    padding=k // 2
                )
                for k in cnn_kernels
            ]
        )

        self.spatial_fc = nn.Linear(
            conv_out * len(cnn_kernels),
            hidden_dim
        )

        # ----------------------------------------------------
        # BiLSTM + Transformer for sequential sentence features
        # ----------------------------------------------------

        self.bilstm = nn.LSTM(
            hidden_dim,
            hidden_dim // 2,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=transformer_layers
        )

        self.sent_proj = nn.Linear(
            hidden_dim * 2,
            hidden_dim
        )

        self.word_proj = nn.Linear(
            word_dim,
            hidden_dim
        )

        # ----------------------------------------------------
        # hierarchical heterogeneous graph learning
        # ----------------------------------------------------

        self.word_word_gat = DenseGraphAttention(
            hidden_dim,
            hidden_dim,
            hidden_dim,
            dropout=dropout
        )

        self.sent_to_word_gat = DenseGraphAttention(
            hidden_dim,
            hidden_dim,
            hidden_dim,
            dropout=dropout
        )

        self.word_to_sent_gat = DenseGraphAttention(
            hidden_dim,
            hidden_dim,
            hidden_dim,
            dropout=dropout
        )

        self.sent_sent_gat = DenseGraphAttention(
            hidden_dim,
            hidden_dim,
            hidden_dim,
            dropout=dropout
        )

        # ----------------------------------------------------
        # LSTM predictor
        # ----------------------------------------------------

        self.predictor_lstm = nn.LSTM(
            hidden_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )

        self.classifier = nn.Linear(
            hidden_dim,
            1
        )

    # ========================================================
    # SPACE-TIME COLLABORATIVE LEARNING
    # ========================================================

    def encode_sentences_space_time(
        self,
        sent_token_ids
    ):

        # sent_token_ids: [N, L]

        emb = self.word_emb(
            sent_token_ids
        )

        # [N, L, D] -> [N, D, L]
        x = emb.transpose(
            1,
            2
        )

        conv_features = []

        for conv in self.convs:

            c = F.relu(
                conv(x)
            )

            c = F.max_pool1d(
                c,
                kernel_size=c.size(-1)
            ).squeeze(-1)

            conv_features.append(c)

        R = torch.cat(
            conv_features,
            dim=-1
        )

        S1 = self.spatial_fc(
            R
        )

        PE = sinusoidal_position_encoding(
            S1.size(0),
            S1.size(1),
            self.device
        )

        S1 = S1 + PE

        # BiLSTM expects [B, N, H]
        lstm_out, _ = self.bilstm(
            S1.unsqueeze(0)
        )

        trans_out = self.transformer(
            lstm_out
        )

        S2 = trans_out.squeeze(0)

        Xs = self.sent_proj(
            torch.cat(
                [
                    S2,
                    S1
                ],
                dim=-1
            )
        )

        return Xs

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        sentences
    ):

        graph = build_hhgraphsum_graph(
            sentences,
            self.vocab,
            self.device
        )

        sent_token_ids = graph["sent_token_ids"]
        word_ids = graph["word_ids"]

        Xs = self.encode_sentences_space_time(
            sent_token_ids
        )

        Xw = self.word_proj(
            self.word_emb(word_ids)
        )

        # ----------------------------------------------------
        # word-word level
        # W = GATT(Xw, Xw, A_w->w)
        # ----------------------------------------------------

        W = self.word_word_gat(
            Xw,
            Xw,
            graph["A_ww"]
        )

        # ----------------------------------------------------
        # sentence-to-word level
        # W' = GATT(W, Xs, A_s->w, TF-IDF)
        # target = word nodes, source = sentence nodes
        # ----------------------------------------------------

        W_prime = self.sent_to_word_gat(
            W,
            Xs,
            graph["A_sw"],
            graph["A_sw_weight"]
        )

        # ----------------------------------------------------
        # word-to-sentence level
        # S = GATT(Xs, W', A_w->s, TF-IDF)
        # target = sentence nodes, source = word nodes
        # ----------------------------------------------------

        S = self.word_to_sent_gat(
            Xs,
            W_prime,
            graph["A_ws"],
            graph["A_ws_weight"]
        )

        # ----------------------------------------------------
        # sentence-sentence level
        # S' = GATT(S, S, A_s<->s)
        # ----------------------------------------------------

        S_prime = self.sent_sent_gat(
            S,
            S,
            graph["A_ss"]
        )

        # ----------------------------------------------------
        # LSTM sentence predictor
        # ----------------------------------------------------

        lstm_out, _ = self.predictor_lstm(
            S_prime.unsqueeze(0)
        )

        logits = self.classifier(
            lstm_out.squeeze(0)
        ).squeeze(-1)

        return logits


# ============================================================
# HHGRAPHSUM TRAINING
# ============================================================

def train_hhgraphsum(
    model,
    dataset,
    epochs=3,
    lr=2e-4
):

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr
    )

    bce_loss_fn = nn.BCEWithLogitsLoss()

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=torch.cuda.is_available()
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0
        count = 0

        for doc in tqdm(
            dataset,
            desc=f"HHGraphSum Epoch {epoch+1}"
        ):

            sentences = get_sentences_from_doc(
                doc
            )

            reference = " ".join(
                doc["headnote_sent"]
            )

            if len(sentences) < 2:
                continue

            labels = create_hhgraphsum_labels(
                sentences,
                reference,
                ratio=0.25,
                min_len=8,
                max_len=30
            ).to(model.device)

            optimizer.zero_grad(
                set_to_none=True
            )

            try:

                with torch.amp.autocast(
                    "cuda",
                    enabled=torch.cuda.is_available()
                ):

                    logits = model(
                        sentences
                    )

                    loss = bce_loss_fn(
                        logits,
                        labels
                    )

                scaler.scale(
                    loss
                ).backward()

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                total_loss += loss.item()
                count += 1

                del labels
                del logits
                del loss

                torch.cuda.empty_cache()
                gc.collect()

            except RuntimeError as e:

                if "out of memory" in str(e).lower():

                    print("OOM skipped")

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    torch.cuda.empty_cache()
                    gc.collect()

                    continue

                else:

                    raise e

        print(
            f"Epoch {epoch+1} Loss: "
            f"{total_loss / max(count, 1):.4f}"
        )

    model.eval()


# ============================================================
# N-GRAM BLOCKING
# ============================================================

def has_ngram_overlap(
    candidate,
    selected,
    n=3
):

    cand_tokens = simple_word_tokenize(
        candidate
    )

    cand_grams = get_ngrams(
        cand_tokens,
        n
    )

    if len(cand_grams) == 0:
        return False

    for sent in selected:

        sent_tokens = simple_word_tokenize(
            sent
        )

        sent_grams = get_ngrams(
            sent_tokens,
            n
        )

        if len(
            cand_grams & sent_grams
        ) > 0:

            return True

    return False


# ============================================================
# HHGRAPHSUM INFERENCE
# Top-k + blocking + original order
# ============================================================

def hhgraphsum_select(
    sentences,
    model,
    num_sentences=10,
    block_ngram=3
):

    model.eval()

    with torch.no_grad():

        logits = model(
            sentences
        )

        scores = torch.sigmoid(
            logits
        ).detach().cpu().numpy()

    ranked_idx = np.argsort(
        scores
    )[::-1]

    selected_indices = []
    selected_sentences = []

    for idx in ranked_idx:

        candidate = sentences[idx]

        if not has_ngram_overlap(
            candidate,
            selected_sentences,
            n=block_ngram
        ):

            selected_indices.append(
                idx
            )

            selected_sentences.append(
                candidate
            )

        if len(selected_indices) >= num_sentences:
            break

    selected_indices = sorted(
        selected_indices
    )

    return " ".join(
        [
            sentences[i]
            for i in selected_indices
        ]
    )

# ============================================================
# CHUNKING
# ============================================================

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

# ============================================================
# GENERATE SUMMARY
# ============================================================

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

# ============================================================
# FACTCC
# ============================================================

def factcc_chunked_fast(
    pred,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():

            logits = factcc_model(**inputs).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))


# ============================================================
# TRAIN HHGRAPHSUM MODEL
# ============================================================

hh_vocab = build_hhgraphsum_vocab(
    train_dataset,
    min_freq=3,
    max_vocab_size=50000
)

hhgraphsum_model = HHGraphSum(
    vocab=hh_vocab,
    device=device,
    word_dim=100,
    hidden_dim=256,
    cnn_kernels=(2, 3, 4),
    lstm_layers=1,
    transformer_layers=1,
    dropout=0.1
).to(device)

train_hhgraphsum(
    hhgraphsum_model,
    train_dataset,
    epochs=3,
    lr=2e-5
)
# ============================================================
# EVALUATION
# ============================================================

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []

bart_scores = []
factcc_scores = []

# ============================================================
# TEST LOOP
# ============================================================

for doc in tqdm(
    test_dataset,
    desc="Evaluating"
):

    # ========================================================
    # REFERENCE
    # ========================================================

    reference = " ".join(
        doc["headnote_sent"]
    )

    # ========================================================
    # JUDGEMENT
    # ========================================================

    if isinstance(
        doc["judgement_sent"][0],
        dict
    ):
    
        sentences = [
            s["sentence"]
            for s in doc["judgement_sent"]
        ]
    
    else:
    
        sentences = doc["judgement_sent"]

    judgement = " ".join(
        preprocess(s)
        for s in sentences
    )

    # ========================================================
    # EXTRACTIVE SELECTION
    # ========================================================

    num_select = estimate_summary_length(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    ranked_text = hhgraphsum_select(
        sentences=sentences,
        model=hhgraphsum_model,
        num_sentences=num_select,
        block_ngram=3
    )

    # ========================================================
    # ABSTRACT GENERATION
    # ========================================================

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # ========================================================
    # ROUGE
    # ========================================================

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )

    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # ========================================================
    # BLEU
    # ========================================================

    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )

    bleu_scores.append(
        bleu_doc["bleu"]
    )

    # ========================================================
    # METEOR
    # ========================================================

    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )

    meteor_scores.append(
        meteor_doc["meteor"]
    )

    # ========================================================
    # BERTScore
    # ========================================================

    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
        bert_doc["f1"][0]
    )

    # ========================================================
    # BARTScore
    # ========================================================

    bart_scores.append(
        bart_scorer.score(
            [pred],
            [reference]
        )[0]
    )

    # ========================================================
    # FACTCC
    # ========================================================

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(
            pred,
            judgement_chunks
        )
    )

    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del pred
    del ranked_text
    del judgement_chunks

    torch.cuda.empty_cache()
    gc.collect()

# ============================================================
# FINAL RESULTS
# ============================================================

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")



# LegalSummNet

In [ ]:
import sys
import os

# ============================================================
# BARTScore Path
# ============================================================
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")


# ============================================================
# IMPORTS
# ============================================================

import json
import gc
import re
import math
import torch
import evaluate
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    pipeline
)

from sentence_transformers import SentenceTransformer
from bart_score import BARTScorer

import nltk

nltk.download("punkt")
nltk.download("stopwords")

# ============================================================
# STOPWORDS
# ============================================================

stop_words = set(stopwords.words("english"))

# ============================================================
# DATASET
# ============================================================

file_path = (
    "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgements/data12.json"
)

with open(file_path, "r") as f:
    dataset = json.load(f)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_dataset = dataset[:5000]
test_dataset = dataset[9500:]

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

# ============================================================
# ABSTRACTION MODEL
# ============================================================

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()

# ============================================================
# EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# BARTScore
# ============================================================

bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# ============================================================
# FACTCC
# ============================================================

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    factcc_model_name
)

factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

# ============================================================
# SBERT
# ============================================================

sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device=device
)

# ============================================================
# PREPROCESS
# ============================================================

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ============================================================
# SUMMARY LENGTH
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )


class LegalSummNet(nn.Module):

    def __init__(
        self,
        device,
        freeze_bert=True
    ):
        super().__init__()

        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        )

        self.bert = AutoModel.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        ).to(device)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
        else:
            for name, param in self.bert.named_parameters():
                param.requires_grad = (
                    "encoder.layer.10" in name
                    or "encoder.layer.11" in name
                )

        self.Ws = nn.Linear(768, 256)
        self.bs = nn.Parameter(torch.zeros(256))
        self.u = nn.Parameter(torch.randn(256))
        self.dropout = nn.Dropout(0.2)

        self.classifier = nn.Linear(256, 1)

    def encode_sentences(
        self,
        sentences,
        batch_size=16
    ):
        embeddings = []

        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i + batch_size]

            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=96
            ).to(self.device)

            if any(p.requires_grad for p in self.bert.parameters()):
                outputs = self.bert(**inputs)
                cls = outputs.last_hidden_state[:, 0, :]
            else:
                with torch.no_grad():
                    outputs = self.bert(**inputs)
                    cls = outputs.last_hidden_state[:, 0, :]

            embeddings.append(cls)

            del inputs
            del outputs

        return torch.cat(embeddings, dim=0)

    def forward(self, sentences):

        sent_emb = self.encode_sentences(sentences)

        hidden = torch.tanh(
            self.Ws(sent_emb) + self.bs
        )

        hidden = self.dropout(hidden)

        attention_logits = torch.matmul(
            hidden,
            self.u
        )

        attention_weights = torch.softmax(
            attention_logits,
            dim=0
        )

        class_logits = self.classifier(
            hidden
        ).squeeze(-1)

        return class_logits, attention_weights

# ============================================================
# ROUGE-ORACLE EXTRACTIVE LABELS
# ============================================================

def get_ngrams(tokens, n):

    return set(
        zip(
            *[
                tokens[i:]
                for i in range(n)
            ]
        )
    )


def lcs_length(x, y):

    m = len(x)
    n = len(y)

    dp = [
        [0] * (n + 1)
        for _ in range(m + 1)
    ]

    for i in range(m):

        for j in range(n):

            if x[i] == y[j]:

                dp[i + 1][j + 1] = (
                    dp[i][j] + 1
                )

            else:

                dp[i + 1][j + 1] = max(
                    dp[i][j + 1],
                    dp[i + 1][j]
                )

    return dp[m][n]


def rouge_oracle_score(sentence, reference):

    sent_tokens = [
        w.lower()
        for w in word_tokenize(sentence)
        if w.isalpha()
    ]

    ref_tokens = [
        w.lower()
        for w in word_tokenize(reference)
        if w.isalpha()
    ]

    if len(sent_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    sent_uni = set(sent_tokens)
    ref_uni = set(ref_tokens)

    rouge1 = len(sent_uni & ref_uni) / max(
        len(ref_uni),
        1
    )

    sent_bi = get_ngrams(
        sent_tokens,
        2
    )

    ref_bi = get_ngrams(
        ref_tokens,
        2
    )

    rouge2 = len(sent_bi & ref_bi) / max(
        len(ref_bi),
        1
    ) if len(ref_bi) > 0 else 0.0

    rougeL = lcs_length(
        sent_tokens,
        ref_tokens
    ) / max(
        len(ref_tokens),
        1
    )

    return (
        rouge1
        +
        rouge2
        +
        rougeL
    ) / 3.0


def create_sentence_labels(
    sentences,
    reference,
    top_ratio=0.25,
    min_len=8,
    max_len=30
):

    n = len(sentences)

    labels = torch.zeros(
        n,
        dtype=torch.float
    )

    if n == 0:
        return labels

    k = estimate_summary_length(
        sentences,
        ratio=top_ratio,
        min_len=min_len,
        max_len=max_len
    )

    scores = []

    for idx, sent in enumerate(sentences):

        score = rouge_oracle_score(
            sent,
            reference
        )

        scores.append(
            (
                idx,
                score
            )
        )

    ranked = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    selected = [
        idx
        for idx, _ in ranked[:min(k, n)]
    ]

    labels[selected] = 1.0

    return labels
  

# ============================================================
# TRAINING
# ============================================================

def train_legalsummnet(
    model,
    dataset,
    epochs=3,
    lr=2e-5,
    lambda_ext=0.7
):

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=lr
    )

    bce_loss_fn = nn.BCEWithLogitsLoss()

    model.train()

    scaler = torch.amp.GradScaler("cuda")
    
    for epoch in range(epochs):

        total_loss = 0
        count = 0

        for doc in tqdm(
            dataset,
            desc=f"Epoch {epoch+1}"
        ):

            # =================================================
            # SENTENCES
            # =================================================

            if isinstance(
                doc["judgement_sent"][0],
                dict
            ):

                sentences = [
                    s["sentence"]
                    for s in doc["judgement_sent"]
                ]

            else:

                sentences = doc["judgement_sent"]

            reference = " ".join(
                doc["headnote_sent"]
            )

            if len(sentences) < 2:
                continue

            # =================================================
            # SOFT LABELS
            # =================================================

            labels = create_sentence_labels(
                sentences,
                reference,
                top_ratio=0.25,
                min_len=8,
                max_len=30
            ).to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
            
                attention_logits, attention_weights = model(
                    sentences
                )
                
                loss = bce_loss_fn(
                    attention_logits,
                    labels
                )
                            
            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )
            
            scaler.step(optimizer)
            
            scaler.update()
            
            total_loss += loss.item()
            
            count += 1
            
            del attention_logits
            del attention_weights
            del loss
            del labels
            
            torch.cuda.empty_cache()
            gc.collect()
            

        avg_loss = total_loss / max(count, 1)

        print(
            f"Epoch {epoch+1} "
            f"Loss: {avg_loss:.4f}"
        )

    model.eval()


def legalsummnet_select(
    sentences,
    model,
    num_sentences=10
):

    model.eval()

    with torch.no_grad():

        class_logits, attention_weights = model(
            sentences
        )

        class_scores = torch.sigmoid(
            class_logits
        )

        scores = (
            0.5 * class_scores
            +
            0.5 * attention_weights
        ).detach().cpu().numpy()

    top_idx = np.argsort(
        scores
    )[-num_sentences:]

    top_idx = sorted(
        top_idx
    )

    selected = [
        sentences[i]
        for i in top_idx
    ]

    return " ".join(selected)

# ============================================================
# CHUNKING
# ============================================================

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

# ============================================================
# GENERATE SUMMARY
# ============================================================

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

# ============================================================
# FACTCC
# ============================================================

def factcc_chunked_fast(
    pred,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():

            logits = factcc_model(**inputs).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))

# ============================================================
# TRAIN MODEL
# ============================================================

legalsumm_model = LegalSummNet(
    device
).to(device)

train_legalsummnet(
    legalsumm_model,
    train_dataset,
    epochs=3,
    lr=2e-5,
    lambda_ext=0.7
)

# ============================================================
# EVALUATION
# ============================================================

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []

bart_scores = []
factcc_scores = []

# ============================================================
# TEST LOOP
# ============================================================

for doc in tqdm(
    test_dataset,
    desc="Evaluating"
):

    # ========================================================
    # REFERENCE
    # ========================================================

    reference = " ".join(
        doc["headnote_sent"]
    )

    # ========================================================
    # JUDGEMENT
    # ========================================================

    sentences = doc["judgement_sent"]

    judgement = " ".join(
        preprocess(s)
        for s in sentences
    )

    # ========================================================
    # EXTRACTIVE SELECTION
    # ========================================================

    num_select = estimate_summary_length(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    ranked_text = legalsummnet_select(
        sentences=sentences,
        model=legalsumm_model,
        num_sentences=num_select
    )

    # ========================================================
    # ABSTRACT GENERATION
    # ========================================================

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # ========================================================
    # ROUGE
    # ========================================================

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )

    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # ========================================================
    # BLEU
    # ========================================================

    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )

    bleu_scores.append(
        bleu_doc["bleu"]
    )

    # ========================================================
    # METEOR
    # ========================================================

    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )

    meteor_scores.append(
        meteor_doc["meteor"]
    )

    # ========================================================
    # BERTScore
    # ========================================================

    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
        bert_doc["f1"][0]
    )

    # ========================================================
    # BARTScore
    # ========================================================

    bart_scores.append(
        bart_scorer.score(
            [pred],
            [reference]
        )[0]
    )

    # ========================================================
    # FACTCC
    # ========================================================

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(
            pred,
            judgement_chunks
        )
    )

    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del pred
    del ranked_text
    del judgement_chunks

    torch.cuda.empty_cache()
    gc.collect()

# ============================================================
# FINAL RESULTS
# ============================================================

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")



# SumHIS

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import random
import evaluate
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
# from summac.model_summac import SummaCConv, SummaCZS
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")
from sklearn.metrics.pairwise import cosine_similarity

stop_words = set(stopwords.words("english"))
from nltk import sent_tokenize

file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgements/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

train_dataset = dataset[:5000]
test_dataset = dataset[9500:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences

from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

# ============================================================
# SUMHIS MODEL
# ============================================================

class SumHiSModel(nn.Module):

    def __init__(
        self,
        device,
        num_clusters=10
    ):

        super().__init__()

        self.device = device

        self.num_clusters = num_clusters

        # ====================================================
        # LEGAL-BERT
        # ====================================================

        self.tokenizer = AutoTokenizer.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        )

        self.encoder = AutoModel.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        ).to(device)

        # ====================================================
        # FREEZE ENCODER
        # ====================================================

        for p in self.encoder.parameters():
            p.requires_grad = False


        # ====================================================
        # CLUSTER EMBEDDINGS
        # ====================================================

        self.cluster_embeddings = nn.Parameter(
            torch.randn(
                num_clusters,
                768
            ) * 0.02
        )

    # ========================================================
    # ENCODE
    # ========================================================
    
    def encode(
        self,
        texts,
        batch_size=8
    ):
    
        embeddings = []
    
        for i in range(0, len(texts), batch_size):
    
            batch = texts[i:i+batch_size]
    
            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            ).to(self.device)
    
            outputs = self.encoder(**inputs)
    
            cls = outputs.last_hidden_state[:, 0, :]
    
            embeddings.append(cls)
    
            del inputs
            del outputs
    
        return torch.cat(embeddings, dim=0)

    # ========================================================
    # DOT PRODUCT SIMILARITY
    # ========================================================

    def similarity(
        self,
        text_emb,
        sent_emb
    ):
    
        return F.cosine_similarity(
            text_emb,
            sent_emb,
            dim=-1
        )

    # ========================================================
    # HIDDEN STRUCTURE DISCOVERY
    # ========================================================

    def hidden_structure(
        self,
        q
    ):

        # ----------------------------------------------------
        # p_j = c_j . q
        # ----------------------------------------------------

        scores = torch.matmul(
            q,
            self.cluster_embeddings.T
        )

        # ----------------------------------------------------
        # softmax normalization
        # ----------------------------------------------------

        weights = F.softmax(
            scores,
            dim=-1
        )

        # ----------------------------------------------------
        # reconstruction
        # o = sum p_j c_j
        # ----------------------------------------------------

        reconstruction = torch.matmul(
            weights,
            self.cluster_embeddings
        )

        return weights, reconstruction

# ============================================================
# TRIPLET GENERATION
# ============================================================

def generate_triplets(
    sentences,
    reference
):

    ref_sents = sent_tokenize(reference)

    if len(ref_sents) == 0:

        return []

    positives = []

    negatives = []

    for s in sentences:

        overlap = max([
            len(
                set(s.split()) &
                set(r.split())
            )
            for r in ref_sents
        ])

        if overlap > 3:

            positives.append(s)

        else:

            negatives.append(s)

    triplets = []

    for p in positives:

        if len(negatives) == 0:
            continue

        n = random.choice(negatives)

        triplets.append(
            (sentences, p, n)
        )

    return triplets

import gc
# ============================================================
# PHASE 1
# SENTENCE RANKING TRAINING
# ============================================================

def train_sumhis(
    model,
    train_dataset,
    epochs=3,
    lr=2e-5,
    lambda_structure=0.3
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0
        count = 0

        for doc in tqdm(
            train_dataset,
            desc=f"SumHiS Epoch {epoch+1}"
        ):

            sentences = doc["judgement_sent"]

            reference = " ".join(
                doc["headnote_sent"]
            )

            if len(sentences) < 2:
                continue

            triplets = generate_triplets(
                sentences,
                reference
            )

            if len(triplets) == 0:
                continue

            text = " ".join(sentences)

            text_emb = model.encode([text])

            for _, pos_sent, neg_sent in triplets:

                pos_emb = model.encode([pos_sent])

                neg_emb = model.encode([neg_sent])

                # ============================================
                # RANKING LOSS
                # ============================================

                pos_sim = model.similarity(
                    text_emb,
                    pos_emb
                )

                neg_sim = model.similarity(
                    text_emb,
                    neg_emb
                )

                logits = torch.stack(
                    [pos_sim, neg_sim],
                    dim=1
                )

                targets = torch.zeros(
                    logits.size(0),
                    dtype=torch.long,
                    device=model.device
                )

                ranking_loss = F.cross_entropy(
                    logits,
                    targets
                )

                # ============================================
                # STRUCTURE LOSS
                # ============================================

                weights, reconstruction = (
                    model.hidden_structure(
                        pos_emb
                    )
                )

                cosine_sim = F.cosine_similarity(
                    pos_emb,
                    reconstruction,
                    dim=-1
                )

                structure_loss = (
                    1 - cosine_sim
                ).mean()

                # ============================================
                # FINAL LOSS
                # ============================================

                loss = (
                    ranking_loss
                    +
                    lambda_structure
                    * structure_loss
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

                count += 1

        avg_loss = total_loss / max(count, 1)

        print(
            f"Epoch {epoch+1} "
            f"Loss: {avg_loss:.4f}"
        )

# ============================================================
# INFERENCE
# ============================================================

def sumhis_select(
    sentences,
    model,
    num_sentences=10,
    threshold=0.2
):

    if len(sentences) == 0:

        return []

    with torch.no_grad():

        # ----------------------------------------------------
        # DOCUMENT EMBEDDING
        # ----------------------------------------------------

        text = " ".join(sentences)

        text_emb = model.encode([text])

        # ----------------------------------------------------
        # SENTENCE EMBEDDINGS
        # ----------------------------------------------------

        sent_emb = model.encode(sentences)

        # ----------------------------------------------------
        # RANKING SCORES
        # ----------------------------------------------------

        sims = model.similarity(
            text_emb,
            sent_emb
        )

        scores = sims.cpu().numpy()

        # ----------------------------------------------------
        # HIDDEN STRUCTURE
        # ----------------------------------------------------

        weights, _ = model.hidden_structure(
            sent_emb
        )

        max_cluster_weight = (
            weights.max(dim=1).values
        ).cpu().numpy()

        # ----------------------------------------------------
        # THRESHOLD FILTERING
        # ----------------------------------------------------

        valid_idx = []

        for i in range(len(sentences)):

            if max_cluster_weight[i] > threshold:

                valid_idx.append(i)

        if len(valid_idx) == 0:

            valid_idx = list(
                range(len(sentences))
            )

        # ----------------------------------------------------
        # FILTERED SCORES
        # ----------------------------------------------------

        filtered_scores = [
            (i, scores[i])
            for i in valid_idx
        ]

        filtered_scores = sorted(
            filtered_scores,
            key=lambda x: x[1],
            reverse=True
        )

        top_k = min(
            num_sentences,
            len(filtered_scores)
        )

        selected_idx = sorted([
            idx
            for idx, _
            in filtered_scores[:top_k]
        ])

        summary = [
            sentences[i]
            for i in selected_idx
        ]

    return summary

sumhis_model = SumHiSModel(
    device=device,
    num_clusters=10
).to(device)


train_sumhis(
    model=sumhis_model,
    train_dataset=train_dataset,
    epochs=3,
    lr=2e-5,
    lambda_structure=0.3
)

# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

# SummaC
# summac_conv = SummaCConv(device=device)
# summac_zs = SummaCZS(device=device)
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))


from transformers import AutoModelForSequenceClassification, AutoTokenizer
sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)


predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(test_dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s)
        for s in doc["judgement_sent"]
        # if not is_metadata_sentence(s)
    )
    sentences = doc["judgement_sent"]
    
    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,   # can tune (0.20–0.35)
        min_len=8,
        max_len=30
    )
    

    selected_sentences = sumhis_select(
        sentences=sentences,
        model=sumhis_model,
        num_sentences=top_n,
        threshold=0.2
    )
    
    ranked_text = " ".join(selected_sentences)

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )
    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # # # -----------------------
    # # # BLEU (DOCUMENT-WISE)
    # # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # # # -----------------------
    # # # METEOR (DOCUMENT-WISE)
    # # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # # # -----------------------
    # # # BERTScore (DOCUMENT-WISE)
    # # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )
    
    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    
    torch.cuda.empty_cache()

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")


# termdiffusum

In [ ]:
# ============================================================
# PURE TERMDIFFUSUM IMPLEMENTATION
# (NO RHETORICAL ROLES)
# ============================================================
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import os
import gc
import re
import json
import math
import torch
import random
import nltk
import evaluate
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)

from sklearn.metrics.pairwise import cosine_similarity

from bart_score import BARTScorer
from sentence_transformers import SentenceTransformer

# ============================================================
# NLTK
# ============================================================

nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

# ============================================================
# DEVICE
# ============================================================

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device0 = torch.device("cuda:0")
device1 = torch.device("cuda:1")

# ============================================================
# LOAD DATASET
# ============================================================

file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgements/data12.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_dataset = dataset[:5000]
test_dataset = dataset[-500:]

print(f"Train Size: {len(train_dataset)}")
print(f"Test Size : {len(test_dataset)}")

# ============================================================
# LOAD ABSTRATIVE MODEL
# ============================================================

abstractive_model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device1)

abstractive_tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

abstractive_model.eval()

# ============================================================
# EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# BARTScore
# ============================================================

# bart_scorer = BARTScorer(
#     device=device,
#     checkpoint="facebook/bart-large-cnn"
# )
bart_scorer = BARTScorer(
    device=device1,
    checkpoint="facebook/bart-large-cnn"
)
# ============================================================
# FACTCC
# ============================================================

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    factcc_model_name
)

# factcc_model = AutoModelForSequenceClassification.from_pretrained(
#     factcc_model_name
# ).to(device)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device1)

factcc_model.eval()

# ============================================================
# SBERT
# ============================================================

# sbert_model = SentenceTransformer(
#     "all-mpnet-base-v2",
#     device=device
# )
sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device="cuda:1"
)

# LEGAL_TERMS = {
#     "court","judge","judgment","appeal","petitioner",
#     "respondent","section","article","constitution",
#     "tribunal","criminal","civil","evidence",
#     "conviction","sentence","writ","plaintiff",
#     "defendant","bench","order","act"
# }
LEGAL_TERMS = {
    # Core Judicial Terms
    "court","judge","bench","order","judgment","decree","ruling","decision","opinion",
    "petitioner","respondent","plaintiff","defendant","appellant","accused","complainant",

    # Constitutional & Statutory References
    "article","section","clause","subsection","provision","statute","act","code","rule","regulation",
    "constitution","amendment","fundamental_rights","directive_principles","preamble",

    # Institutions & Proceedings
    "tribunal","commission","authority","high_court","supreme_court","sessions_court",
    "writ","mandamus","certiorari","habeas_corpus","prohibition","quo_warranto",
    "appeal","revision","review","special_leave_petition","slp","interim_order",

    # Criminal & Civil Law
    "criminal","civil","offence","crime","charge","fir","investigation","prosecution",
    "conviction","acquittal","sentence","punishment","bail","remand","custody","trial",
    "evidence","witness","testimony","cross_examination","affidavit",

    # Procedural & Administrative
    "plea","petition","application","submission","argument","hearing","proceedings",
    "interim_relief","stay_order","injunction","contempt","costs","damages",
    "jurisdiction","maintainability","locus_standi","cause_of_action","ratio_decidendi","obiter_dicta",

    # Specialized Legal Vocabulary
    "precedent","case_law","citation","headnote","doctrine","principle","interpretation",
    "arbitration","mediation","conciliation","settlement","negotiation",
    "liability","negligence","tort","contract","agreement","breach"
}

def sentence_entropy(sentence):

    words = word_tokenize(sentence.lower())

    words = [
        w for w in words
        if w.isalpha()
    ]

    if len(words) == 0:
        return 0

    counts = Counter(words)

    total = len(words)

    entropy = 0

    for c in counts.values():

        p = c / total

        entropy -= p * math.log(p + 1e-12)

    return entropy

def legal_term_score(sentence):

    words = set(
        word_tokenize(sentence.lower())
    )

    count = sum(
        1 for w in words
        if w in LEGAL_TERMS
    )

    return (
        int(count > 0)
        +
        0.5 * count
    )

def position_score(idx, total):

    return math.exp(
        -((idx-total/2)**2)
        /
        (total/2 + 1e-8)
    )

def compute_sentence_weights(sentences):

    scores = []

    N = len(sentences)

    for i,s in enumerate(sentences):

        score = (
            sentence_entropy(s)
            +
            legal_term_score(s)
            +
            position_score(i,N)
        )

        scores.append(score)

    scores = np.array(scores)

    scores = (
        scores - scores.min()
    ) / (
        scores.max() - scores.min()
        + 1e-8
    )

    return torch.tensor(
        scores,
        dtype=torch.float
    )


# ============================================================
# TERMDIFFUSUM MODEL
# ============================================================

# ============================================================
# REAL TERMDIFFUSUM-STYLE MODEL
# Partial diffusion + multifactor sentence weighting
# + ROUGE-aware candidate reranking
# ============================================================

class TermDiffuSum(nn.Module):

    def __init__(
        self,
        device0,
        device1,
        hidden_size=768,
        diffusion_steps=200,
        beta0=1e-4,
        gamma=1e-4,
        lambda_w=0.02,
        lambda_entropy=1.0,
        margin=0.1,
        train_encoder=False
    ):

        super().__init__()

        self.device0 = device0
        self.device1 = device1

        self.hidden_size = hidden_size
        self.T = diffusion_steps

        self.beta0 = beta0
        self.gamma = gamma
        self.lambda_w = lambda_w
        self.lambda_entropy = lambda_entropy
        self.margin = margin

        self.train_encoder = train_encoder

        # ====================================================
        # SENTENCE ENCODER -> GPU1
        # ====================================================

        self.tokenizer = AutoTokenizer.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        )

        self.encoder = AutoModel.from_pretrained(
            "nlpaueb/legal-bert-base-uncased"
        ).to(self.device1)

        if not train_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        # ====================================================
        # MLP PROJECTION -> GPU0
        # Hin = MLP(Encoder(.))
        # ====================================================

        self.proj = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size)
        ).to(self.device0)

        # ====================================================
        # TIME EMBEDDING + REVERSE DENOISER -> GPU0
        # Predicts clean summary embedding x0_s
        # ====================================================

        self.time_embed = nn.Embedding(
            diffusion_steps + 1,
            128
        ).to(self.device0)

        self.reverse_net = nn.Sequential(
            nn.Linear(hidden_size * 2 + 128, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size)
        ).to(self.device0)

    # ========================================================
    # LEGAL TERM / ENTROPY / POSITION SENTENCE WEIGHT e(s)
    # ========================================================

    def sentence_entropy(self, sentence):

        words = [
            w.lower()
            for w in word_tokenize(sentence)
            if w.isalpha()
        ]

        if len(words) == 0:
            return 0.0

        counts = Counter(words)
        total = len(words)

        entropy = 0.0

        for c in counts.values():

            p = c / total
            entropy -= p * math.log(p + 1e-12)

        return entropy


    def legal_term_score(self, sentence, lambda1=0.5):

        words = set(
            w.lower()
            for w in word_tokenize(sentence)
            if w.isalpha()
        )

        count = sum(
            1
            for w in words
            if w in LEGAL_TERMS
        )

        return float(count > 0) + lambda1 * count


    def position_weight(self, idx, total):

        if total <= 1:
            return 1.0

        # beginning/end are emphasized
        p = idx + 1
        maxp = total

        begin_score = math.exp(
            -((p - 1) ** 2)
            /
            ((maxp / 2) ** 2 + 1e-8)
        )

        end_score = math.exp(
            -((p - maxp) ** 2)
            /
            ((maxp / 2) ** 2 + 1e-8)
        )

        return max(begin_score, end_score)


    def compute_sentence_weights(self, sentences):

        scores = []

        total = len(sentences)

        for i, sent in enumerate(sentences):

            score = (
                self.lambda_entropy * self.sentence_entropy(sent)
                +
                self.legal_term_score(sent)
                +
                self.position_weight(i, total)
            )

            scores.append(score)

        scores = np.array(scores, dtype=np.float32)

        scores = (
            scores - scores.min()
        ) / (
            scores.max() - scores.min() + 1e-8
        )

        return torch.tensor(
            scores,
            dtype=torch.float,
            device=self.device0
        )

    # ========================================================
    # SENTENCE ENCODING
    # Encoder(D), Encoder(S), then MLP projection
    # ========================================================

    def encode(self, texts, batch_size=8):

        all_embs = []

        for i in range(0, len(texts), batch_size):

            batch = texts[i:i + batch_size]

            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(self.device1)

            if self.train_encoder:
                outputs = self.encoder(**inputs)
            else:
                with torch.no_grad():
                    outputs = self.encoder(**inputs)

            cls = outputs.last_hidden_state[:, 0, :]

            cls = cls.to(self.device0)

            emb = self.proj(cls)

            all_embs.append(emb)

            del inputs
            del outputs
            del cls

            torch.cuda.empty_cache()

        return torch.cat(all_embs, dim=0)

    # ========================================================
    # ONE-STEP MARKOV TRANSITION
    # x0 ~ N(Hin, beta0 I)
    # ========================================================

    def one_step_markov(self, H):

        noise = torch.randn_like(H)

        x0 = H + math.sqrt(self.beta0) * noise

        return x0

    # ========================================================
    # MULTIFACTOR FUSION BETA SCHEDULE
    # beta_t^i = t/T + gamma + lambda(t)e(si)
    # lambda(t)=lambda_w sin(t/T*pi)
    # ========================================================

    def weighted_beta(
        self,
        t,
        sent_weights
    ):

        t_float = t.float() / self.T

        lambda_t = self.lambda_w * torch.sin(
            t_float * math.pi
        )

        beta = (
            t_float
            +
            self.gamma
            +
            lambda_t * sent_weights
        )

        beta = torch.clamp(
            beta,
            min=1e-5,
            max=0.999
        )

        return beta

    # ========================================================
    # PARTIAL FORWARD DIFFUSION
    # Document embeddings stay fixed.
    # Summary embeddings are noised.
    # ========================================================

    def forward_diffusion(
        self,
        x0_s,
        sent_weights,
        t
    ):

        noise = torch.randn_like(x0_s)

        beta_t = self.weighted_beta(
            t,
            sent_weights
        ).unsqueeze(1)

        x_t_s = (
            torch.sqrt(1.0 - beta_t) * x0_s
            +
            torch.sqrt(beta_t) * noise
        )

        return x_t_s

    # ========================================================
    # REVERSE DIFFUSION
    # Predict clean summary representation from noisy summary
    # conditioned on document context.
    # ========================================================

    def reverse_diffusion(
        self,
        x_t_s,
        doc_context,
        t
    ):

        t_emb = self.time_embed(t)

        context = doc_context.repeat(
            x_t_s.size(0),
            1
        )

        inp = torch.cat(
            [
                x_t_s,
                context,
                t_emb
            ],
            dim=1
        )

        pred_x0_s = self.reverse_net(inp)

        return pred_x0_s

    # ========================================================
    # SIMPLE ROUGE-LIKE SCORE FOR CANDIDATE RANKING
    # ========================================================

    def rouge_like_score(
        self,
        candidate_sentences,
        reference_sentences
    ):

        cand = " ".join(candidate_sentences)
        ref = " ".join(reference_sentences)

        cand_tokens = [
            w.lower()
            for w in word_tokenize(cand)
            if w.isalpha()
        ]

        ref_tokens = [
            w.lower()
            for w in word_tokenize(ref)
            if w.isalpha()
        ]

        if len(cand_tokens) == 0 or len(ref_tokens) == 0:
            return 0.0

        cand_uni = set(cand_tokens)
        ref_uni = set(ref_tokens)

        r1 = len(cand_uni & ref_uni) / max(len(ref_uni), 1)

        cand_bi = set(zip(cand_tokens, cand_tokens[1:]))
        ref_bi = set(zip(ref_tokens, ref_tokens[1:]))

        if len(ref_bi) == 0:
            r2 = 0.0
        else:
            r2 = len(cand_bi & ref_bi) / max(len(ref_bi), 1)

        return (r1 + r2) / 2.0

    # ========================================================
    # BUILD MULTIPLE CANDIDATE SUMMARIES
    # from generated summary embeddings
    # ========================================================

    def build_candidate_indices(
        self,
        Hd,
        pred_summary_emb,
        summary_size,
        k_candidates=5
    ):

        sim = torch.matmul(
            F.normalize(pred_summary_emb, dim=1),
            F.normalize(Hd, dim=1).T
        )

        sent_scores = sim.max(dim=0).values

        ranked = torch.argsort(
            sent_scores,
            descending=True
        ).detach().cpu().numpy()

        candidates = []

        for shift in range(k_candidates):

            chosen = ranked[
                shift:shift + summary_size
            ]

            if len(chosen) < summary_size:
                chosen = ranked[:summary_size]

            chosen = sorted(
                list(set(chosen.tolist()))
            )

            candidates.append(chosen)

        return candidates

    # ========================================================
    # f(S) = cos(Hd, Hs)
    # ========================================================

    def candidate_score(
        self,
        Hd,
        cand_indices
    ):

        doc_vec = Hd.mean(dim=0)

        cand_vec = Hd[
            cand_indices
        ].mean(dim=0)

        score = F.cosine_similarity(
            doc_vec.unsqueeze(0),
            cand_vec.unsqueeze(0)
        )

        return score.squeeze(0)

    # ========================================================
    # ROUGE-AWARE CONTRASTIVE RERANKING LOSS
    # Lctr = max(0, f(Cj)-f(Ci)+rho)
    # where ROUGE(Ci) > ROUGE(Cj)
    # ========================================================

    def reranking_loss(
        self,
        Hd,
        pred_summary_emb,
        document_sentences,
        reference_summary_sentences,
        summary_size,
        k_candidates=5
    ):

        candidates = self.build_candidate_indices(
            Hd,
            pred_summary_emb,
            summary_size=summary_size,
            k_candidates=k_candidates
        )

        if len(candidates) <= 1:

            return torch.tensor(
                0.0,
                device=self.device0
            )

        rouge_ranked = []

        for cand in candidates:

            cand_sents = [
                document_sentences[i]
                for i in cand
            ]

            rouge_score = self.rouge_like_score(
                cand_sents,
                reference_summary_sentences
            )

            model_score = self.candidate_score(
                Hd,
                cand
            )

            rouge_ranked.append(
                (
                    rouge_score,
                    model_score
                )
            )

        rouge_ranked = sorted(
            rouge_ranked,
            key=lambda x: x[0],
            reverse=True
        )

        losses = []

        for i in range(len(rouge_ranked)):

            for j in range(i + 1, len(rouge_ranked)):

                score_i = rouge_ranked[i][1]
                score_j = rouge_ranked[j][1]

                losses.append(
                    torch.relu(
                        score_j - score_i + self.margin
                    )
                )

        if len(losses) == 0:

            return torch.tensor(
                0.0,
                device=self.device0
            )

        return torch.stack(losses).mean()

    # ========================================================
    # FORWARD TRAINING
    # L = Lse + Ldiffusion + Lctr
    # ========================================================

    def forward(
        self,
        document_sentences,
        reference_summary_sentences
    ):

        Hd = self.encode(
            document_sentences
        )

        Hs = self.encode(
            reference_summary_sentences
        )

        x0_d = self.one_step_markov(
            Hd
        )

        x0_s = self.one_step_markov(
            Hs
        )

        doc_context = x0_d.mean(
            dim=0,
            keepdim=True
        )

        batch_size = x0_s.size(0)

        t = torch.randint(
            1,
            self.T,
            (batch_size,),
            device=self.device0
        )

        sent_weights = self.compute_sentence_weights(
            reference_summary_sentences
        )

        x_t_s = self.forward_diffusion(
            x0_s,
            sent_weights,
            t
        )

        pred_x0_s = self.reverse_diffusion(
            x_t_s,
            doc_context,
            t
        )

        # ----------------------------------------------------
        # Ldiffusion: clean representation reconstruction
        # ----------------------------------------------------

        diffusion_loss = F.mse_loss(
            pred_x0_s,
            x0_s
        )

        # ----------------------------------------------------
        # Lse approximation:
        # summary embeddings should align with document space
        # ----------------------------------------------------

        sim = torch.matmul(
            F.normalize(Hs, dim=1),
            F.normalize(Hd, dim=1).T
        )

        sentence_encoder_loss = 1.0 - sim.max(dim=1).values.mean()

        # ----------------------------------------------------
        # Lctr: ROUGE-aware candidate reranking
        # ----------------------------------------------------

        summary_size = min(
            len(reference_summary_sentences),
            len(document_sentences)
        )

        ctr_loss = self.reranking_loss(
            Hd=Hd,
            pred_summary_emb=pred_x0_s,
            document_sentences=document_sentences,
            reference_summary_sentences=reference_summary_sentences,
            summary_size=summary_size,
            k_candidates=5
        )

        loss = (
            sentence_encoder_loss
            +
            diffusion_loss
            +
            ctr_loss
        )

        return loss

    # ========================================================
    # INFERENCE
    # Reverse diffusion + similarity mapping
    # ========================================================

    def summarize(
        self,
        document_sentences,
        summary_size=10
    ):

        self.eval()

        with torch.no_grad():

            Hd = self.encode(
                document_sentences
            )

            x0_d = self.one_step_markov(
                Hd
            )

            doc_context = x0_d.mean(
                dim=0,
                keepdim=True
            )

            x_t_s = torch.randn(
                summary_size,
                self.hidden_size,
                device=self.device0
            )

            # Use highest document sentence weights as proxy
            # for generated summary slots during inference
            doc_weights = self.compute_sentence_weights(
                document_sentences
            )

            if doc_weights.numel() >= summary_size:

                proxy_weights = torch.topk(
                    doc_weights,
                    summary_size
                ).values

            else:

                pad = torch.ones(
                    summary_size - doc_weights.numel(),
                    device=self.device0
                ) * doc_weights.mean()

                proxy_weights = torch.cat(
                    [
                        doc_weights,
                        pad
                    ],
                    dim=0
                )

            for step in reversed(
                range(1, self.T)
            ):

                t = torch.full(
                    (summary_size,),
                    step,
                    device=self.device0,
                    dtype=torch.long
                )

                pred_x0_s = self.reverse_diffusion(
                    x_t_s,
                    doc_context,
                    t
                )

                beta_t = self.weighted_beta(
                    t,
                    proxy_weights
                ).unsqueeze(1)

                noise = torch.randn_like(
                    x_t_s
                ) if step > 1 else torch.zeros_like(
                    x_t_s
                )

                x_t_s = (
                    torch.sqrt(1.0 - beta_t) * pred_x0_s
                    +
                    torch.sqrt(beta_t) * noise
                )

            sim = torch.matmul(
                F.normalize(pred_x0_s, dim=1),
                F.normalize(Hd, dim=1).T
            )

            scores = sim.max(dim=0).values

            ranked = torch.argsort(
                scores,
                descending=True
            ).detach().cpu().numpy()

            selected = []

            for idx in ranked:

                if idx not in selected:
                    selected.append(int(idx))

                if len(selected) >= summary_size:
                    break

            selected = sorted(selected)

            return [
                document_sentences[i]
                for i in selected
            ]

# ============================================================
# INITIALIZE MODEL
# ============================================================

# diffusion_model = TermDiffuSum(device).to(device)
diffusion_model = TermDiffuSum(
    device0=device0,
    device1=device1,
    hidden_size=768,
    diffusion_steps=200,
    beta0=1e-4,
    gamma=1e-4,
    lambda_w=0.02,
    lambda_entropy=1.0,
    margin=0.1,
    train_encoder=False
)

diffusion_model.encoder = diffusion_model.encoder.to(device1)

optimizer = torch.optim.AdamW(
    diffusion_model.parameters(),
    lr=2e-5
)

print("Encoder   :", next(diffusion_model.encoder.parameters()).device)
print("Proj      :", next(diffusion_model.proj.parameters()).device)
print("ReverseNet:", next(diffusion_model.reverse_net.parameters()).device)
# ============================================================
# TRAINING
# ============================================================

EPOCHS = 3

for epoch in range(EPOCHS):

    diffusion_model.train()

    total_loss = 0
    step  = 0

    for doc in tqdm(
        train_dataset,
        desc=f"Epoch {epoch+1}"
    ):

        document_sentences = doc["judgement_sent"]
        reference_summary_sentences = doc["headnote_sent"]

        if (
            len(document_sentences) < 5
            or
            len(reference_summary_sentences) < 2
        ):
            continue

        optimizer.zero_grad()

        loss = diffusion_model(
            document_sentences,
            reference_summary_sentences
        )

        loss.backward()
        if step % 50 == 0:

            grad_value = None
        
            for p in diffusion_model.proj.parameters():
                if p.grad is not None:
                    grad_value = p.grad.abs().mean().item()
                    break
        
            if grad_value is None:
                grad_value = 0.0
        
            print(
                f"Loss={loss.item():.4f}",
                f"Grad={grad_value:.6e}"
            )
        
        torch.nn.utils.clip_grad_norm_(
                    diffusion_model.parameters(),
                    1.0
        )

        loss_value = loss.item()
        optimizer.step()

        optimizer.zero_grad(set_to_none=True)
        
        del document_sentences
        del reference_summary_sentences      
        del loss
        
        total_loss += loss_value
        step += 1

    avg_loss = total_loss / max(step, 1)
    if step % 20 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    print(
        f"Epoch {epoch+1} Loss: "
        f"{avg_loss:.4f}"
    )

# ============================================================
# SAVE MODEL
# ============================================================

torch.save(
    diffusion_model.state_dict(),
    "pure_termdiffusum.pt"
)

print("Model Saved!")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        # inputs = tokenizer(
        #     chunk,
        #     return_tensors="pt",
        #     truncation=True,
        #     max_length=1024
        # ).to(device)
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device1)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

def factcc_chunked_fast(
    pred,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device1)

        with torch.no_grad():

            logits = factcc_model(**inputs).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))

# ============================================================
# EVALUATION
# ============================================================

diffusion_model.eval()

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []
bart_scores = []
factcc_scores = []

for doc in tqdm(
    test_dataset,
    desc="Evaluating"
):

    reference = " ".join(
        doc["headnote_sent"]
    )

    document_sentences = doc["judgement_sent"]

    judgement = " ".join(
        document_sentences
    )

    target_len = estimate_summary_length(
        document_sentences
    )

    # ========================================================
    # EXTRACTIVE SUMMARY
    # ========================================================

    extracted_sentences = diffusion_model.summarize(
        document_sentences,
        summary_size=target_len
    )

    extractive_summary = " ".join(
        extracted_sentences
    )

    # ========================================================
    # ABSTRACTIVE SUMMARY
    # ========================================================

    pred = generate_summary_chunked(
        extractive_summary,
        abstractive_tokenizer,
        abstractive_model
    )

    # ========================================================
    # ROUGE
    # ========================================================

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )

    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # ========================================================
    # BLEU
    # ========================================================

    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )

    bleu_scores.append(
        bleu_doc["bleu"]
    )

    # ========================================================
    # METEOR
    # ========================================================

    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )

    meteor_scores.append(
        meteor_doc["meteor"]
    )

    # ========================================================
    # BERTSCORE
    # ========================================================

    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
        bert_doc["f1"][0]
    )

    # ========================================================
    # BARTSCORE
    # ========================================================

    bart_scores.append(
        bart_scorer.score(
            [pred],
            [reference]
        )[0]
    )

    # ========================================================
    # FACTCC
    # ========================================================

    judgement_chunks = chunk_text(
        judgement,
        abstractive_tokenizer
    )

    factcc_scores.append(
        factcc_chunked_fast(
            pred,
            judgement_chunks
        )
    )
    del extracted_sentences
    del extractive_summary
    del pred
    del judgement
    del judgement_chunks

    gc.collect()
    torch.cuda.empty_cache()

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================ FINAL RESULTS ================\n")

print(f"ROUGE-1     : {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     : {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     : {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  : {np.mean(rougelsum_scores):.4f}")

print(f"BLEU        : {np.mean(bleu_scores):.4f}")
print(f"METEOR      : {np.mean(meteor_scores):.4f}")

print(f"BERTScore   : {np.mean(bert_scores):.4f}")
print(f"BARTScore   : {np.mean(bart_scores):.4f}")

print(f"FactCC      : {np.mean(factcc_scores):.4f}")

print("\n================================================")

# inLegalBERT + KMeans

In [ ]:
# ============================================================
# STANDARD LIBRARIES
# ============================================================

import os
import sys
import gc
import re
import json
import math

# ============================================================
# CORE LIBRARIES
# ============================================================

import numpy as np
from tqdm import tqdm

# ============================================================
# PYTORCH
# ============================================================

import torch
import torch.nn.functional as F

# ============================================================
# NLP LIBRARIES
# ============================================================

import nltk
import spacy
import evaluate

from nltk.corpus import stopwords

# ============================================================
# TRANSFORMERS
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)
from sentence_transformers import SentenceTransformer

# ============================================================
# SCIKIT-LEARN
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# BARTScore
# ============================================================

sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

from bart_score import BARTScorer

# ============================================================
# INITIAL SETUP
# ============================================================

nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================

DATASET_PATH = (
    "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"
)

with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)


# dataset = dataset[400:]

# ============================================================
# LOAD SUMMARIZATION MODEL
# ============================================================

MODEL_PATH = (
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH
).to(device)

summarizer_model.eval()

# ============================================================
# LOAD EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# LOAD BARTScore
# ============================================================

bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# ============================================================
# LOAD FACTCC
# ============================================================

FACTCC_MODEL = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    FACTCC_MODEL
)

factcc_model = AutoModelForSequenceClassification.from_pretrained(
    FACTCC_MODEL
).to(device)

factcc_model.eval()

# ============================================================
# LOAD SBERT
# ============================================================

# sbert_model = SentenceTransformer(
#     "all-mpnet-base-v2",
#     device=device
# )


# ============================================================
# LOAD INLEGALBERT FOR EMBEDDINGS
# ============================================================

embedding_tokenizer = AutoTokenizer.from_pretrained(
    "law-ai/InLegalBERT"
)

embedding_model = AutoModel.from_pretrained(
    "law-ai/InLegalBERT"
).to(device)

embedding_model.eval()


# ============================================================
# TEXT PREPROCESSING
# ============================================================

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ============================================================
# K-MEANS + InLegalBERT EXTRACTIVE SUMMARIZER
# (Inspired by 2025 Legal Summarization Methodology)
# ============================================================

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

# ============================================================
# SENTENCE FILTERING
# ============================================================

def filter_sentences(sentences):

    filtered = []

    for sent in sentences:

        sent = sent.strip()

        if len(sent.split()) < 5:
            continue

        alpha_ratio = (
            sum(c.isalpha() for c in sent)
            /
            max(len(sent), 1)
        )

        if alpha_ratio < 0.4:
            continue

        procedural_patterns = [
            r"section\s+\d+",
            r"rule\s+\d+",
            r"article\s+\d+",
            r"order\s+\d+"
        ]

        procedural_hits = sum(
            bool(re.search(p, sent.lower()))
            for p in procedural_patterns
        )

        # skip only if heavily procedural
        if procedural_hits >= 2:
            continue

        filtered.append(sent)

    return filtered


# ============================================================
# SENTENCE EMBEDDINGS USING InLegalBERT
# ============================================================

# ============================================================
# SENTENCE EMBEDDINGS
# ============================================================

def get_sentence_embeddings(
    sentences,
    tokenizer,
    model,
    batch_size=8
):

    embeddings = []

    for i in range(0, len(sentences), batch_size):

        batch = sentences[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        ).to(device)

        with torch.no_grad():

            outputs = model(
                **inputs
            )

        # mean pooling
        hidden = outputs.last_hidden_state

        mask = inputs["attention_mask"].unsqueeze(-1)

        masked_hidden = hidden * mask

        summed = masked_hidden.sum(dim=1)
        
        counts = mask.sum(dim=1).clamp(min=1e-9)
        
        pooled = summed / counts

        embeddings.append(
            pooled.cpu().numpy()
        )

    return np.vstack(embeddings)



# ============================================================
# JUDGMENT RESULT EXTRACTION
# ============================================================

def extract_judgment_result(text):

    result_keywords = [
        "dismissed",
        "allowed",
        "disposed of",
        "convicted",
        "acquitted",
        "granted",
        "rejected"
    ]

    sentences = sent_tokenize(text)

    last_sentences = sentences[-3:]

    for s in reversed(last_sentences):

        lower = s.lower()

        for k in result_keywords:

            if k in lower:

                return s

    return ""


# ============================================================
# K-MEANS CLUSTERING + REPRESENTATIVE SENTENCE SELECTION
# ============================================================

def cluster_and_rank(
    sentences,
    embeddings,
    num_sentences
):

    num_sentences = min(
        num_sentences,
        len(sentences)
    )
    
    if len(sentences) <= num_sentences:
        return sentences

    # --------------------------------------------------------
    # K-MEANS
    # --------------------------------------------------------
    embeddings = embeddings / (
        np.linalg.norm(
            embeddings,
            axis=1,
            keepdims=True
        ) + 1e-8
    )
    kmeans = KMeans(
        n_clusters=num_sentences,
        random_state=42,
        n_init=10
    )

    cluster_ids = kmeans.fit_predict(
        embeddings
    )

    centers = kmeans.cluster_centers_

    selected_indices = set()

    # --------------------------------------------------------
    # REPRESENTATIVE SENTENCE FROM EACH CLUSTER
    # --------------------------------------------------------

    for cluster_id in range(num_sentences):

        cluster_idx = np.where(
            cluster_ids == cluster_id
        )[0]

        cluster_emb = embeddings[cluster_idx]

        center = centers[cluster_id]

        sims = cosine_similarity(
            [center],
            cluster_emb
        )[0]

        best_local_idx = np.argmax(sims)

        best_sentence_idx = cluster_idx[
            best_local_idx
        ]

        selected_indices.add(
            int(best_sentence_idx)
        )

    # preserve original legal order
    selected_indices = sorted(
        selected_indices
    )

    return [
        sentences[i]
        for i in selected_indices
    ]


# ============================================================
# COMPLETE EXTRACTIVE SUMMARIZATION PIPELINE
# ============================================================

def summarize_judgment_extractively(
    text,
    tokenizer,
    model,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    # --------------------------------------------------------
    # SENTENCE TOKENIZATION
    # --------------------------------------------------------

    sentences = sent_tokenize(text)

    # --------------------------------------------------------
    # FILTERING
    # --------------------------------------------------------

    filtered_sentences = filter_sentences(
        sentences
    )

    if len(filtered_sentences) == 0:
        return text

    # --------------------------------------------------------
    # DYNAMIC SUMMARY LENGTH
    # --------------------------------------------------------

    num_sentences = estimate_summary_length(
        filtered_sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

    # --------------------------------------------------------
    # EMBEDDINGS
    # --------------------------------------------------------

    embeddings = get_sentence_embeddings(
        filtered_sentences,
        tokenizer,
        model
    )

    # --------------------------------------------------------
    # CLUSTERING + SELECTION
    # --------------------------------------------------------

    selected_sentences = cluster_and_rank(
        filtered_sentences,
        embeddings,
        num_sentences
    )

    # --------------------------------------------------------
    # JUDGMENT RESULT
    # --------------------------------------------------------

    judgment_result = extract_judgment_result(
        text
    )

    # --------------------------------------------------------
    # FINAL SUMMARY
    # --------------------------------------------------------

    summary = " ".join(
        selected_sentences
    )

    if judgment_result:

        if judgment_result not in summary:

            summary += " " + judgment_result

    return summary
    

# ============================================================
# SUMMARY LENGTH ESTIMATION
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )



# ============================================================
# TEXT CHUNKING
# ============================================================

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=150
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

# ============================================================
# CHUNK-BASED SUMMARY GENERATION
# ============================================================

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

# ============================================================
# FACTCC EVALUATION
# ============================================================

def factcc_chunked_fast(
    prediction,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            prediction,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():

            logits = factcc_model(
                **inputs
            ).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))

# ============================================================
# MAIN EVALUATION LOOP
# ============================================================

# results = {}

# for variant_name, config in ABLATION_VARIANTS.items():
# print(f"\nRunning Variant: {variant_name}")

rouge1_scores = []
rouge2_scores = []
rougel_scores = []
rougelsum_scores = []

bleu_scores = []
meteor_scores = []
bert_scores = []
bart_scores = []

factcc_scores = []

print("\nRunning KMeans-InLegalBERT Extractive Pipeline")

for doc in tqdm(dataset, desc="Extractive Summarization"):

    # ----------------------------------------------------
    # REFERENCE SUMMARY
    # ----------------------------------------------------

    reference = " ".join(
            doc["headnote_sent"]
    )

    # ----------------------------------------------------
    # SOURCE JUDGMENT
    # ----------------------------------------------------

    judgement = " ".join(
        s["sentence"]
        for s in doc["judgement_roles"]
        if isinstance(s, dict)
    )

    # --------------------------------------------------------
    # EXTRACTIVE SUMMARY
    # --------------------------------------------------------

    extractive_summary = summarize_judgment_extractively(
        text=judgement,
        tokenizer=embedding_tokenizer,
        model=embedding_model,
        ratio=0.25,
        min_len=8,
        max_len=30
    )

    # ----------------------------------------------------
    # SUMMARY GENERATION
    # ----------------------------------------------------

    prediction = generate_summary_chunked(
            extractive_summary,
            tokenizer,
            summarizer_model
    )

    # ----------------------------------------------------
    # ROUGE
    # ----------------------------------------------------

    rouge_result = rouge.compute(
            predictions=[prediction],
            references=[reference]
    )

    rouge1_scores.append(
            rouge_result["rouge1"]
    )

    rouge2_scores.append(
            rouge_result["rouge2"]
    )

    rougel_scores.append(
            rouge_result["rougeL"]
    )

    rougelsum_scores.append(
            rouge_result["rougeLsum"]
    )

    # ----------------------------------------------------
    # BLEU
    # ----------------------------------------------------

    bleu_result = bleu.compute(
            predictions=[prediction],
            references=[[reference]]
    )

    bleu_scores.append(
            bleu_result["bleu"]
    )

    # ----------------------------------------------------
    # METEOR
    # ----------------------------------------------------

    meteor_result = meteor.compute(
            predictions=[prediction],
            references=[reference]
    )

    meteor_scores.append(
            meteor_result["meteor"]
    )

    # ----------------------------------------------------
    # BERTScore
    # ----------------------------------------------------

    bert_result = bertscore.compute(
        predictions=[prediction],
        references=[reference],
        lang="en"
    )

    bert_scores.append(
            bert_result["f1"][0]
    )

    # ----------------------------------------------------
    # BARTScore
    # ----------------------------------------------------

    bart_scores.append(
        bart_scorer.score(
                [prediction],
                [reference]
        )[0]
    )

    # ----------------------------------------------------
    # FACTCC
    # ----------------------------------------------------

    judgement_chunks = chunk_text(
            judgement,
            factcc_tokenizer,
            max_tokens=900,
            stride=150
    )

    factcc_scores.append(
        factcc_chunked_fast(
                prediction,
                judgement_chunks
        )
    )


    torch.cuda.empty_cache()

    gc.collect()

# ========================================================
# PRINT RESULTS
# ========================================================

print(f"ROUGE-1    : {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2    : {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L    : {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM : {np.mean(rougelsum_scores):.4f}")
print(f"BLEU       : {np.mean(bleu_scores):.4f}")
print(f"METEOR     : {np.mean(meteor_scores):.4f}")
print(f"BERTScore  : {np.mean(bert_scores):.4f}")
print(f"BARTScore  : {np.mean(bart_scores):.4f}")
print(f"FactCC     : {np.mean(factcc_scores):.4f}")


# GreedSUM

In [ ]:
import sysimport os



# Add the BARTScore repo folder to Python path

sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import

import bart_scoreprint("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer

# from summac.model_summac import SummaCConv, SummaCZS

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
nltk.download("punkt")
nltk.download("stopwords")
from sklearn.metrics.pairwise import cosine_similarity

stop_words = set(stopwords.words("english"))
from nltk import sent_tokenize

file_path = "/kaggle/input/datasets/pawank1999/indian-supreme-court-judgments/data12.json"

with open(file_path, "r") as f:
    full_dataset = json.load(f)

train_dataset = full_dataset[:5000]
dataset = full_dataset[9500:]

print("Train documents:", len(train_dataset))
print("Test documents :", len(dataset))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained("/kaggle/input/datasets/pawank1999/distilbart-9500-output").to(device)

tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/pawank1999/distilbart-9500-output")

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

BARTScore

bart_scorer = BARTScorer(device=device,checkpoint="facebook/bart-large-cnn")

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [w.lower() for w in word_tokenize(sent) if w.isalpha() and w.lower() not in stop_words]
        processed_sentences.append(" ".join(words))
        return sentences, processed_sentences

from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans

from sklearn.metrics import silhouette_score

============================================================

PAPER-ALIGNED GREEDSUM WITH DYNAMIC MIN-DF

============================================================

class GreedSum:

    def __init__(
        self,
        min_df=2,
        max_features=8000
    ):
    
        self.min_df = min_df
        self.max_features = max_features
    
    # --------------------------------------------------------
    # BASIC PREPROCESSING
    # --------------------------------------------------------
    
    def clean_sentence(self, sentence):
    
        sentence = sentence.lower()
    
        sentence = re.sub(
            r"[^a-z0-9\s]",
            " ",
            sentence
        )
    
        sentence = re.sub(
            r"\s+",
            " ",
            sentence
        )
    
        return sentence.strip()
    
    # --------------------------------------------------------
    # EXACT GREEDSUM EXTRACTION
    # --------------------------------------------------------
    
    def summarize(
        self,
        sentences,
        num_sentences=10
    ):
    
        if len(sentences) == 0:
            return []
    
        if len(sentences) <= num_sentences:
            return sentences
    
        cleaned_sentences = [
            self.clean_sentence(s)
            for s in sentences
        ]
    
        cleaned_sentences = [
            s if len(s.strip()) > 0 else "empty"
            for s in cleaned_sentences
        ]
    
        try:
    
            vectorizer = TfidfVectorizer(
                stop_words="english",
                min_df=self.min_df,
                max_features=self.max_features
            )
    
            tfidf_matrix = vectorizer.fit_transform(
                cleaned_sentences
            ).toarray()
    
        except ValueError:
    
            vectorizer = TfidfVectorizer(
                stop_words="english",
                min_df=1,
                max_features=self.max_features
            )
    
            tfidf_matrix = vectorizer.fit_transform(
                cleaned_sentences
            ).toarray()
    
        selected_indices = []
    
        working_matrix = tfidf_matrix.copy()
    
        for _ in range(
            min(num_sentences, len(sentences))
        ):
    
            sentence_scores = working_matrix.sum(
                axis=1
            )
    
            for idx in selected_indices:
                sentence_scores[idx] = -1.0
    
            best_idx = int(
                np.argmax(sentence_scores)
            )
    
            if sentence_scores[best_idx] <= 0:
                break
    
            selected_indices.append(
                best_idx
            )
    
            selected_terms = (
                working_matrix[best_idx] > 0
            )
    
            working_matrix[
                :,
                selected_terms
            ] = 0.0
    
        selected_indices = sorted(
            selected_indices
        )
    
        return [
            sentences[i]
            for i in selected_indices
        ]
   

============================================================

DYNAMIC MIN-DF OPTIMIZATION USING TF-IDF + PCA + K-MEANS

============================================================

def build_corpus_texts(dataset):

    corpus_texts = []
    
    for doc in dataset:
    
        text = " ".join(
            doc["judgement_sent"]
        )
    
        text = re.sub(
            r"[^a-zA-Z0-9\s]",
            " ",
            text
        )
    
        text = re.sub(
            r"\s+",
            " ",
            text
        ).strip()
    
        corpus_texts.append(text)
    
    return corpus_texts

def find_optimal_k(X,min_k=2,max_k=8):

    best_k = min_k
    best_score = -1
    
    max_k = min(
        max_k,
        X.shape[0] - 1
    )
    
    for k in range(min_k, max_k + 1):
    
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )
    
        labels = kmeans.fit_predict(X)
    
        if len(set(labels)) < 2:
            continue
    
        score = silhouette_score(
            X,
            labels
        )
    
        if score > best_score:
            best_score = score
            best_k = k
    
    return best_k

def compute_cluster_min_df(corpus_texts,labels,min_value=1,max_value=5):

    cluster_min_df = {}
    
    unique_clusters = sorted(
        set(labels)
    )
    
    for cluster_id in unique_clusters:
    
        docs = [
            corpus_texts[i]
            for i in range(len(corpus_texts))
            if labels[i] == cluster_id
        ]
    
        tokenized_docs = []
    
        for text in docs:
    
            tokens = set(
                w.lower()
                for w in word_tokenize(text)
                if w.isalpha()
                and w.lower() not in stop_words
            )
    
            tokenized_docs.append(tokens)
    
        df_counter = Counter()
    
        for tokens in tokenized_docs:
    
            for token in tokens:
                df_counter[token] += 1
    
        if len(df_counter) == 0:
    
            cluster_min_df[cluster_id] = min_value
            continue
    
        df_values = np.array(
            list(df_counter.values())
        )
    
        avg_df = int(
            np.round(
                np.mean(df_values)
            )
        )
    
        dynamic_min_df = max(
            min_value,
            min(avg_df, max_value)
        )
    
        cluster_min_df[cluster_id] = dynamic_min_df
    
    return cluster_min_df

class DynamicMinDFGreedSum:

    def __init__(
        self,
        train_dataset,
        max_features=8000,
        pca_components=50,
        max_clusters=8
    ):
    
        self.max_features = max_features
    
        self.corpus_texts = build_corpus_texts(
            train_dataset
        )
    
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            min_df=1,
            max_features=max_features
        )
    
        X_tfidf = self.vectorizer.fit_transform(
            self.corpus_texts
        )
    
        X_dense = X_tfidf.toarray()
    
        n_components = min(
            pca_components,
            X_dense.shape[0],
            X_dense.shape[1]
        )
    
        self.pca = PCA(
            n_components=n_components,
            random_state=42
        )
    
        X_pca = self.pca.fit_transform(
            X_dense
        )
    
        optimal_k = find_optimal_k(
            X_pca,
            min_k=2,
            max_k=max_clusters
        )
    
        self.kmeans = KMeans(
            n_clusters=optimal_k,
            random_state=42,
            n_init=10
        )
    
        self.labels = self.kmeans.fit_predict(
            X_pca
        )
    
        self.cluster_min_df = compute_cluster_min_df(
            self.corpus_texts,
            self.labels
        )
    
        print(
            "Optimal clusters:",
            optimal_k
        )
    
        print(
            "Cluster min_df:",
            self.cluster_min_df
        )
    
    def get_document_cluster(
        self,
        sentences
    ):
    
        text = " ".join(sentences)
    
        X = self.vectorizer.transform(
            [text]
        ).toarray()
    
        X_pca = self.pca.transform(
            X
        )
    
        cluster_id = int(
            self.kmeans.predict(
                X_pca
            )[0]
        )
    
        return cluster_id
    
    def summarize(
        self,
        sentences,
        num_sentences=10
    ):
    
        cluster_id = self.get_document_cluster(
            sentences
        )
    
        min_df = self.cluster_min_df.get(
            cluster_id,
            2
        )
    
        summarizer = GreedSum(
            min_df=min_df,
            max_features=self.max_features
        )
    
        return summarizer.summarize(
            sentences,
            num_sentences=num_sentences
        )

# FactCC

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(factcc_model_name).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
    
def estimate_summary_length(sentences,ratio=0.35,min_len=10,max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """Splits text into overlapping token chunks"""
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap
    
    return chunks


def generate_summary_chunked(text, tokenizer, model):
    
    """Chunk → summarize → merge"""
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []
    
    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)
    
        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )
    
        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
    
        chunk_summaries.append(summary)
    
    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary

def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)
    
        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)
    
        chunk_scores.append(probs[0, 1].item())
    
    return float(np.mean(chunk_scores))


from transformers import AutoModelForSequenceClassification, AutoTokenizer
sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)


from transformers import pipeline
from nltk.tokenize import sent_tokenize
predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

greedsum = DynamicMinDFGreedSum(train_dataset=train_dataset,max_features=8000,pca_components=50,max_clusters=8)

for doc in tqdm(dataset, desc="Evaluating"):
    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(preprocess(s)for s in doc["judgement_sent"]# if not is_metadata_sentence(s))sentences = doc["judgement_sent"]

    top_n = estimate_summary_length(
        sentences,
        ratio=0.25,   # can tune (0.20–0.35)
        min_len=8,
        max_len=30
    )
    
    
    selected_sentences = greedsum.summarize(
        sentences,
        num_sentences=top_n
    )
    
    ranked_text = " ".join(selected_sentences)

    
    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )
    # print("done2")
    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])
    
    # # # -----------------------
    # # # BLEU (DOCUMENT-WISE)
    # # # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])
    
    # # # -----------------------
    # # # METEOR (DOCUMENT-WISE)
    # # # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])
    
    # # # -----------------------
    # # # BERTScore (DOCUMENT-WISE)
    # # # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])
    

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )
    
    
    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    torch.cuda.empty_cache()
    
    
print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

# BERT Kmeans

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
from transformers import AutoModelForSeq2SeqLM
nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)
def estimate_summary_length(sentences,
                            ratio=0.35,
                            min_len=10,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))
def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()

def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)
ROLE_PRIORITY = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "RLC": 0.55,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PRE_NOT_RELIED": 0.4,
    "PREAMBLE": 0.3,
    "NONE": 0.2
}


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

# SummaC
# summac_conv = SummaCConv(device=device)
# summac_zs = SummaCZS(device=device)
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)

def estimate_n_clusters(sentences,
                        ratio=0.25,
                        min_len=8,
                        max_len=30):

    return estimate_summary_length(
        sentences,
        ratio=ratio,
        min_len=min_len,
        max_len=max_len
    )

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import HdpModel
from gensim.corpora import Dictionary

from transformers import BertModel, BertTokenizer


class BERTKMeansSummarizer:
    def __init__(self, model_name='bert-base-uncased', device='cpu'):
        self.device = device
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name).to(device)
        self.model.eval()

    def get_embeddings(self, sentences, batch_size=8):
        embeddings = []

        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]

            inputs = self.tokenizer(
                batch,
                return_tensors='pt',
                truncation=True,
                padding=True,
                max_length=128
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)

            cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS
            embeddings.append(cls_emb.cpu().numpy())

            del inputs, outputs, cls_emb
            torch.cuda.empty_cache()

        return np.vstack(embeddings)

    def summarize(self, sentences, num_sentences=5):
        if len(sentences) <= num_sentences:
            return sentences

        embeddings = self.get_embeddings(sentences)

        k = min(num_sentences, len(sentences))
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)

        selected_indices = []

        for i in range(k):
            cluster_indices = np.where(kmeans.labels_ == i)[0]
            cluster_embeddings = embeddings[cluster_indices]

            center = kmeans.cluster_centers_[i]

            distances = np.linalg.norm(cluster_embeddings - center, axis=1)
            closest_idx = cluster_indices[np.argmin(distances)]

            selected_indices.append(closest_idx)

        selected_indices = sorted(set(selected_indices))

        return [sentences[i] for i in selected_indices]
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
def summac_chunked_fast(model, pred, judgement_chunks):

    pred_sents = sent_tokenize(pred)

    scores = []

    for chunk in judgement_chunks:
        for ps in pred_sents:

            score = model.score([chunk], [ps])["scores"][0]
            scores.append(score)

    return float(np.mean(scores))
import gc


def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks
def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary
    
bert_kmeans = BERTKMeansSummarizer(device=device)

predictions, references = [], []

rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
bleu_scores, meteor_scores, bert_scores = [], [], []
bart_scores = []
factcc_scores = []

for doc in tqdm(dataset, desc="Evaluating"):

    reference = " ".join(doc["headnote_sent"])
    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )
    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]
    
    target_len = estimate_summary_length(
        sentences,
        ratio=0.25,
        min_len=8,
        max_len=30
    )
    
    selected_sentences = bert_kmeans.summarize(
        sentences,
        num_sentences=target_len
    )
    
    ranked_text = " ".join(selected_sentences)

    # -----------------------
    # Generate Summary
    # -----------------------
    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    # -----------------------
    # ROUGE (DOCUMENT-WISE)
    # -----------------------
    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )
    rouge1_scores.append(r["rouge1"])
    rouge2_scores.append(r["rouge2"])
    rougel_scores.append(r["rougeL"])
    rougelsum_scores.append(r["rougeLsum"])

    # -----------------------
    # BLEU (DOCUMENT-WISE)
    # -----------------------
    bleu_doc = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )
    bleu_scores.append(bleu_doc["bleu"])

    # -----------------------
    # METEOR (DOCUMENT-WISE)
    # -----------------------
    meteor_doc = meteor.compute(
        predictions=[pred],
        references=[reference]
    )
    meteor_scores.append(meteor_doc["meteor"])

    # -----------------------
    # BERTScore (DOCUMENT-WISE)
    # -----------------------
    bert_doc = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )
    bert_scores.append(bert_doc["f1"][0])

    # -----------------------
    # BARTScore (DOCUMENT-WISE)
    # -----------------------
    bart_scores.append(
        bart_scorer.score([pred], [reference])[0]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )
    factcc_scores.append(
        factcc_chunked_fast(pred, judgement_chunks)
    )

    del reference
    del judgement
    del sentences
    del roles
    del ranked_text
    del pred
    del judgement_chunks
    del bleu_doc
    del meteor_doc
    del bert_doc
    del r
    torch.cuda.empty_cache()
    gc.collect()

print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")
